# Perfiles de clientes que dejan el paquete Premium

## Introducción

La consigna solicita identificar perfiles diferentes entre los clientes que dejan el paquete Premium y proponer acciones comerciales basadas en la evidencia observada. El objetivo es comprender en qué se diferencian entre sí quienes ya dejaron de aparecer en la base.

La información disponible contiene observaciones mensuales de clientes entre marzo y agosto de 2021. Como no existe una variable que registre directamente la cancelación contractual, la baja se aproxima mediante la desaparición del cliente en los meses siguientes. Por este motivo, se distinguen:

- **Bajas estrictas:** clientes cuya ausencia puede sostenerse durante al menos dos meses observables.
- **Casos censurados:** clientes cuya permanencia fuera de la base no puede confirmarse porque la ventana temporal termina.
- **Clientes intermitentes:** clientes que desaparecen temporalmente y luego regresan.

Las etiquetas `BAJA+1` y `BAJA+2` se utilizan como estados temporales de una misma trayectoria y no como dos tipos diferentes de cliente. A partir de ellas se construye un único evento de salida por cliente.

El análisis sigue el siguiente recorrido:

1. Auditoría de la estructura y calidad de los datos.
2. Construcción de las etiquetas `BAJA+1`, `BAJA+2` y `CONTINUA`.
3. Identificación de bajas estrictas, censuradas e intermitentes.
4. Comparación entre bajas y clientes que continúan dentro del mismo mes.
5. Construcción de variables temporales mediante lags, deltas y ventanas móviles.
6. Segmentación de las bajas con historia suficiente.
7. Caracterización de los perfiles y formulación de acciones preliminares.
8. Evaluación de sensibilidad y exposición de las limitaciones.

La pertenencia al paquete Premium se asume a partir del contexto de la consigna, ya que el dataset no contiene una variable que permita comprobarla directamente.

## 1. Resumen ejecutivo

El análisis identifica 3.977 bajas estrictas, 1.103 casos censurados a derecha y 23 intermitentes. Los casos del último mes se utilizan como sensibilidad; quienes desaparecen y regresan se mantienen separados.

La segmentación se construye sobre las 1.991 bajas estrictas con tres meses consecutivos de historia. La solución principal contiene cuatro perfiles: Vinculados multicanal (450), Digitales selectivos (541), Vinculados por nómina (104) y Desvinculación avanzada (896). Se utiliza una muestra balanceada de bajas y continuadores, cinco bosques con semillas declaradas y la proximidad promedio de Random Forest.

La solución es más estable que `k=3` dentro de los diagnósticos de semillas, pero muestra sensibilidad a la selección de continuadores y a incorporar censurados. Por ello se interpreta como una taxonomía útil y provisional. Las acciones propuestas son hipótesis para experimentar: el dataset no permite atribuir causalidad ni medir por sí solo la efectividad incremental de una campaña.


## 2. Diseño metodológico

El análisis distingue tres unidades que no deben confundirse:

1. **Fila cliente-mes:** indica si un cliente está presente en una fotografía mensual y permite construir las etiquetas temporales `BAJA+1`, `BAJA+2` y `CONTINUA`.
2. **Evento de salida:** reúne las distintas etiquetas temporales asociadas con una misma desaparición y evita contar dos veces una única baja.
3. **Perfil de comportamiento:** agrupa eventos de salida con características semejantes de actividad, productos, saldos y evolución previa.

La clase temporal se construye mediante el procedimiento desarrollado en SQL durante la materia: se generan todas las combinaciones cliente-mes, se identifica la presencia mediante `mes_0` y se observan los meses siguientes con `LEAD()`.

Posteriormente se distinguen tres poblaciones:

- bajas estrictas;
- casos censurados por el final de la ventana;
- clientes intermitentes que desaparecen y regresan.

La segmentación principal se construye únicamente con bajas estrictas que poseen tres meses consecutivos de historia. Los clientes que continúan se utilizan como referencia para construir una representación de comportamiento, pero no se incorporan como miembros de los clusters.

Los casos censurados se emplean para evaluar sensibilidad. Los 23 clientes intermitentes se conservan como un grupo especial y se analizan separadamente, porque su regreso impide considerarlos bajas permanentes.

El feature engineering temporal sigue el procedimiento visto en `z402_Feature_Engineering_en_SQL.ipynb`. La representación con Random Forest toma como antecedente `z501_cluster_rf.py`; posteriormente se aplica clustering jerárquico para identificar perfiles entre las bajas.


## 3. Configuración del entorno

En esta sección se centralizan las librerías, rutas y semillas necesarias para reproducir el análisis. Las importaciones se realizan una sola vez y no se instalan paquetes dentro del notebook.

Las semillas permiten repetir los controles de estabilidad. La cantidad final de clusters y sus nombres comerciales se justifican más adelante, después de comparar las soluciones candidatas.

In [64]:
from pathlib import Path
import os
import platform
import hashlib
import warnings

import duckdb
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import seaborn as sns
import sklearn

from IPython.display import Image, Markdown, display
from matplotlib.patches import FancyBboxPatch
from scipy.cluster.hierarchy import dendrogram, fcluster, linkage
from scipy.optimize import linear_sum_assignment
from scipy.sparse import csr_matrix
from scipy.spatial.distance import squareform
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    adjusted_rand_score,
    balanced_accuracy_score,
    classification_report,
    f1_score,
    silhouette_score,
)
from sklearn.model_selection import LeaveOneGroupOut, cross_val_predict
from sklearn.pipeline import Pipeline

PROJECT_DIR = Path.cwd().resolve()
OUTPUT_DIR = PROJECT_DIR / "eda_miranda_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RF_SEEDS = [214363, 42, 2026, 12345, 987654]
CONTROL_SEED = 214363
CONTROL_SENSITIVITY_SEEDS = [9031, 17539]
K_FINAL = 4
N_TREES = 300
MIN_SAMPLES_LEAF = 50

PROFILE_ORDER = [
    "Transaccionales multicanal",
    "Vinculados por nómina",
    "Vinculados por crédito",
    "Desvinculación avanzada",
]

PROFILE_COLORS = {
    "Transaccionales multicanal": "#2F6B9A",
    "Vinculados por nómina": "#2A9D8F",
    "Vinculados por crédito": "#8C6BB1",
    "Desvinculación avanzada": "#D97732",
}

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

dataset_candidates = []
if os.environ.get("DMEYF_DATASET"):
    dataset_candidates.append(Path(os.environ["DMEYF_DATASET"]).expanduser())
dataset_candidates.extend([
    PROJECT_DIR / "data" / "competencia_01_crudo.csv",
    Path(
        "/Users/manuelmarasco/Library/CloudStorage/"
        "GoogleDrive-hola@manuelmarasco.com/Mi unidad/"
        "DMEyF/2026/notebooks/data/competencia_01_crudo.csv"
    ),
])

DATASET_PATH = next((p for p in dataset_candidates if p.exists()), None)
if DATASET_PATH is None:
    raise FileNotFoundError(
        "No se encontró competencia_01_crudo.csv. Definir DMEYF_DATASET "
        "o ubicarlo en ./data/."
    )

versions = pd.DataFrame({
    "componente": ["Python", "DuckDB", "pandas", "NumPy", "SciPy", "scikit-learn", "Matplotlib"],
    "versión": [
        platform.python_version(), duckdb.__version__, pd.__version__, np.__version__,
        scipy.__version__, sklearn.__version__, matplotlib.__version__,
    ],
})
display(versions)
print("Dataset:", DATASET_PATH)
print("Salidas:", OUTPUT_DIR)


,componente,versión
0,Python,3.9.6
1,DuckDB,1.4.5
2,pandas,2.3.3
3,NumPy,2.0.2
4,SciPy,1.13.1
5,scikit-learn,1.6.1
6,Matplotlib,3.9.4


Dataset: /Users/manuelmarasco/Library/CloudStorage/GoogleDrive-hola@manuelmarasco.com/Mi unidad/DMEyF/2026/notebooks/data/competencia_01_crudo.csv
Salidas: /Users/manuelmarasco/Desktop/dmeyf402/eda_miranda_outputs


In [2]:
# Configuración de SQL 
%load_ext sql

%config SqlMagic.autopandas = True
%config SqlMagic.feedback = False
%config SqlMagic.displaycon = False

# Una única conexión compartida por SQL y Python.
con = duckdb.connect(database=":memory:")
%sql con

%sql PRAGMA disable_progress_bar
%sql SET preserve_insertion_order = false

# Nombre utilizado por la plantilla SQL de JupySQL.
archivo_crudo = str(DATASET_PATH)

## 4. Carga y auditoría inicial

Se verifica la unidad de observación, el identificador y el período antes de definir la baja. El dataset crudo se lee sin modificar y se copia a una base DuckDB en memoria.


In [4]:
%%sql

CREATE OR REPLACE TABLE competencia_01_crudo AS
SELECT *
FROM read_csv_auto(
    '{{archivo_crudo}}',
    sample_size = -1
);

,Count
0,983061


### 4.1. Auditoría del identificador

Antes de convertir `numero_de_cliente` se verifica su tipo, la presencia de nulos, la existencia de valores decimales y su rango numérico. Esto permite comprobar que el identificador puede transformarse en un número entero sin alterar su valor.

In [5]:
%%sql

SELECT
    (
        SELECT data_type
        FROM duckdb_columns()
        WHERE table_name = 'competencia_01_crudo'
          AND column_name = 'numero_de_cliente'
    ) AS tipo_actual,

    COUNT(*) FILTER (
        WHERE numero_de_cliente IS NULL
    ) AS ids_nulos,

    COUNT(*) FILTER (
        WHERE numero_de_cliente <> FLOOR(numero_de_cliente)
    ) AS ids_con_decimales,

    COUNT(*) FILTER (
        WHERE ABS(numero_de_cliente) > 9007199254740991
    ) AS ids_fuera_rango_exacto_double,

    MIN(numero_de_cliente) AS id_minimo,
    MAX(numero_de_cliente) AS id_maximo

FROM competencia_01_crudo;

,tipo_actual,ids_nulos,ids_con_decimales,ids_fuera_rango_exacto_double,id_minimo,id_maximo
0,DOUBLE,0,0,0,"12,159,854.00","78,254,956.00"


Los identificadores no contienen nulos, decimales ni valores fuera del rango exacto. Por lo tanto, pueden convertirse de `DOUBLE` a `BIGINT` sin perder información. La tabla original se conserva y se crea una vista normalizada para los joins posteriores.

In [6]:
%%sql

CREATE OR REPLACE VIEW raw AS
SELECT * REPLACE (
    CAST(numero_de_cliente AS BIGINT) AS numero_de_cliente,
    CAST(foto_mes AS BIGINT) AS foto_mes
)
FROM competencia_01_crudo;

,Count


### 4.2. Unidad de observación y duplicados

Se verifica si cada combinación de cliente y mes aparece una sola vez. Si existieran varias filas para el mismo cliente en una misma fotografía mensual, sería necesario resolver esos duplicados antes de construir las trayectorias.

In [7]:
%%sql

WITH cliente_mes AS (
    SELECT
        numero_de_cliente,
        foto_mes,
        COUNT(*) AS cantidad
    FROM competencia_01_crudo
    GROUP BY
        numero_de_cliente,
        foto_mes
)

SELECT
    (SELECT COUNT(*) FROM competencia_01_crudo)
        AS filas_originales,

    COUNT(*) AS pares_cliente_mes,

    COUNT(DISTINCT numero_de_cliente)
        AS clientes,

    COUNT(DISTINCT foto_mes)
        AS meses,

    SUM(
        CASE
            WHEN cantidad > 1 THEN cantidad - 1
            ELSE 0
        END
    ) AS duplicados_cliente_mes

FROM cliente_mes;

,filas_originales,pares_cliente_mes,clientes,meses,duplicados_cliente_mes
0,983061,983061,169727,6,0.00


La cantidad de filas coincide con la cantidad de combinaciones cliente-mes y no se detectan duplicados. Por lo tanto, la unidad de observación es una fila por cliente y por mes, y no es necesario eliminar ni agregar registros antes de construir el panel temporal.

In [8]:
%%sql

SELECT
    foto_mes,
    COUNT(*) AS filas,
    COUNT(DISTINCT numero_de_cliente) AS clientes

FROM competencia_01_crudo

GROUP BY foto_mes
ORDER BY foto_mes;

,foto_mes,filas,clientes
0,202103,162900,162900
1,202104,163284,163284
2,202105,163768,163768
3,202106,164114,164114
4,202107,164348,164348
5,202108,164647,164647


La base contiene seis fotografías mensuales, desde marzo hasta agosto de 2021. En todos los meses la cantidad de filas coincide con la cantidad de clientes únicos, lo que confirma nuevamente que no existen duplicados cliente-mes.

La cantidad de clientes aumenta gradualmente durante la ventana. Por este motivo, una ausencia no debe interpretarse automáticamente como baja: también es necesario considerar clientes que ingresan durante el período, ausencias transitorias y casos que no pueden confirmarse por el final de la observación.

### 4.3. Cobertura temporal

Se calcula la cantidad de registros y de clientes únicos en cada fotografía mensual. Esto permite verificar la cobertura del panel y observar si la población cambia durante la ventana analizada.

## 5. Construcción de la clase temporal

Se construye un panel completo que combina todos los clientes con todos los meses disponibles. De esta manera, la ausencia de un cliente en un mes posterior puede observarse explícitamente.

Siguiendo el procedimiento utilizado en clase, cada registro cliente-mes se clasifica como:

- **BAJA+1:** el cliente no aparece en el mes siguiente.
- **BAJA+2:** aparece en el mes siguiente, pero no dos meses después.
- **CONTINUA:** aparece en los dos meses posteriores.

Estas etiquetas describen la proximidad temporal de la desaparición del cliente; todavía no representan segmentos de comportamiento.


In [9]:
%%sql

CREATE OR REPLACE TABLE competencia_01 AS

WITH periodos AS (
    SELECT DISTINCT foto_mes
    FROM competencia_01_crudo
),

clientes AS (
    SELECT DISTINCT numero_de_cliente
    FROM competencia_01_crudo
),

todo AS (
    SELECT
        numero_de_cliente,
        foto_mes
    FROM clientes
    CROSS JOIN periodos
),

clasificador AS (
    SELECT
        c.*,

        IF(
            c.numero_de_cliente IS NULL,
            0,
            1
        ) AS mes_0,

        CASE
            WHEN LEAD(mes_0, 1) OVER (
                PARTITION BY t.numero_de_cliente
                ORDER BY t.foto_mes
            ) = 0
                THEN 'BAJA+1'

            WHEN LEAD(mes_0, 1) OVER (
                PARTITION BY t.numero_de_cliente
                ORDER BY t.foto_mes
            ) = 1
            AND LEAD(mes_0, 2) OVER (
                PARTITION BY t.numero_de_cliente
                ORDER BY t.foto_mes
            ) = 0
                THEN 'BAJA+2'

            WHEN LEAD(mes_0, 1) OVER (
                PARTITION BY t.numero_de_cliente
                ORDER BY t.foto_mes
            ) = 1
            AND LEAD(mes_0, 2) OVER (
                PARTITION BY t.numero_de_cliente
                ORDER BY t.foto_mes
            ) = 1
                THEN 'CONTINUA'

            ELSE NULL
        END AS clase_ternaria

    FROM todo AS t

    LEFT JOIN competencia_01_crudo AS c
        USING (numero_de_cliente, foto_mes)
)

SELECT *
FROM clasificador
WHERE mes_0 = 1
ORDER BY foto_mes;

,Count
0,983061


### 5.1. Distribución mensual de las clases temporales

In [12]:
%%sql

SELECT
    foto_mes,
    COALESCE(clase_ternaria, 'SIN_ETIQUETA') AS clase,
    COUNT(*) AS clientes

FROM competencia_01

GROUP BY
    foto_mes,
    clase

ORDER BY
    foto_mes,
    clase;

,foto_mes,clase,clientes
0,202103,BAJA+1,1019
1,202103,BAJA+2,960
2,202103,CONTINUA,160921
3,202104,BAJA+1,964
4,202104,BAJA+2,1139
5,202104,CONTINUA,161181
6,202105,BAJA+1,1143
7,202105,BAJA+2,870
8,202105,CONTINUA,161755
9,202106,BAJA+1,874


Los registros sin etiqueta se explican por el final de la ventana temporal. En julio solamente puede verificarse la presencia del mes siguiente, mientras que en agosto no existe una fotografía posterior. Por lo tanto, esos registros no deben interpretarse como errores ni como bajas confirmadas.

### Interpretación de la trayectoria temporal

La clase temporal describe una fila cliente-mes y no clasifica permanentemente al cliente.

Una trayectoria típica puede representarse de la siguiente manera:

| Momento | Situación |
|---|---|
| Dos meses antes de la ausencia | `BAJA+2` |
| Un mes antes de la ausencia | `BAJA+1` |
| Mes siguiente | Ausencia |

Los 4.067 clientes que alguna vez presentan `BAJA+2` también aparecen posteriormente como `BAJA+1`. La ausencia de la combinación `tiene_baja_1 = 0` y `tiene_baja_2 = 1` confirma que:

\[
BAJA+2 es Subconjunto BAJA+1
\]

Los 1.036 clientes que solamente presentan `BAJA+1` no cuentan con una observación previa válida que permita identificarlos también como `BAJA+2`. Esto puede ocurrir por el comienzo de la ventana, por un ingreso posterior o por una discontinuidad en su historia.

En consecuencia, `BAJA+1` contiene todos los eventos de ausencia observados, mientras que `BAJA+2` identifica la parte de esos eventos que dispone de mayor historia previa.

### 5.2. Relación entre BAJA+1 y BAJA+2

In [21]:
%%sql

WITH marcas_cliente AS (
    SELECT
        numero_de_cliente,

        MAX(
            CASE
                WHEN clase_ternaria = 'BAJA+1' THEN 1
                ELSE 0
            END
        ) AS tiene_baja_1,

        MAX(
            CASE
                WHEN clase_ternaria = 'BAJA+2' THEN 1
                ELSE 0
            END
        ) AS tiene_baja_2

    FROM competencia_01

    GROUP BY numero_de_cliente
)

SELECT
    tiene_baja_1,
    tiene_baja_2,
    COUNT(*) AS clientes

FROM marcas_cliente

GROUP BY
    tiene_baja_1,
    tiene_baja_2

ORDER BY
    tiene_baja_1,
    tiene_baja_2;

,tiene_baja_1,tiene_baja_2,clientes
0,0,0,164624
1,1,0,1036
2,1,1,4067


In [27]:
resumen_relacion_bajas = %sql SELECT COUNT(DISTINCT numero_de_cliente) FILTER (WHERE clase_ternaria = 'BAJA+1') AS clientes_baja_1, COUNT(DISTINCT numero_de_cliente) FILTER (WHERE clase_ternaria = 'BAJA+2') AS clientes_baja_2 FROM competencia_01

clientes_baja_1 = int(
    resumen_relacion_bajas.loc[0, "clientes_baja_1"]
)

clientes_baja_2 = int(
    resumen_relacion_bajas.loc[0, "clientes_baja_2"]
)

clientes_solo_baja_1 = (
    clientes_baja_1 - clientes_baja_2
)

fig, ax = plt.subplots(figsize=(12, 5))
ax.axis("off")


def dibujar_caja(x, y, ancho, alto, titulo, detalle, color):
    caja = FancyBboxPatch(
        (x, y),
        ancho,
        alto,
        boxstyle="round,pad=0.02,rounding_size=0.025",
        linewidth=1.8,
        edgecolor=color,
        facecolor=color + "20",
        transform=ax.transAxes,
    )
    ax.add_patch(caja)

    ax.text(
        x + ancho / 2,
        y + alto * 0.65,
        titulo,
        ha="center",
        va="center",
        weight="bold",
        fontsize=12,
        transform=ax.transAxes,
    )

    ax.text(
        x + ancho / 2,
        y + alto * 0.30,
        detalle,
        ha="center",
        va="center",
        fontsize=11,
        transform=ax.transAxes,
    )


dibujar_caja(
    0.04, 0.58, 0.22, 0.22,
    "BAJA+2",
    f"{clientes_baja_2:,} clientes\nDos meses antes",
    "#6C8EBF",
)

dibujar_caja(
    0.04, 0.16, 0.22, 0.22,
    "Solamente BAJA+1",
    f"{clientes_solo_baja_1:,} clientes\nSin BAJA+2 observable",
    "#D9A441",
)

dibujar_caja(
    0.39, 0.38, 0.22, 0.25,
    "BAJA+1",
    f"{clientes_baja_1:,} clientes\nUn mes antes",
    "#2F6B9A",
)

dibujar_caja(
    0.74, 0.38, 0.22, 0.25,
    "AUSENCIA",
    f"{clientes_baja_1:,} eventos\nMes siguiente",
    "#8C6BB1",
)

ax.annotate(
    "",
    xy=(0.39, 0.52),
    xytext=(0.26, 0.67),
    xycoords=ax.transAxes,
    arrowprops=dict(arrowstyle="->", linewidth=2, color="#666666"),
)

ax.annotate(
    "",
    xy=(0.39, 0.45),
    xytext=(0.26, 0.27),
    xycoords=ax.transAxes,
    arrowprops=dict(arrowstyle="->", linewidth=2, color="#666666"),
)

ax.annotate(
    "",
    xy=(0.74, 0.50),
    xytext=(0.61, 0.50),
    xycoords=ax.transAxes,
    arrowprops=dict(arrowstyle="->", linewidth=2, color="#666666"),
)

ax.set_title(
    "Relación temporal entre BAJA+2, BAJA+1 y la ausencia",
    fontsize=15,
    pad=15,
)

fig.tight_layout()

fig.savefig(
    OUTPUT_DIR / "01_relacion_bajas.png",
    dpi=160,
    bbox_inches="tight",
)

plt.close(fig)

![Relación temporal entre BAJA+2, BAJA+1 y ausencia](eda_miranda_outputs/01_relacion_bajas.png)

## 6. Definición de los eventos de salida y de las poblaciones

In [22]:
%%sql

CREATE OR REPLACE TABLE calendario AS
SELECT
    foto_mes,
    ROW_NUMBER() OVER (ORDER BY foto_mes) AS indice_mes
FROM (
    SELECT DISTINCT foto_mes
    FROM raw
)
ORDER BY foto_mes;


CREATE OR REPLACE TABLE panel_presencia AS

WITH clientes AS (
    SELECT DISTINCT numero_de_cliente
    FROM raw
)

SELECT
    c.numero_de_cliente,
    m.foto_mes,
    m.indice_mes,

    CASE
        WHEN r.numero_de_cliente IS NULL THEN 0
        ELSE 1
    END AS presente

FROM clientes AS c

CROSS JOIN calendario AS m

LEFT JOIN raw AS r
    USING (numero_de_cliente, foto_mes);


CREATE OR REPLACE TABLE eventos_baja AS

WITH secuencia AS (
    SELECT
        *,

        LEAD(foto_mes, 1) OVER ventana_cliente
            AS primer_mes_ausente,

        LEAD(presente, 1) OVER ventana_cliente
            AS presente_siguiente,

        COALESCE(
            MAX(presente) OVER (
                PARTITION BY numero_de_cliente
                ORDER BY foto_mes
                ROWS BETWEEN 2 FOLLOWING AND UNBOUNDED FOLLOWING
            ),
            0
        ) AS vuelve_mas_adelante,

        COUNT(*) OVER (
            PARTITION BY numero_de_cliente
            ORDER BY foto_mes
            ROWS BETWEEN 1 FOLLOWING AND UNBOUNDED FOLLOWING
        ) AS meses_observables_despues

    FROM panel_presencia

    WINDOW ventana_cliente AS (
        PARTITION BY numero_de_cliente
        ORDER BY foto_mes
    )
),

transiciones AS (
    SELECT
        numero_de_cliente,
        foto_mes AS ultimo_mes_activo,
        primer_mes_ausente,
        vuelve_mas_adelante,
        meses_observables_despues

    FROM secuencia

    WHERE presente = 1
      AND presente_siguiente = 0
)

SELECT
    *,

    ROW_NUMBER() OVER (
        PARTITION BY numero_de_cliente
        ORDER BY ultimo_mes_activo
    ) AS numero_evento,

    CASE
        WHEN vuelve_mas_adelante = 1
            THEN 'intermitente'

        WHEN meses_observables_despues = 1
            THEN 'censurada'

        ELSE 'estricta'
    END AS poblacion

FROM transiciones;

,Count
0,5103


In [23]:
auditoria_eventos = %sql SELECT poblacion, COUNT(*) AS eventos, COUNT(DISTINCT numero_de_cliente) AS clientes, COUNT(*) - COUNT(DISTINCT (numero_de_cliente, numero_evento)) AS claves_duplicadas FROM eventos_baja GROUP BY poblacion ORDER BY poblacion

display(auditoria_eventos)

assert auditoria_eventos["claves_duplicadas"].sum() == 0

auditoria_clientes_eventos = %sql SELECT COUNT(*) AS clientes_con_mas_de_un_evento FROM (SELECT numero_de_cliente FROM eventos_baja GROUP BY numero_de_cliente HAVING COUNT(*) > 1)

assert auditoria_clientes_eventos.loc[
    0,
    "clientes_con_mas_de_un_evento"
] == 0

eventos_por_mes = %sql SELECT poblacion, ultimo_mes_activo, COUNT(*) AS clientes FROM eventos_baja GROUP BY poblacion, ultimo_mes_activo ORDER BY poblacion, ultimo_mes_activo

display(eventos_por_mes)

,poblacion,eventos,clientes,claves_duplicadas
0,censurada,1103,1103,0
1,estricta,3977,3977,0
2,intermitente,23,23,0


,poblacion,ultimo_mes_activo,clientes
0,censurada,202107,1103
1,estricta,202103,1013
2,estricta,202104,955
3,estricta,202105,1138
4,estricta,202106,871
5,intermitente,202103,6
6,intermitente,202104,9
7,intermitente,202105,5
8,intermitente,202106,3


**Interpretación.** La población principal excluye tanto la censura del final de la ventana como las ausencias transitorias. “Baja estricta” significa ausencia sostenida durante el seguimiento disponible, no una cancelación contractual demostrada. Los censurados se incorporarán como sensibilidad y los intermitentes se estudiarán por separado.


### 6.1. Integración de la clase temporal con la base normalizada

La clase ternaria se calcula sobre el panel completo y se incorpora únicamente a las filas observadas. `BAJA+1` y `BAJA+2` son estados en meses diferentes de una misma trayectoria; no son dos tipos de cliente.


In [25]:
%%sql

CREATE OR REPLACE TABLE datos_etiquetados AS

SELECT
    r.*,
    cal.indice_mes,
    c.clase_ternaria

FROM raw AS r

INNER JOIN competencia_01 AS c
    ON r.numero_de_cliente =
       CAST(c.numero_de_cliente AS BIGINT)

   AND r.foto_mes =
       CAST(c.foto_mes AS BIGINT)

INNER JOIN calendario AS cal
    ON r.foto_mes = cal.foto_mes;


SELECT
    COUNT(*) AS filas,

    COUNT(DISTINCT numero_de_cliente)
        AS clientes,

    COUNT(*) - COUNT(
        DISTINCT (numero_de_cliente, foto_mes)
    ) AS duplicados_cliente_mes,

    COUNT(*) FILTER (
        WHERE clase_ternaria IS NULL
    ) AS filas_sin_etiqueta,

    MIN(indice_mes) AS primer_indice_mes,
    MAX(indice_mes) AS ultimo_indice_mes

FROM datos_etiquetados;

,filas,clientes,duplicados_cliente_mes,filas_sin_etiqueta,primer_indice_mes,ultimo_indice_mes
0,983061,169727,0,327892,1,6


![Clasificación de las ausencias observadas](eda_miranda_outputs/02_poblaciones.png)

### 6.2. Poblaciones resultantes

In [26]:
conteos_eventos = (
    auditoria_eventos
    .set_index("poblacion")["clientes"]
    .to_dict()
)

total_eventos = sum(conteos_eventos.values())

fig, ax = plt.subplots(figsize=(11, 4.8))
ax.axis("off")


def dibujar_poblacion(
    x,
    y,
    ancho,
    alto,
    titulo,
    detalle,
    color,
):
    caja = FancyBboxPatch(
        (x, y),
        ancho,
        alto,
        boxstyle="round,pad=0.02,rounding_size=0.025",
        linewidth=1.8,
        edgecolor=color,
        facecolor=color + "20",
        transform=ax.transAxes,
    )

    ax.add_patch(caja)

    ax.text(
        x + ancho / 2,
        y + alto * 0.65,
        titulo,
        ha="center",
        va="center",
        weight="bold",
        fontsize=12,
        transform=ax.transAxes,
    )

    ax.text(
        x + ancho / 2,
        y + alto * 0.30,
        detalle,
        ha="center",
        va="center",
        fontsize=11,
        transform=ax.transAxes,
    )


dibujar_poblacion(
    0.35,
    0.68,
    0.30,
    0.20,
    "Ausencias observadas",
    f"{total_eventos:,} clientes",
    "#444444",
)

poblaciones = [
    (
        0.03,
        "Bajas estrictas",
        "estricta",
        "#2F6B9A",
    ),
    (
        0.36,
        "Casos censurados",
        "censurada",
        "#D9A441",
    ),
    (
        0.69,
        "Intermitentes",
        "intermitente",
        "#8C6BB1",
    ),
]

for x, titulo, clave, color in poblaciones:
    cantidad = int(
        conteos_eventos.get(clave, 0)
    )

    porcentaje = (
        100 * cantidad / total_eventos
    )

    ax.annotate(
        "",
        xy=(x + 0.14, 0.49),
        xytext=(0.50, 0.68),
        xycoords=ax.transAxes,
        arrowprops=dict(
            arrowstyle="->",
            color="#666666",
            linewidth=1.5,
        ),
    )

    dibujar_poblacion(
        x,
        0.20,
        0.28,
        0.22,
        titulo,
        f"{cantidad:,} clientes · {porcentaje:.1f}%",
        color,
    )

ax.set_title(
    "Clasificación de las ausencias observadas",
    fontsize=15,
    pad=10,
)

fig.tight_layout()

ruta_poblaciones = (
    OUTPUT_DIR / "02_poblaciones.png"
)

fig.savefig(
    ruta_poblaciones,
    dpi=160,
    bbox_inches="tight",
)

plt.close(fig)

### 7.1. Distribución de los valores nulos

Se calcula la cantidad y el porcentaje de valores nulos para cada variable. La auditoría se realiza sobre todas las columnas, sin limitar previamente el análisis a una familia de productos.

In [28]:
%%sql

CREATE OR REPLACE TABLE auditoria_nulos AS

WITH resumen_ancho AS (
    SELECT
        COUNT(*) AS total_filas,
        COUNT(*) - COUNT(COLUMNS(*))
    FROM raw
),

resumen_largo AS (
    UNPIVOT resumen_ancho

    ON COLUMNS(
        * EXCLUDE (total_filas)
    )

    INTO
        NAME variable
        VALUE cantidad_nulos
)

SELECT
    variable,
    cantidad_nulos,

    ROUND(
        100.0 * cantidad_nulos / total_filas,
        2
    ) AS porcentaje_nulos

FROM resumen_largo

WHERE cantidad_nulos > 0;


SELECT *
FROM auditoria_nulos

ORDER BY
    cantidad_nulos DESC,
    variable;

,variable,cantidad_nulos,porcentaje_nulos
0,Master_Finiciomora,978240,99.51
1,Visa_Finiciomora,969739,98.64
2,Master_cadelantosefectivo,579470,58.95
3,Master_cconsumos,579470,58.95
4,Master_madelantodolares,579470,58.95
5,Master_madelantopesos,579470,58.95
6,Master_mconsumosdolares,579470,58.95
7,Master_mconsumospesos,579470,58.95
8,Master_mconsumototal,579470,58.95
9,Master_mpagosdolares,579470,58.95


### 7.2. Patrones de nulidad por familia

In [29]:
%%sql

SELECT
    CASE
        WHEN starts_with(variable, 'Master_')
            THEN 'Master'

        WHEN starts_with(variable, 'Visa_')
            THEN 'Visa'

        WHEN starts_with(variable, 'mtarjeta_')
            THEN 'Tarjetas agregadas'

        ELSE 'Otras variables'
    END AS familia,

    cantidad_nulos,
    porcentaje_nulos,

    COUNT(*) AS cantidad_variables,

    STRING_AGG(
        variable,
        ', '
        ORDER BY variable
    ) AS variables

FROM auditoria_nulos

GROUP BY
    familia,
    cantidad_nulos,
    porcentaje_nulos

ORDER BY
    cantidad_nulos DESC,
    familia;

,familia,cantidad_nulos,porcentaje_nulos,cantidad_variables,variables
0,Master,978240,99.51,1,Master_Finiciomora
1,Visa,969739,98.64,1,Visa_Finiciomora
2,Master,579470,58.95,9,"Master_cadelantosefectivo, Master_cconsumos, M..."
3,Visa,127760,13.00,9,"Visa_cadelantosefectivo, Visa_cconsumos, Visa_..."
4,Master,100127,10.19,1,Master_fultimo_cierre
5,Master,99644,10.14,11,"Master_Fvencimiento, Master_delinquency, Maste..."
6,Visa,47536,4.84,1,Visa_fultimo_cierre
7,Visa,47186,4.80,11,"Visa_Fvencimiento, Visa_delinquency, Visa_fech..."
8,Tarjetas agregadas,29178,2.97,1,mtarjeta_master_descuentos
9,Tarjetas agregadas,20694,2.11,1,mtarjeta_visa_descuentos


La auditoría detecta valores nulos exclusivamente en variables asociadas con tarjetas. Sin embargo, los porcentajes forman patrones diferentes, por lo que no se asume que todos los nulos tengan el mismo significado.

En particular, se distingue entre:

- ausencia de un producto;
- producto existente sin consumo informado;
- consumo expresamente igual a cero;
- ausencia de un evento, como el inicio de una mora;
- campos cuyo monto no fue informado aunque exista un indicador de actividad.

### 7.3. Diferencia entre tenencia, consumo nulo y consumo cero

In [31]:
%%sql

WITH resumen_tarjetas AS (

    SELECT
        'Master' AS tarjeta,

        COUNT(*) FILTER (
            WHERE Master_mlimitecompra IS NOT NULL
        ) AS registros_con_limite,

        COUNT(*) FILTER (
            WHERE Master_mlimitecompra IS NOT NULL
              AND Master_mconsumototal IS NULL
        ) AS consumos_nulos,

        COUNT(*) FILTER (
            WHERE Master_mlimitecompra IS NOT NULL
              AND Master_mconsumototal = 0
        ) AS consumos_cero,

        COUNT(*) FILTER (
            WHERE Master_mlimitecompra IS NOT NULL
              AND Master_mconsumototal > 0
        ) AS consumos_positivos,

        COUNT(*) FILTER (
            WHERE Master_mlimitecompra IS NOT NULL
              AND Master_mconsumototal < 0
        ) AS consumos_negativos

    FROM raw

    UNION ALL

    SELECT
        'Visa',

        COUNT(*) FILTER (
            WHERE Visa_mlimitecompra IS NOT NULL
        ),

        COUNT(*) FILTER (
            WHERE Visa_mlimitecompra IS NOT NULL
              AND Visa_mconsumototal IS NULL
        ),

        COUNT(*) FILTER (
            WHERE Visa_mlimitecompra IS NOT NULL
              AND Visa_mconsumototal = 0
        ),

        COUNT(*) FILTER (
            WHERE Visa_mlimitecompra IS NOT NULL
              AND Visa_mconsumototal > 0
        ),

        COUNT(*) FILTER (
            WHERE Visa_mlimitecompra IS NOT NULL
              AND Visa_mconsumototal < 0
        )

    FROM raw
)

SELECT
    *,

    ROUND(
        100.0 * consumos_nulos
        / NULLIF(registros_con_limite, 0),
        2
    ) AS porcentaje_nulo,

    ROUND(
        100.0 * consumos_cero
        / NULLIF(registros_con_limite, 0),
        2
    ) AS porcentaje_cero,

    ROUND(
        100.0 * consumos_positivos
        / NULLIF(registros_con_limite, 0),
        2
    ) AS porcentaje_positivo

FROM resumen_tarjetas

ORDER BY tarjeta;

,tarjeta,registros_con_limite,consumos_nulos,consumos_cero,consumos_positivos,consumos_negativos,porcentaje_nulo,porcentaje_cero,porcentaje_positivo
0,Master,883417,479826,135213,267209,1169,54.31,15.31,30.25
1,Visa,935875,80574,89188,757298,8815,8.61,9.53,80.92


La existencia de un límite de compra indica que el producto está registrado. Sin embargo, dentro de esos registros coexisten consumos nulos, iguales a cero, positivos y negativos.

Por lo tanto, un consumo nulo no puede interpretarse automáticamente como ausencia de tarjeta ni reemplazarse indistintamente por cero. Los valores negativos se conservan, dado que pueden representar ajustes, devoluciones o reversas.

### 7.4. Relación entre descuentos y montos nulos

La auditoría permite adoptar las siguientes decisiones:

1. No se eliminan filas por contener valores nulos.
2. No se reemplazan masivamente los nulos por cero.
3. Los campos administrativos, fechas, estados y códigos de Visa y Master no se utilizan para construir la segmentación.
4. Los consumos nulos se mantienen diferenciados de los consumos iguales a cero.
5. Los montos faltantes de descuentos se conservan como información ausente, dado que existen registros donde el evento ocurrió.
6. Las variables utilizadas por los modelos se imputan dentro del proceso de modelado y, cuando resulte relevante, se acompaña la imputación con un indicador de dato faltante.
7. Los valores negativos se conservan porque pueden representar saldos deudores, ajustes, devoluciones o reversas.

In [35]:
%%sql

SELECT
    'Master' AS tarjeta,

    COUNT(*) FILTER (
        WHERE mtarjeta_master_descuentos IS NULL
          AND ctarjeta_master_descuentos = 0
    ) AS monto_nulo_sin_descuentos,

    COUNT(*) FILTER (
        WHERE mtarjeta_master_descuentos IS NULL
          AND ctarjeta_master_descuentos > 0
    ) AS monto_nulo_con_descuentos,

    COUNT(*) FILTER (
        WHERE mtarjeta_master_descuentos IS NOT NULL
          AND ctarjeta_master_descuentos = 0
    ) AS monto_informado_sin_descuentos,

    COUNT(*) FILTER (
        WHERE mtarjeta_master_descuentos IS NOT NULL
          AND ctarjeta_master_descuentos > 0
    ) AS monto_informado_con_descuentos

FROM raw

UNION ALL

SELECT
    'Visa',

    COUNT(*) FILTER (
        WHERE mtarjeta_visa_descuentos IS NULL
          AND ctarjeta_visa_descuentos = 0
    ),

    COUNT(*) FILTER (
        WHERE mtarjeta_visa_descuentos IS NULL
          AND ctarjeta_visa_descuentos > 0
    ),

    COUNT(*) FILTER (
        WHERE mtarjeta_visa_descuentos IS NOT NULL
          AND ctarjeta_visa_descuentos = 0
    ),

    COUNT(*) FILTER (
        WHERE mtarjeta_visa_descuentos IS NOT NULL
          AND ctarjeta_visa_descuentos > 0
    )

FROM raw;

,tarjeta,monto_nulo_sin_descuentos,monto_nulo_con_descuentos,monto_informado_sin_descuentos,monto_informado_con_descuentos
0,Master,0,29178,936557,17326
1,Visa,0,20694,923581,38786


Todos los montos nulos de descuentos aparecen en registros donde la cantidad de descuentos es positiva. En consecuencia, el evento existió, pero su monto no fue informado.

Este patrón es compatible con una ausencia parcial de información o con una regla particular del sistema de origen. Estos valores no se reemplazan por cero. Si la variable se utiliza posteriormente, la ausencia del monto deberá conservarse mediante imputación y un indicador de dato faltante.

### 7.5. Estabilidad temporal de los patrones de nulidad

In [37]:
%%sql

SELECT
    foto_mes,

    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE Master_Finiciomora IS NOT NULL
        ) / COUNT(*),
        2
    ) AS master_con_inicio_mora_pct,

    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE Visa_Finiciomora IS NOT NULL
        ) / COUNT(*),
        2
    ) AS visa_con_inicio_mora_pct,

    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE Master_fultimo_cierre IS NULL
        ) / COUNT(*),
        2
    ) AS master_cierre_nulo_pct,

    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE Visa_fultimo_cierre IS NULL
        ) / COUNT(*),
        2
    ) AS visa_cierre_nulo_pct,

    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE mtarjeta_master_descuentos IS NULL
              AND ctarjeta_master_descuentos > 0
        )
        / NULLIF(
            COUNT(*) FILTER (
                WHERE ctarjeta_master_descuentos > 0
            ),
            0
        ),
        2
    ) AS master_monto_descuento_nulo_pct,

    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE mtarjeta_visa_descuentos IS NULL
              AND ctarjeta_visa_descuentos > 0
        )
        / NULLIF(
            COUNT(*) FILTER (
                WHERE ctarjeta_visa_descuentos > 0
            ),
            0
        ),
        2
    ) AS visa_monto_descuento_nulo_pct

FROM raw

GROUP BY foto_mes
ORDER BY foto_mes;

,foto_mes,master_con_inicio_mora_pct,visa_con_inicio_mora_pct,master_cierre_nulo_pct,visa_cierre_nulo_pct,master_monto_descuento_nulo_pct,visa_monto_descuento_nulo_pct
0,202103,0.69,0.91,10.37,4.86,64.55,30.41
1,202104,0.60,0.87,10.34,4.92,70.16,29.70
2,202105,0.50,2.65,10.31,4.97,68.28,38.50
3,202106,0.40,0.57,10.14,4.97,57.97,29.20
4,202107,0.42,0.60,9.71,4.42,49.30,43.87
5,202108,0.34,2.53,10.25,4.88,68.06,38.99


Los patrones de nulidad aparecen durante toda la ventana temporal. Los porcentajes de cierres nulos se mantienen estables, mientras que el inicio de mora informado representa un evento poco frecuente.

La ausencia del monto de descuentos también se repite en todos los meses. Por lo tanto, no se observa evidencia de un error aislado de una única carga, sino de una regla sistemática de registro o de una ausencia parcial de información.

Sin un diccionario de datos no puede determinarse la causa exacta. En consecuencia, estos nulos se conservan como información faltante y no se reemplazan automáticamente por cero.

### 7.6. Validez numérica y valores negativos

Se verifica que las principales variables monetarias no contengan valores infinitos. Los valores negativos se contabilizan, pero no se eliminan automáticamente porque pueden representar saldos deudores, pérdidas, ajustes o reversas.

In [38]:
%%sql

SELECT
    COUNT(*) FILTER (
        WHERE NOT isfinite(mcuentas_saldo)
    ) AS infinitos_saldo,

    COUNT(*) FILTER (
        WHERE NOT isfinite(mrentabilidad)
    ) AS infinitos_rentabilidad,

    COUNT(*) FILTER (
        WHERE NOT isfinite(mpayroll)
    ) AS infinitos_payroll,

    COUNT(*) FILTER (
        WHERE NOT isfinite(mcomisiones_mantenimiento)
    ) AS infinitos_mantenimiento,

    COUNT(*) FILTER (
        WHERE NOT isfinite(mcomisiones_otras)
    ) AS infinitos_otras_comisiones,

    COUNT(*) FILTER (
        WHERE mcuentas_saldo < 0
    ) AS saldos_negativos,

    COUNT(*) FILTER (
        WHERE mrentabilidad < 0
    ) AS rentabilidades_negativas,

    COUNT(*) FILTER (
        WHERE mcomisiones_mantenimiento < 0
    ) AS mantenimientos_negativos,

    COUNT(*) FILTER (
        WHERE mcomisiones_otras < 0
    ) AS otras_comisiones_negativas

FROM raw;

,infinitos_saldo,infinitos_rentabilidad,infinitos_payroll,infinitos_mantenimiento,infinitos_otras_comisiones,saldos_negativos,rentabilidades_negativas,mantenimientos_negativos,otras_comisiones_negativas
0,0,0,0,0,0,191268,227560,13762,17213


### 7.7. Decisiones de tratamiento

La auditoría de calidad permite adoptar las siguientes decisiones:

1. No se eliminan filas por contener valores nulos.
2. No se reemplazan masivamente los nulos por cero.
3. Los consumos nulos se mantienen diferenciados de los consumos expresamente iguales a cero.
4. Los montos faltantes de descuentos se conservan como información ausente, dado que existen registros donde el evento ocurrió.
5. Los campos administrativos, fechas, estados y códigos de Visa y Master no se utilizan para construir la segmentación.
6. Las variables utilizadas por los modelos se imputan dentro del proceso de modelado y, cuando corresponde, se acompaña la imputación con indicadores de dato faltante.
7. Los valores negativos se conservan porque son valores numéricos finitos y pueden representar saldos deudores, pérdidas, ajustes, devoluciones o reversas.
8. Los importes identificados como pesos y dólares no se suman entre sí.
9. La matriz final entregada a los modelos se audita nuevamente para comprobar la ausencia de nulos e infinitos después del preprocesamiento.

Con estas decisiones se preserva la información original sin introducir supuestos de negocio que no pueden validarse con el diccionario disponible.

## 8. Comparación descriptiva: bajas y continuadores

Esta comparación responde si, en el último mes activo, quienes dejan el paquete muestran menor vinculación o uso que clientes que continúan. Es una asociación descriptiva; no identifica causas.


In [39]:
%%sql

CREATE OR REPLACE TABLE comparacion_t0 AS

SELECT
    e.numero_de_cliente,
    e.ultimo_mes_activo AS foto_mes_ancla,
    'Baja estricta' AS grupo,

    r.* EXCLUDE (
        numero_de_cliente,
        foto_mes
    )

FROM eventos_baja AS e

INNER JOIN raw AS r
    ON e.numero_de_cliente =
       r.numero_de_cliente

   AND e.ultimo_mes_activo =
       r.foto_mes

WHERE e.poblacion = 'estricta'


UNION ALL BY NAME


SELECT
    d.numero_de_cliente,
    d.foto_mes AS foto_mes_ancla,
    'Continúa' AS grupo,

    d.* EXCLUDE (
        numero_de_cliente,
        foto_mes,
        indice_mes,
        clase_ternaria
    )

FROM datos_etiquetados AS d

WHERE d.clase_ternaria = 'CONTINUA';


CREATE OR REPLACE TABLE resumen_comparacion_t0 AS

SELECT
    foto_mes_ancla,
    grupo,

    COUNT(*) AS clientes,

    MEDIAN(cproductos)
        AS productos_mediana,

    MEDIAN(ctrx_quarter)
        AS transacciones_mediana,

    AVG(
        CAST(cpayroll_trx > 0 AS DOUBLE)
    ) * 100 AS porcentaje_payroll,

    AVG(
        CAST(
            (
                ctarjeta_visa_transacciones
                + ctarjeta_master_transacciones
            ) > 0
            AS DOUBLE
        )
    ) * 100 AS porcentaje_usa_tarjeta,

    AVG(
        CAST(
            (
                chomebanking_transacciones
                + cmobile_app_trx
            ) > 0
            AS DOUBLE
        )
    ) * 100 AS porcentaje_digital,

    AVG(
        CAST(
            ctransferencias_emitidas > 0
            AS DOUBLE
        )
    ) * 100 AS porcentaje_transfiere,

    MEDIAN(mcuentas_saldo)
        AS saldo_mediano,

    MEDIAN(mrentabilidad)
        AS rentabilidad_mediana

FROM comparacion_t0

GROUP BY
    foto_mes_ancla,
    grupo;


SELECT *
FROM resumen_comparacion_t0

ORDER BY
    foto_mes_ancla,
    grupo;

,foto_mes_ancla,grupo,clientes,productos_mediana,transacciones_mediana,porcentaje_payroll,porcentaje_usa_tarjeta,porcentaje_digital,porcentaje_transfiere,saldo_mediano,rentabilidad_mediana
0,202103,Baja estricta,1013,6.00,18.00,4.24,43.83,68.61,22.61,"-1,751.24","1,755.33"
1,202103,Continúa,160921,7.00,105.00,54.63,86.69,94.28,60.94,"35,036.93","1,239.26"
2,202104,Baja estricta,955,6.00,18.00,4.61,43.87,69.21,23.56,"-1,935.35","1,733.09"
3,202104,Continúa,161181,7.00,106.00,54.72,86.98,94.30,60.36,"34,084.74","1,112.37"
4,202105,Baja estricta,1138,6.00,16.00,5.80,38.05,64.59,18.80,"-2,156.07","1,823.28"
5,202105,Continúa,161755,7.00,106.00,55.56,86.98,94.45,60.66,"33,728.59","1,211.22"
6,202106,Baja estricta,871,6.00,23.00,5.74,48.34,75.43,25.37,"-1,910.55","1,769.30"
7,202106,Continúa,162142,7.00,106.00,55.86,87.45,94.45,61.95,"49,156.99","1,350.80"


In [40]:
resumen_comparacion = %sql SELECT * FROM resumen_comparacion_t0 ORDER BY foto_mes_ancla, grupo

fig, axes = plt.subplots(
    2,
    2,
    figsize=(13, 8),
    sharex=True,
)

metricas = [
    (
        "transacciones_mediana",
        "Transacciones trimestrales",
        "Cantidad",
    ),
    (
        "porcentaje_payroll",
        "Clientes con payroll",
        "%",
    ),
    (
        "porcentaje_usa_tarjeta",
        "Clientes que usan tarjeta",
        "%",
    ),
    (
        "saldo_mediano",
        "Saldo mediano de cuentas",
        "$ nominales",
    ),
]

for ax, (
    variable,
    titulo,
    unidad,
) in zip(
    axes.ravel(),
    metricas,
):
    for grupo, color in [
        ("Baja estricta", "#D97732"),
        ("Continúa", "#2F6B9A"),
    ]:
        datos_grupo = resumen_comparacion[
            resumen_comparacion["grupo"] == grupo
        ]

        ax.plot(
            datos_grupo[
                "foto_mes_ancla"
            ].astype(str),
            datos_grupo[variable],
            marker="o",
            linewidth=2.2,
            label=grupo,
            color=color,
        )

    ax.set_title(titulo)
    ax.set_ylabel(unidad)
    ax.set_xlabel("Mes de anclaje")

axes[0, 0].legend(
    frameon=False
)

fig.suptitle(
    "Bajas estrictas frente a clientes que continúan",
    fontsize=15,
)

fig.text(
    0.5,
    0.01,
    (
        "Comparación realizada dentro de cada mes. "
        "Los denominadores se informan en la tabla anterior."
    ),
    ha="center",
    fontsize=9,
)

fig.tight_layout(
    rect=[0, 0.03, 1, 0.96]
)

ruta_comparacion = (
    OUTPUT_DIR
    / "03_bajas_vs_continuadores.png"
)

fig.savefig(
    ruta_comparacion,
    dpi=160,
    bbox_inches="tight",
)

plt.close(fig)

![Comparación entre bajas estrictas y continuadores](eda_miranda_outputs/03_bajas_vs_continuadores.png)

**Conclusión descriptiva.** Las bajas estrictas llegan a su último mes activo con menos productos, una actividad transaccional considerablemente menor, menor presencia de payroll, menor uso de tarjetas y canales digitales, menos transferencias y una posición de saldo más débil.

El patrón se repite dentro de cada mes, lo que reduce el riesgo de atribuirlo exclusivamente a inflación o estacionalidad. Sin embargo, la comparación es descriptiva y no demuestra causalidad.

La rentabilidad mensual observada es mayor entre las bajas. Este resultado no se interpreta todavía como mayor valor económico, dado que puede estar influido por cargos de mantenimiento y otras comisiones. Su composición se analiza posteriormente.

## 9. Feature engineering temporal


El comportamiento de cada cliente se representa utilizando tanto su situación actual como su evolución previa. Se aplican las funciones analíticas vistas en la materia:

- `LAG()` para recuperar valores de meses anteriores;
- deltas para medir cambios recientes;
- promedio, mínimo, máximo y rango en una ventana móvil;
- `REGR_SLOPE()` para estimar la tendencia temporal;
- indicadores de abandono para detectar actividades presentes en el pasado y ausentes en el mes actual.

Las variables se calculan primero sobre todo el panel de observaciones activas. Posteriormente se selecciona el último mes activo de cada baja.

Cada ventana de tres meses contiene el mes actual y los dos meses anteriores. No se utilizan observaciones futuras dentro de las features. Las filas sin historia suficiente se conservan inicialmente y se identifican mediante controles específicos.

La tabla resultante mantiene las variables originales y agrega las variables temporales generadas sobre las principales dimensiones de comportamiento.


In [41]:
campos_temporales = [
    "ctrx_quarter",
    "mcuentas_saldo",
    "cproductos",
    "mrentabilidad",
    "mcomisiones_mantenimiento",
    "mpayroll",
    "tarjetas_consumo",
    "mtransferencias_emitidas",
]

campos_eventos = [
    "cpayroll_trx",
    "tarjetas_cantidad",
    "tarjetas_transacciones",
    "debitos_automaticos",
    "pagos_servicios",
    "transacciones_digitales",
    "cantidad_creditos",
    "cantidad_inversiones",
    "cantidad_seguros",
]

expresiones_derivadas = []

for campo in campos_temporales:
    expresiones_derivadas.extend([
        f"LAG({campo}, 1) OVER w AS {campo}_lag_1",
        f"LAG({campo}, 2) OVER w AS {campo}_lag_2",

        (
            f"{campo} - LAG({campo}, 1) OVER w "
            f"AS {campo}_delta_1"
        ),

        (
            f"{campo} - LAG({campo}, 2) OVER w "
            f"AS {campo}_delta_2"
        ),

        (
            f"AVG({campo}) OVER w3 "
            f"AS {campo}_media_3"
        ),

        (
            f"MIN({campo}) OVER w3 "
            f"AS {campo}_minimo_3"
        ),

        (
            f"MAX({campo}) OVER w3 "
            f"AS {campo}_maximo_3"
        ),

        (
            f"MAX({campo}) OVER w3 "
            f"- MIN({campo}) OVER w3 "
            f"AS {campo}_rango_3"
        ),

        (
            f"REGR_SLOPE({campo}, indice_mes) OVER w3 "
            f"AS {campo}_pendiente_3"
        ),
    ])

for campo in campos_eventos:
    expresiones_derivadas.extend([
        f"LAG({campo}, 1) OVER w AS {campo}_lag_1",
        f"LAG({campo}, 2) OVER w AS {campo}_lag_2",

        (
            f"{campo} - LAG({campo}, 1) OVER w "
            f"AS {campo}_delta_1"
        ),

        (
            f"{campo} - LAG({campo}, 2) OVER w "
            f"AS {campo}_delta_2"
        ),

        (
            "CASE "
            f"WHEN {campo} = 0 "
            "AND GREATEST("
            f"COALESCE(LAG({campo}, 1) OVER w, 0), "
            f"COALESCE(LAG({campo}, 2) OVER w, 0)"
            ") > 0 "
            "THEN 1 ELSE 0 END "
            f"AS abandono_{campo}_3m"
        ),
    ])

sql_derivadas = ",\n".join(
    expresiones_derivadas
)

print(
    "Variables temporales generadas:",
    len(expresiones_derivadas),
)

Variables temporales generadas: 117


In [42]:
%%sql

CREATE OR REPLACE TABLE fe_panel AS

WITH base AS (
    SELECT
        d.*,

        d.ctarjeta_visa
            + d.ctarjeta_master
            AS tarjetas_cantidad,

        d.ctarjeta_visa_transacciones
            + d.ctarjeta_master_transacciones
            AS tarjetas_transacciones,

        d.mtarjeta_visa_consumo
            + d.mtarjeta_master_consumo
            AS tarjetas_consumo,

        d.ccuenta_debitos_automaticos
            + d.ctarjeta_visa_debitos_automaticos
            + d.ctarjeta_master_debitos_automaticos
            AS debitos_automaticos,

        d.cpagodeservicios
            + d.cpagomiscuentas
            AS pagos_servicios,

        d.chomebanking_transacciones
            + d.cmobile_app_trx
            AS transacciones_digitales,

        d.cprestamos_personales
            + d.cprestamos_prendarios
            + d.cprestamos_hipotecarios
            AS cantidad_creditos,

        d.cplazo_fijo
            + d.cinversion1
            + d.cinversion2
            + d.cforex
            AS cantidad_inversiones,

        d.cseguro_vida
            + d.cseguro_auto
            + d.cseguro_vivienda
            + d.cseguro_accidentes_personales
            AS cantidad_seguros

    FROM datos_etiquetados AS d
)

SELECT
    base.*,

    COUNT(*) OVER w
        AS observaciones_hasta_mes,

    indice_mes
        - LAG(indice_mes, 1) OVER w
        AS salto_1,

    indice_mes
        - LAG(indice_mes, 2) OVER w
        AS salto_2,

    {{sql_derivadas}}

FROM base

WINDOW
    w AS (
        PARTITION BY numero_de_cliente
        ORDER BY foto_mes
    ),

    w3 AS (
        PARTITION BY numero_de_cliente
        ORDER BY foto_mes
        ROWS BETWEEN 2 PRECEDING
                 AND CURRENT ROW
    );

,Count
0,983061


In [43]:
%%sql

SELECT
    COUNT(*) AS filas,

    COUNT(DISTINCT numero_de_cliente)
        AS clientes,

    COUNT(DISTINCT foto_mes)
        AS meses,

    COUNT(*) - COUNT(
        DISTINCT (numero_de_cliente, foto_mes)
    ) AS duplicados_cliente_mes,

    COUNT(*) FILTER (
        WHERE observaciones_hasta_mes >= 3
    ) AS filas_con_tres_observaciones,

    COUNT(*) FILTER (
        WHERE observaciones_hasta_mes >= 3
          AND salto_1 = 1
          AND salto_2 = 2
    ) AS filas_con_tres_meses_consecutivos,

    (
        SELECT COUNT(*)
        FROM duckdb_columns()
        WHERE table_name = 'fe_panel'
    ) AS cantidad_columnas

FROM fe_panel;

,filas,clientes,meses,duplicados_cliente_mes,filas_con_tres_observaciones,filas_con_tres_meses_consecutivos,cantidad_columnas
0,983061,169727,6,0,646032,645999,285


La auditoría es correcta:
- hay 983.061 filas y ningún duplicado;
- 646.032 filas poseen al menos tres observaciones;
- 645.999 tienen además tres meses consecutivos;
- solo 33 filas tienen tres observaciones con algún salto;
- las 285 columnas son coherentes con las variables originales, agregados y 117 features temporales.

La diferencia de 33 registros entre las filas con tres observaciones y las filas con tres meses consecutivos demuestra que una cantidad suficiente de observaciones no garantiza una trayectoria mensual continua.

Para la segmentación principal se exige que los tres meses sean consecutivos. De esta manera, los lags y deltas representan verdaderamente uno y dos meses hacia atrás.

## 10. Muestra de referencia

La segmentación se estima sobre bajas con tres meses consecutivos. Para cada mes de anclaje se selecciona igual cantidad de continuadores, sin repetir clientes entre mayo y junio. La selección de controles es determinística y puede variarse para medir sensibilidad.


In [44]:
%%sql

CREATE OR REPLACE TABLE bajas_estrictas_fe AS

SELECT
    e.numero_evento,
    e.ultimo_mes_activo AS foto_mes_ancla,
    e.primer_mes_ausente,
    e.poblacion,

    f.* EXCLUDE (foto_mes),

    CASE
        WHEN f.observaciones_hasta_mes >= 3
         AND f.salto_1 = 1
         AND f.salto_2 = 2
            THEN 1
        ELSE 0
    END AS historia_3m_completa

FROM eventos_baja AS e

INNER JOIN fe_panel AS f
    ON e.numero_de_cliente =
       f.numero_de_cliente

   AND e.ultimo_mes_activo =
       f.foto_mes

WHERE e.poblacion = 'estricta';


CREATE OR REPLACE TABLE candidatos_control AS

SELECT *
FROM fe_panel

WHERE foto_mes IN (
    202105,
    202106
)

  AND clase_ternaria = 'CONTINUA'

  AND observaciones_hasta_mes >= 3
  AND salto_1 = 1
  AND salto_2 = 2;


SELECT
    foto_mes_ancla,
    historia_3m_completa,
    COUNT(*) AS clientes

FROM bajas_estrictas_fe

GROUP BY
    foto_mes_ancla,
    historia_3m_completa

ORDER BY
    foto_mes_ancla,
    historia_3m_completa;

,foto_mes_ancla,historia_3m_completa,clientes
0,202103,0,1013
1,202104,0,955
2,202105,0,9
3,202105,1,1129
4,202106,0,9
5,202106,1,862


In [45]:
strict_all = %sql SELECT * FROM bajas_estrictas_fe ORDER BY foto_mes_ancla, numero_de_cliente

strict_complete = (
    strict_all[
        strict_all["historia_3m_completa"] == 1
    ]
    .copy()
)

control_candidates = %sql SELECT * FROM candidatos_control ORDER BY foto_mes, numero_de_cliente

targets_mes = (
    strict_complete
    .groupby("foto_mes_ancla")
    .size()
    .to_dict()
)


def seleccionar_controles(seed):
    generador = np.random.default_rng(seed)

    candidatos = control_candidates.copy()

    candidatos["_orden_aleatorio"] = (
        generador.random(
            len(candidatos)
        )
    )

    ids_bajas = set(
        strict_complete[
            "numero_de_cliente"
        ].astype("int64")
    )

    candidatos = candidatos[
        ~candidatos[
            "numero_de_cliente"
        ].astype("int64").isin(ids_bajas)
    ]

    elegidos = []
    ids_utilizados = set()

    for mes in sorted(targets_mes):
        cantidad_objetivo = int(
            targets_mes[mes]
        )

        disponibles = candidatos[
            (
                candidatos["foto_mes"] == mes
            )
            & (
                ~candidatos[
                    "numero_de_cliente"
                ].astype("int64").isin(
                    ids_utilizados
                )
            )
        ]

        seleccion = (
            disponibles
            .nsmallest(
                cantidad_objetivo,
                "_orden_aleatorio",
            )
            .drop(
                columns="_orden_aleatorio"
            )
        )

        assert (
            len(seleccion)
            == cantidad_objetivo
        )

        elegidos.append(seleccion)

        ids_utilizados.update(
            seleccion[
                "numero_de_cliente"
            ].astype("int64")
        )

    resultado = pd.concat(
        elegidos,
        ignore_index=True,
    )

    assert resultado[
        "numero_de_cliente"
    ].is_unique

    return resultado


controls_main = seleccionar_controles(
    CONTROL_SEED
)

auditoria_muestra = pd.concat(
    [
        (
            strict_complete
            .groupby("foto_mes_ancla")
            .size()
            .rename("clientes")
            .reset_index()
            .rename(
                columns={
                    "foto_mes_ancla": "foto_mes"
                }
            )
            .assign(
                grupo="Baja estricta"
            )
        ),

        (
            controls_main
            .groupby("foto_mes")
            .size()
            .rename("clientes")
            .reset_index()
            .assign(
                grupo="Continúa"
            )
        ),
    ],
    ignore_index=True,
)[
    [
        "foto_mes",
        "grupo",
        "clientes",
    ]
].sort_values(
    [
        "foto_mes",
        "grupo",
    ]
)

print(
    "Bajas estrictas:",
    len(strict_all),
)

print(
    "Con tres meses consecutivos:",
    len(strict_complete),
)

print(
    "Con historia parcial:",
    len(strict_all)
    - len(strict_complete),
)

print(
    "Controles seleccionados:",
    len(controls_main),
)

display(auditoria_muestra)

assert len(strict_all) == 3977
assert len(strict_complete) == 1991
assert len(controls_main) == 1991

Bajas estrictas: 3977
Con tres meses consecutivos: 1991
Con historia parcial: 1986
Controles seleccionados: 1991


,foto_mes,grupo,clientes
0,202105,Baja estricta,1129
2,202105,Continúa,1129
1,202106,Baja estricta,862
3,202106,Continúa,862


La muestra quedó correctamente balanceada:
- 1.991 bajas estrictas;
- 1.991 continuadores;
- mismo número de cada grupo dentro de mayo y junio;
- cada cliente aparece una sola vez.

Los continuadores se utilizan exclusivamente para que el Random Forest aprenda qué comportamientos diferencian a las bajas. No serán incorporados como miembros de los clusters: la segmentación posterior se realizará únicamente sobre las 1.991 bajas estrictas con tres meses consecutivos.

## 11. Random Forest como representación de comportamiento

El Random Forest se utiliza como método supervisado para aprender qué combinaciones de comportamiento distinguen a una baja estricta de un cliente que continúa dentro del mismo mes.

El objetivo no es construir un modelo causal ni evaluar una campaña. Las hojas de los árboles se utilizarán posteriormente para calcular la proximidad entre las bajas.

Se incluyen todas las variables numéricas disponibles en `fe_panel`, junto con sus lags, deltas, ventanas y tendencias. Solamente se excluyen:

- identificadores;
- variables temporales de control;
- la clase objetivo;
- fechas y estados administrativos de tarjetas;
- variables constantes dentro de la muestra.

Los valores faltantes se imputan mediante la mediana y se agregan indicadores de ausencia. De esta manera, el modelo puede distinguir entre un valor observado y un valor que fue imputado.


In [46]:
def construir_muestra(controles):
    bajas = strict_complete.copy()

    bajas["grupo"] = 1
    bajas["mes_modelo"] = (
        bajas["foto_mes_ancla"]
    )

    continuadores = controles.copy()

    continuadores["grupo"] = 0
    continuadores["mes_modelo"] = (
        continuadores["foto_mes"]
    )

    muestra = pd.concat(
        [
            bajas,
            continuadores,
        ],
        ignore_index=True,
        sort=False,
    )

    muestra = (
        muestra
        .sort_values(
            [
                "grupo",
                "mes_modelo",
                "numero_de_cliente",
            ],
            kind="mergesort",
        )
        .reset_index(drop=True)
    )

    assert muestra[
        "numero_de_cliente"
    ].is_unique

    return muestra


sample_main = construir_muestra(
    controls_main
)


columnas_administrativas = {
    "Master_Finiciomora",
    "Master_Fvencimiento",
    "Master_delinquency",
    "Master_fechaalta",
    "Master_fultimo_cierre",
    "Master_status",
    "Visa_Finiciomora",
    "Visa_Fvencimiento",
    "Visa_delinquency",
    "Visa_fechaalta",
    "Visa_fultimo_cierre",
    "Visa_status",
}


columnas_excluidas = {
    "numero_de_cliente",
    "foto_mes",
    "indice_mes",
    "clase_ternaria",
    "active_quarter",
    "observaciones_hasta_mes",
    "salto_1",
    "salto_2",
} | columnas_administrativas


features_candidatas = [
    columna
    for columna in control_candidates.columns

    if columna not in columnas_excluidas

    and pd.api.types.is_numeric_dtype(
        control_candidates[columna]
    )
]


features_constantes = [
    columna
    for columna in features_candidatas

    if sample_main[
        columna
    ].nunique(
        dropna=False
    ) <= 1
]


FEATURES_RF = [
    columna
    for columna in features_candidatas

    if columna not in features_constantes
]


X_rf_crudo = (
    sample_main[FEATURES_RF]
    .replace(
        [
            np.inf,
            -np.inf,
        ],
        np.nan,
    )
    .astype(np.float32)
)


y_rf = (
    sample_main["grupo"]
    .astype(np.int8)
    .to_numpy()
)


imputador_rf = SimpleImputer(
    strategy="median",
    add_indicator=True,
    keep_empty_features=True,
)


X_rf = (
    imputador_rf
    .fit_transform(X_rf_crudo)
    .astype(np.float32)
)


NOMBRES_FEATURES_RF = (
    imputador_rf
    .get_feature_names_out(
        FEATURES_RF
    )
)


print(
    "Clientes:",
    len(sample_main),
)

print(
    "Features candidatas:",
    len(features_candidatas),
)

print(
    "Features constantes retiradas:",
    len(features_constantes),
)

print(
    "Features utilizadas antes de imputar:",
    len(FEATURES_RF),
)

print(
    "Features después de agregar indicadores:",
    X_rf.shape[1],
)

print(
    "Nulos antes de imputar:",
    int(
        X_rf_crudo
        .isna()
        .sum()
        .sum()
    ),
)

print(
    "Nulos después de imputar:",
    int(
        np.isnan(X_rf).sum()
    ),
)

print(
    "Infinitos después de imputar:",
    int(
        np.isinf(X_rf).sum()
    ),
)

print(
    "Variables constantes retiradas:",
    features_constantes,
)


display(
    pd.crosstab(
        sample_main["mes_modelo"],
        sample_main["grupo"],
        margins=True,
    )
)


assert X_rf.shape[0] == 3982
assert np.isnan(X_rf).sum() == 0
assert np.isinf(X_rf).sum() == 0

Clientes: 3982
Features candidatas: 265
Features constantes retiradas: 0
Features utilizadas antes de imputar: 265
Features después de agregar indicadores: 299
Nulos antes de imputar: 45754
Nulos después de imputar: 0
Infinitos después de imputar: 0
Variables constantes retiradas: []


grupo,0,1,All
mes_modelo,,,
202105,1129,1129,2258
202106,862,862,1724
All,1991,1991,3982


## 12. Perfiles de Random Forest y clustering jerárquico

El Random Forest se entrena para distinguir bajas estrictas de continuadores del mismo mes. Su predicción no constituye el resultado final: la estructura de los árboles se utiliza para construir una representación del comportamiento de cada cliente.

Siguiendo el procedimiento utilizado en la materia, cada baja se representa mediante la frecuencia con que cada variable constituye el último corte antes de la hoja alcanzada. De esta forma, el perfil resume qué atributos fueron relevantes para ubicar al cliente dentro del bosque.

Para reducir la dependencia de una única semilla se entrenan cinco bosques. Los perfiles obtenidos se promedian y el clustering se calcula exclusivamente sobre las 1.991 bajas estrictas con tres meses consecutivos de historia.

Sobre la matriz resultante se aplica clustering jerárquico mediante el método de Ward. Esta combinación permite examinar visualmente la jerarquía de las particiones mediante un dendrograma.

In [54]:
def atributo_ultimo_split(arbol):
    """
    Para cada nodo identifica la variable utilizada en el último
    corte antes de llegar a ese nodo.

    En las hojas representa la variable que terminó definiendo
    la asignación de la observación.
    """
    estructura = arbol.tree_

    atributo_padre = np.full(
        estructura.node_count,
        -1,
        dtype=np.int32,
    )

    nodos_internos = np.flatnonzero(
        estructura.children_left != -1
    )

    atributo_padre[
        estructura.children_left[nodos_internos]
    ] = estructura.feature[nodos_internos]

    atributo_padre[
        estructura.children_right[nodos_internos]
    ] = estructura.feature[nodos_internos]

    return atributo_padre


def construir_perfil_bosque(
    modelo,
    matriz_clientes,
):
    """
    Construye una fila por cliente y una columna por variable.

    Cada valor indica la proporción de árboles en los que esa
    variable fue el último corte que definió la hoja del cliente.
    """
    hojas = modelo.apply(
        matriz_clientes
    )

    cantidad_clientes = hojas.shape[0]
    cantidad_variables = modelo.n_features_in_

    conteos = np.zeros(
        (
            cantidad_clientes,
            cantidad_variables,
        ),
        dtype=np.float32,
    )

    filas = np.arange(
        cantidad_clientes
    )

    for indice_arbol, arbol in enumerate(
        modelo.estimators_
    ):
        mapa_atributos = atributo_ultimo_split(
            arbol
        )

        atributos_hoja = mapa_atributos[
            hojas[:, indice_arbol]
        ]

        validos = atributos_hoja >= 0

        np.add.at(
            conteos,
            (
                filas[validos],
                atributos_hoja[validos],
            ),
            1,
        )

    suma_cliente = conteos.sum(
        axis=1,
        keepdims=True,
    )

    assert np.all(
        suma_cliente > 0
    )

    return (
        conteos
        / suma_cliente
    )


mascara_bajas = (
    y_rf == 1
)

X_bajas_rf = X_rf[
    mascara_bajas
]

ids_cluster = (
    sample_main.loc[
        mascara_bajas,
        "numero_de_cliente",
    ]
    .astype("int64")
    .to_numpy()
)

assert len(ids_cluster) == 1991
assert pd.Series(ids_cluster).is_unique


rf_models = []
perfiles_por_semilla = []
resultados_bosques = []

for semilla in RF_SEEDS:
    modelo = RandomForestClassifier(
        n_estimators=N_TREES,
        min_samples_leaf=MIN_SAMPLES_LEAF,
        max_features="sqrt",
        class_weight="balanced",
        oob_score=True,
        n_jobs=-1,
        random_state=semilla,
    )

    modelo.fit(
        X_rf,
        y_rf,
    )

    perfil_semilla = construir_perfil_bosque(
        modelo,
        X_bajas_rf,
    )

    rf_models.append(
        modelo
    )

    perfiles_por_semilla.append(
        perfil_semilla
    )

    resultados_bosques.append({
        "semilla": semilla,
        "oob_descriptivo": modelo.oob_score_,
        "features_entrada": len(FEATURES_RF),
        "features_modelo": X_rf.shape[1],
    })


resumen_bosques = pd.DataFrame(
    resultados_bosques
)


# Se promedian los perfiles obtenidos con las cinco semillas.
perfil_promedio = np.mean(
    np.stack(
        perfiles_por_semilla,
        axis=0,
    ),
    axis=0,
    dtype=np.float64,
)


# Sólo se conservan variables utilizadas por al menos un árbol.
atributos_utilizados = (
    perfil_promedio.sum(axis=0) > 0
)


perfiles_bosque_final = pd.DataFrame(
    perfil_promedio[
        :,
        atributos_utilizados,
    ],
    index=pd.Index(
        ids_cluster,
        name="numero_de_cliente",
    ),
    columns=np.asarray(
        NOMBRES_FEATURES_RF
    )[
        atributos_utilizados
    ],
)


# Ward construye la jerarquía sobre los perfiles de comportamiento.
linkage_final = linkage(
    perfiles_bosque_final.to_numpy(
        dtype=np.float64
    ),
    method="ward",
)


hash_ids = hashlib.sha256(
    ids_cluster.tobytes()
).hexdigest()[:16]

hash_perfiles = hashlib.sha256(
    np.ascontiguousarray(
        perfiles_bosque_final.to_numpy()
    ).tobytes()
).hexdigest()[:16]


print(
    "Clientes representados:",
    perfiles_bosque_final.shape[0],
)

print(
    "Atributos utilizados:",
    perfiles_bosque_final.shape[1],
)

print(
    "Hash de identificadores:",
    hash_ids,
)

print(
    "Hash de perfiles:",
    hash_perfiles,
)

print(
    "Suma mínima por cliente:",
    perfiles_bosque_final.sum(axis=1).min(),
)

print(
    "Suma máxima por cliente:",
    perfiles_bosque_final.sum(axis=1).max(),
)

display(
    resumen_bosques.round(4)
)

Clientes representados: 1991
Atributos utilizados: 230
Hash de identificadores: 2048c0f97c03e1ff
Hash de perfiles: f6aa6fa4c0fc3321
Suma mínima por cliente: 1.0000000020489088
Suma máxima por cliente: 1.0000000169500705


,semilla,oob_descriptivo,features_entrada,features_modelo
0,214363,0.83,265,299
1,42,0.83,265,299
2,2026,0.83,265,299
3,12345,0.83,265,299
4,987654,0.83,265,299


Los cinco bosques presentan un desempeño OOB cercano a 0,83 y, por lo tanto, encuentran una señal reproducible para diferenciar bajas estrictas y continuadores.

Para representar a cada cliente se aplicó el procedimiento utilizado en la materia: se contó la frecuencia con que cada variable constituyó el último corte antes de una hoja del Random Forest. Los perfiles obtenidos con cinco semillas se promediaron para reducir la dependencia de una ejecución particular.

De los 299 atributos disponibles después de la imputación, 230 fueron utilizados por al menos un árbol. Cada perfil suma aproximadamente uno y describe la participación relativa de esos atributos en la representación del cliente.

El OOB evalúa la capacidad descriptiva del bosque, pero no determina por sí mismo la cantidad apropiada de segmentos.

## 13. Elección de la cantidad de perfiles

La cantidad de perfiles no se decide utilizando una única métrica. Se consideran conjuntamente:

- el dendrograma y sus espacios verticales;
- el tamaño de los grupos;
- la separación geométrica mediante silhouette;
- la estabilidad entre bosques con semillas diferentes;
- la estabilidad al retirar uno de los bosques;
- la posibilidad de interpretar los grupos como segmentos de comportamiento.

Silhouette se utiliza solamente como diagnóstico. No se interpreta como una medida definitiva de la existencia de grupos humanos naturales.

In [55]:
def etiquetas_desde_perfiles(
    perfiles,
    cantidad_clusters,
):
    """
    Construye una jerarquía con Ward y la corta en la
    cantidad solicitada de segmentos.
    """
    vinculacion = linkage(
        perfiles,
        method="ward",
    )

    etiquetas = fcluster(
        vinculacion,
        t=cantidad_clusters,
        criterion="maxclust",
    )

    return etiquetas


matriz_perfiles_final = (
    perfiles_bosque_final
    .to_numpy(dtype=np.float64)
)


cantidad_clientes_cluster = (
    matriz_perfiles_final.shape[0]
)


# ============================================================
# 1. Separación, tamaños y espacio vertical del dendrograma
# ============================================================

evaluacion_k = []

for k in range(2, 7):
    etiquetas = fcluster(
        linkage_final,
        t=k,
        criterion="maxclust",
    )

    tamanios = (
        pd.Series(etiquetas)
        .value_counts()
        .sort_values(ascending=False)
    )

    indice_altura_inferior = (
        cantidad_clientes_cluster
        - k
        - 1
    )

    indice_altura_superior = (
        cantidad_clientes_cluster
        - k
    )

    altura_inferior = float(
        linkage_final[
            indice_altura_inferior,
            2,
        ]
    )

    altura_superior = float(
        linkage_final[
            indice_altura_superior,
            2,
        ]
    )

    espacio_vertical = (
        altura_superior
        - altura_inferior
    )

    evaluacion_k.append({
        "k": k,

        # Se informa como referencia descriptiva.
        # No se utiliza aisladamente para elegir k.
        "silhouette_descriptivo": (
            silhouette_score(
                matriz_perfiles_final,
                etiquetas,
                metric="euclidean",
            )
        ),

        "espacio_vertical": espacio_vertical,

        "cluster_minimo": int(
            tamanios.min()
        ),

        "cluster_maximo": int(
            tamanios.max()
        ),

        "porcentaje_minimo": (
            100
            * tamanios.min()
            / cantidad_clientes_cluster
        ),

        "tamanios": tamanios.tolist(),

        "altura_corte": (
            altura_inferior
            + altura_superior
        ) / 2,
    })


evaluacion_k = pd.DataFrame(
    evaluacion_k
)


# ============================================================
# 2. Estabilidad entre las cinco semillas
# ============================================================

perfiles_semilla_utilizados = [
    perfil[
        :,
        atributos_utilizados,
    ]

    for perfil
    in perfiles_por_semilla
]


estabilidad_k = []

for k in [3, 4, 5]:
    etiquetas_por_semilla = [
        etiquetas_desde_perfiles(
            perfil,
            k,
        )

        for perfil
        in perfiles_semilla_utilizados
    ]

    aris_semillas = [
        adjusted_rand_score(
            etiquetas_por_semilla[i],
            etiquetas_por_semilla[j],
        )

        for i in range(
            len(etiquetas_por_semilla)
        )

        for j in range(
            i + 1,
            len(etiquetas_por_semilla)
        )
    ]

    etiquetas_ensemble = fcluster(
        linkage_final,
        t=k,
        criterion="maxclust",
    )

    aris_sin_un_bosque = []

    for bosque_omitido in range(
        len(perfiles_semilla_utilizados)
    ):
        perfil_reducido = np.mean(
            np.stack(
                [
                    perfil
                    for indice, perfil
                    in enumerate(
                        perfiles_semilla_utilizados
                    )
                    if indice != bosque_omitido
                ],
                axis=0,
            ),
            axis=0,
            dtype=np.float64,
        )

        etiquetas_reducidas = (
            etiquetas_desde_perfiles(
                perfil_reducido,
                k,
            )
        )

        aris_sin_un_bosque.append(
            adjusted_rand_score(
                etiquetas_ensemble,
                etiquetas_reducidas,
            )
        )

    estabilidad_k.append({
        "k": k,

        "ARI_semillas_promedio": (
            np.mean(aris_semillas)
        ),

        "ARI_semillas_minimo": (
            np.min(aris_semillas)
        ),

        "ARI_sin_un_bosque_promedio": (
            np.mean(
                aris_sin_un_bosque
            )
        ),

        "ARI_sin_un_bosque_minimo": (
            np.min(
                aris_sin_un_bosque
            )
        ),
    })


estabilidad_k = pd.DataFrame(
    estabilidad_k
)


# ============================================================
# 3. Evaluación conjunta
# ============================================================

evaluacion_completa_k = (
    evaluacion_k
    .merge(
        estabilidad_k,
        on="k",
        how="left",
    )
)


display(
    evaluacion_completa_k.round(4)
)

/Users/manuelmarasco/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/manuelmarasco/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/manuelmarasco/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/manuelmarasco/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/manuelmarasco/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/manuelmarasco/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/manuelmarasco/Library/Python/3.9/li

,k,silhouette_descriptivo,espacio_vertical,cluster_minimo,cluster_maximo,porcentaje_minimo,tamanios,altura_corte,ARI_semillas_promedio,ARI_semillas_minimo,ARI_sin_un_bosque_promedio,ARI_sin_un_bosque_minimo
0,2,0.21,0.36,877,1114,44.05,"[1114, 877]",1.81,NaN,NaN,NaN,NaN
1,3,0.23,0.54,92,1114,4.62,"[1114, 785, 92]",1.36,0.67,0.54,0.70,0.57
2,4,0.23,0.15,92,979,4.62,"[979, 785, 135, 92]",1.01,0.55,0.37,0.70,0.58
3,5,0.17,0.10,92,785,4.62,"[785, 693, 286, 135, 92]",0.89,0.64,0.45,0.68,0.56
4,6,0.16,0.23,92,693,4.62,"[693, 615, 286, 170, 135, 92]",0.73,NaN,NaN,NaN,NaN


In [56]:
# Altura que coloreará las cuatro ramas candidatas.
altura_k4 = float(
    evaluacion_k.loc[
        evaluacion_k["k"] == 4,
        "altura_corte",
    ].iloc[0]
)


fig, ax = plt.subplots(
    figsize=(17, 9)
)


dendrogram(
    linkage_final,
    truncate_mode="lastp",
    p=30,
    show_leaf_counts=True,
    show_contracted=True,
    leaf_rotation=0,
    leaf_font_size=9,

    # Las ramas situadas debajo del corte de k=4
    # quedan diferenciadas por color.
    color_threshold=altura_k4,
    above_threshold_color="#4D4D4D",

    ax=ax,
)


colores_corte = {
    3: "#D62728",
    4: "#2468B4",
    5: "#E76F00",
}


for k in [3, 4, 5]:
    altura = float(
        evaluacion_k.loc[
            evaluacion_k["k"] == k,
            "altura_corte",
        ].iloc[0]
    )

    ax.axhline(
        y=altura,
        color=colores_corte[k],
        linestyle="--",
        linewidth=2.2,
        label=(
            f"Corte para k={k} "
            f"· altura={altura:.2f}"
        ),
    )


ax.set_title(
    "Dendrograma de perfiles de clientes dados de baja",
    fontsize=17,
    pad=16,
)


ax.set_xlabel(
    "Ramas finales; entre paréntesis se indica la cantidad de clientes",
    fontsize=11,
)


ax.set_ylabel(
    "Distancia de unión entre perfiles",
    fontsize=11,
)


ax.grid(
    axis="y",
    alpha=0.20,
)


ax.legend(
    loc="upper left",
    frameon=True,
)


fig.tight_layout()


ruta_dendrograma = (
    OUTPUT_DIR
    / "04_dendrograma_rf.png"
)


fig.savefig(
    ruta_dendrograma,
    dpi=180,
    bbox_inches="tight",
)


plt.close(fig)

![Dendrograma de las bajas estrictas](eda_miranda_outputs/04_dendrograma_rf.png)

El gráfico muestra cuatro ramas visibles, pero las métricas indican que el corte más sólido es k=3.
La situación es:
k=3:  [1.114] [785] [92]
             ↓
k=4:  [979] [135] [785] [92]
La pregunta decisiva no es si el grupo de 135 tiene el mismo tamaño, sino si posee un comportamiento claramente diferente del grupo de 979. Si lo tiene, k=4 puede justificarse como segmentación de negocio. Si no, sería una fragmentación débil y convendría k=3.


In [57]:
# ============================================================
# Comparación dirigida entre k=3 y k=4
# ============================================================

etiquetas_k3 = fcluster(
    linkage_final,
    t=3,
    criterion="maxclust",
)

etiquetas_k4 = fcluster(
    linkage_final,
    t=4,
    criterion="maxclust",
)


comparacion_asignaciones = pd.DataFrame({
    "numero_de_cliente": ids_cluster,
    "cluster_k3": etiquetas_k3,
    "cluster_k4": etiquetas_k4,
})


tabla_cruce_k3_k4 = pd.crosstab(
    comparacion_asignaciones["cluster_k3"],
    comparacion_asignaciones["cluster_k4"],
    margins=True,
)


tabla_porcentajes_k3_k4 = (
    pd.crosstab(
        comparacion_asignaciones["cluster_k3"],
        comparacion_asignaciones["cluster_k4"],
        normalize="index",
    )
    .mul(100)
)


display(
    tabla_cruce_k3_k4
)

display(
    tabla_porcentajes_k3_k4.round(2)
)


# Se identifica automáticamente el único grupo de k=3
# que se divide al pasar a k=4.
tabla_sin_totales = pd.crosstab(
    comparacion_asignaciones["cluster_k3"],
    comparacion_asignaciones["cluster_k4"],
)


cantidad_hijos = (
    tabla_sin_totales
    .gt(0)
    .sum(axis=1)
)


cluster_k3_dividido = (
    cantidad_hijos.idxmax()
)


clusters_k4_resultantes = (
    tabla_sin_totales.loc[
        cluster_k3_dividido
    ]
    .loc[
        lambda serie: serie > 0
    ]
    .index
    .tolist()
)


print(
    "Cluster de k=3 que se divide:",
    cluster_k3_dividido,
)

print(
    "Ramas resultantes en k=4:",
    clusters_k4_resultantes,
)


# ============================================================
# Recuperación de variables reales de negocio
# ============================================================

base_comparacion_k = (
    strict_complete
    .set_index("numero_de_cliente")
    .loc[ids_cluster]
    .reset_index()
    .copy()
)


base_comparacion_k[
    "cluster_k3"
] = etiquetas_k3

base_comparacion_k[
    "cluster_k4"
] = etiquetas_k4


base_comparacion_k["payroll_activo"] = (
    base_comparacion_k["cpayroll_trx"] > 0
).astype(int)

base_comparacion_k["usa_tarjeta"] = (
    base_comparacion_k["tarjetas_transacciones"] > 0
).astype(int)

base_comparacion_k["usa_digital"] = (
    base_comparacion_k["transacciones_digitales"] > 0
).astype(int)

base_comparacion_k["emite_transferencias"] = (
    base_comparacion_k["ctransferencias_emitidas"] > 0
).astype(int)

base_comparacion_k["tiene_credito"] = (
    base_comparacion_k["cantidad_creditos"] > 0
).astype(int)

base_comparacion_k["tiene_inversion"] = (
    base_comparacion_k["cantidad_inversiones"] > 0
).astype(int)

base_comparacion_k["tiene_seguro"] = (
    base_comparacion_k["cantidad_seguros"] > 0
).astype(int)

base_comparacion_k["usa_debitos"] = (
    base_comparacion_k["debitos_automaticos"] > 0
).astype(int)

base_comparacion_k["paga_servicios"] = (
    base_comparacion_k["pagos_servicios"] > 0
).astype(int)

base_comparacion_k["saldo_negativo"] = (
    base_comparacion_k["mcuentas_saldo"] < 0
).astype(int)

base_comparacion_k[
    "mantenimiento_positivo"
] = (
    base_comparacion_k[
        "mcomisiones_mantenimiento"
    ] > 0
).astype(int)


columnas_familias = [
    "payroll_activo",
    "usa_tarjeta",
    "usa_digital",
    "emite_transferencias",
    "tiene_credito",
    "tiene_inversion",
    "tiene_seguro",
    "usa_debitos",
    "paga_servicios",
]


base_comparacion_k["familias_activas"] = (
    base_comparacion_k[
        columnas_familias
    ]
    .sum(axis=1)
)


def porcentaje(serie):
    return 100 * serie.mean()


# Sólo se estudian los clientes pertenecientes al grupo
# que se divide entre k=3 y k=4.
base_rama_dividida = (
    base_comparacion_k.loc[
        base_comparacion_k["cluster_k3"]
        == cluster_k3_dividido
    ]
    .copy()
)


perfil_ramas_k4 = (
    base_rama_dividida
    .groupby("cluster_k4")
    .agg(
        clientes=(
            "numero_de_cliente",
            "nunique",
        ),

        edad_mediana=(
            "cliente_edad",
            "median",
        ),

        antiguedad_mediana=(
            "cliente_antiguedad",
            "median",
        ),

        productos_mediana=(
            "cproductos",
            "median",
        ),

        familias_activas_mediana=(
            "familias_activas",
            "median",
        ),

        transacciones_t0=(
            "ctrx_quarter",
            "median",
        ),

        transacciones_media_3m=(
            "ctrx_quarter_media_3",
            "median",
        ),

        porcentaje_payroll=(
            "payroll_activo",
            porcentaje,
        ),

        porcentaje_usa_tarjeta=(
            "usa_tarjeta",
            porcentaje,
        ),

        porcentaje_digital=(
            "usa_digital",
            porcentaje,
        ),

        porcentaje_transferencias=(
            "emite_transferencias",
            porcentaje,
        ),

        porcentaje_credito=(
            "tiene_credito",
            porcentaje,
        ),

        porcentaje_inversion=(
            "tiene_inversion",
            porcentaje,
        ),

        porcentaje_seguro=(
            "tiene_seguro",
            porcentaje,
        ),

        porcentaje_debitos=(
            "usa_debitos",
            porcentaje,
        ),

        porcentaje_servicios=(
            "paga_servicios",
            porcentaje,
        ),

        saldo_mediano=(
            "mcuentas_saldo",
            "median",
        ),

        porcentaje_saldo_negativo=(
            "saldo_negativo",
            porcentaje,
        ),

        rentabilidad_mensual_mediana=(
            "mrentabilidad",
            "median",
        ),

        rentabilidad_anual_mediana=(
            "mrentabilidad_annual",
            "median",
        ),

        porcentaje_mantenimiento=(
            "mantenimiento_positivo",
            porcentaje,
        ),
    )
    .reset_index()
)


perfil_ramas_k4[
    "porcentaje_dentro_del_grupo_k3"
] = (
    100
    * perfil_ramas_k4["clientes"]
    / perfil_ramas_k4["clientes"].sum()
)


display(
    perfil_ramas_k4.round(2)
)

cluster_k4,1,2,3,4,All
cluster_k3,,,,,
1,135,979,0,0,1114
2,0,0,92,0,92
3,0,0,0,785,785
All,135,979,92,785,1991


cluster_k4,1,2,3,4
cluster_k3,,,,
1,12.12,87.88,0.00,0.00
2,0.00,0.00,100.00,0.00
3,0.00,0.00,0.00,100.00


Cluster de k=3 que se divide: 1
Ramas resultantes en k=4: [1, 2]


,cluster_k4,clientes,edad_mediana,antiguedad_mediana,productos_mediana,familias_activas_mediana,transacciones_t0,transacciones_media_3m,porcentaje_payroll,porcentaje_usa_tarjeta,porcentaje_digital,porcentaje_transferencias,porcentaje_credito,porcentaje_inversion,porcentaje_seguro,porcentaje_debitos,porcentaje_servicios,saldo_mediano,porcentaje_saldo_negativo,rentabilidad_mensual_mediana,rentabilidad_anual_mediana,porcentaje_mantenimiento,porcentaje_dentro_del_grupo_k3
0,1,135,45.00,69.00,7.00,4.00,72.00,77.00,66.67,56.30,93.33,61.48,9.63,12.59,21.48,52.59,40.00,"7,178.78",22.22,"1,153.53","9,086.31",17.78,12.12
1,2,979,47.00,115.00,7.00,3.00,37.00,40.00,1.02,69.87,86.52,32.18,4.29,7.66,21.55,61.59,19.71,117.30,43.51,"1,703.68","9,670.50",68.64,87.88


Las métricas estructurales favorecen k=3, pero ese corte mezcla dos poblaciones muy diferentes:
- Rama de 135 clientes: 66,67% con payroll, 72 transacciones, 4 familias activas.
- Rama de 979 clientes: apenas 1,02% con payroll, 37 transacciones, 3 familias activas.
El grupo de 135 representa 6,8% de todas las bajas analizadas. No es un residuo microscópico y tiene un comportamiento concreto: clientes mucho más vinculados por nómina y actividad. k=3 ocultaría ese perfil.
La conclusión correcta es:
k=3 es el corte estadísticamente más natural, pero k=4 agrega una separación de negocio material e interpretable. Por eso se conserva k=4, reconociendo que su estabilidad entre semillas es moderada.

In [58]:
# ============================================================
# Visualización de la separación adicional de k=4
# ============================================================

fila_payroll = perfil_ramas_k4.loc[
    perfil_ramas_k4[
        "porcentaje_payroll"
    ].idxmax()
]

fila_sin_payroll = perfil_ramas_k4.loc[
    perfil_ramas_k4[
        "porcentaje_payroll"
    ].idxmin()
]


fig, ax = plt.subplots(
    figsize=(12, 6)
)

ax.axis("off")


# Grupo original en k=3.
ax.text(
    0.50,
    0.80,
    (
        "GRUPO EN k=3\n"
        f"{len(base_rama_dividida):,} clientes\n"
        "Un único perfil"
    ),
    ha="center",
    va="center",
    fontsize=13,
    weight="bold",
    bbox={
        "boxstyle": "round,pad=1",
        "facecolor": "#E8EEF5",
        "edgecolor": "#2F6B9A",
        "linewidth": 2,
    },
    transform=ax.transAxes,
)


# Rama con payroll.
ax.text(
    0.25,
    0.30,
    (
        "PERFIL VINCULADO POR NÓMINA\n"
        f"{int(fila_payroll['clientes']):,} clientes\n"
        f"{fila_payroll['porcentaje_dentro_del_grupo_k3']:.1f}% del grupo original\n\n"
        f"Payroll activo: {fila_payroll['porcentaje_payroll']:.1f}%\n"
        f"Transacciones: {fila_payroll['transacciones_t0']:.0f}\n"
        f"Promedio de 3 meses: {fila_payroll['transacciones_media_3m']:.0f}\n"
        f"Familias activas: {fila_payroll['familias_activas_mediana']:.0f}"
    ),
    ha="center",
    va="center",
    fontsize=11,
    bbox={
        "boxstyle": "round,pad=1",
        "facecolor": "#FFF1D6",
        "edgecolor": "#D99A27",
        "linewidth": 2,
    },
    transform=ax.transAxes,
)


# Rama restante.
ax.text(
    0.75,
    0.30,
    (
        "PERFIL SIN VINCULACIÓN POR NÓMINA\n"
        f"{int(fila_sin_payroll['clientes']):,} clientes\n"
        f"{fila_sin_payroll['porcentaje_dentro_del_grupo_k3']:.1f}% del grupo original\n\n"
        f"Payroll activo: {fila_sin_payroll['porcentaje_payroll']:.1f}%\n"
        f"Transacciones: {fila_sin_payroll['transacciones_t0']:.0f}\n"
        f"Promedio de 3 meses: {fila_sin_payroll['transacciones_media_3m']:.0f}\n"
        f"Familias activas: {fila_sin_payroll['familias_activas_mediana']:.0f}"
    ),
    ha="center",
    va="center",
    fontsize=11,
    bbox={
        "boxstyle": "round,pad=1",
        "facecolor": "#E4F3EA",
        "edgecolor": "#3A8D5D",
        "linewidth": 2,
    },
    transform=ax.transAxes,
)


ax.annotate(
    "",
    xy=(0.25, 0.50),
    xytext=(0.46, 0.70),
    xycoords=ax.transAxes,
    arrowprops={
        "arrowstyle": "->",
        "linewidth": 2,
        "color": "#666666",
    },
)


ax.annotate(
    "",
    xy=(0.75, 0.50),
    xytext=(0.54, 0.70),
    xycoords=ax.transAxes,
    arrowprops={
        "arrowstyle": "->",
        "linewidth": 2,
        "color": "#666666",
    },
)


ax.set_title(
    "Separación adicional aportada por k=4",
    fontsize=16,
    pad=18,
)


fig.tight_layout()


ruta_justificacion_k4 = (
    OUTPUT_DIR
    / "04b_justificacion_k4.png"
)


fig.savefig(
    ruta_justificacion_k4,
    dpi=180,
    bbox_inches="tight",
)


plt.close(fig)

![Justificación de la separación adicional de k=4](eda_miranda_outputs/04b_justificacion_k4.png)

**Decisión sobre la cantidad de perfiles.** El dendrograma y las medidas de estabilidad indican que `k=3` constituye el corte estructuralmente más fuerte. Sin embargo, esa solución reúne dentro de un mismo segmento a 135 clientes con una vinculación por nómina y una actividad transaccional considerablemente mayores que las de los otros 979 clientes de la misma rama.

La solución `k=4` conserva esta población como un perfil separado. El grupo representa aproximadamente el 6,8% de las bajas con historia completa, por lo que no se considera un residuo excepcional. Además, la diferencia se manifiesta en varias dimensiones coherentes —payroll, transacciones y familias activas— y posee una interpretación comercial directa.

En consecuencia, se adopta `k=4` como una decisión de segmentación de negocio. No se presenta como el óptimo matemático: su estabilidad entre semillas es inferior a la de `k=3`, por lo que la taxonomía deberá validarse con nuevas cohortes y períodos adicionales.

In [59]:
# ============================================================
# Estabilidad específica de cada segmento de k=4
# ============================================================

def alinear_con_referencia(
    etiquetas_referencia,
    etiquetas_candidatas,
):
    """
    Alinea los números arbitrarios de los clusters mediante
    la correspondencia que maximiza la cantidad de clientes
    coincidentes.
    """
    etiquetas_ref = np.sort(
        np.unique(etiquetas_referencia)
    )

    etiquetas_cand = np.sort(
        np.unique(etiquetas_candidatas)
    )

    matriz_cruce = pd.crosstab(
        pd.Series(
            etiquetas_referencia,
            name="referencia",
        ),
        pd.Series(
            etiquetas_candidatas,
            name="candidata",
        ),
    ).reindex(
        index=etiquetas_ref,
        columns=etiquetas_cand,
        fill_value=0,
    )

    filas, columnas = (
        linear_sum_assignment(
            -matriz_cruce.to_numpy()
        )
    )

    correspondencia = {
        etiquetas_cand[columna]:
        etiquetas_ref[fila]

        for fila, columna
        in zip(filas, columnas)
    }

    return np.array([
        correspondencia[etiqueta]
        for etiqueta
        in etiquetas_candidatas
    ])


etiquetas_referencia_k4 = (
    etiquetas_k4.copy()
)


cluster_payroll_k4 = int(
    fila_payroll["cluster_k4"]
)


estabilidad_segmentos_k4 = []
perfil_payroll_por_semilla = []


for semilla, perfil_semilla in zip(
    RF_SEEDS,
    perfiles_semilla_utilizados,
):
    etiquetas_semilla = (
        etiquetas_desde_perfiles(
            perfil_semilla,
            4,
        )
    )

    etiquetas_alineadas = (
        alinear_con_referencia(
            etiquetas_referencia_k4,
            etiquetas_semilla,
        )
    )

    for cluster in sorted(
        np.unique(
            etiquetas_referencia_k4
        )
    ):
        pertenece_referencia = (
            etiquetas_referencia_k4
            == cluster
        )

        pertenece_semilla = (
            etiquetas_alineadas
            == cluster
        )

        interseccion = np.sum(
            pertenece_referencia
            & pertenece_semilla
        )

        union = np.sum(
            pertenece_referencia
            | pertenece_semilla
        )

        cantidad_referencia = np.sum(
            pertenece_referencia
        )

        cantidad_semilla = np.sum(
            pertenece_semilla
        )

        estabilidad_segmentos_k4.append({
            "semilla": semilla,
            "cluster_k4": cluster,
            "clientes_referencia": (
                cantidad_referencia
            ),
            "clientes_semilla": (
                cantidad_semilla
            ),
            "clientes_coincidentes": (
                interseccion
            ),
            "jaccard": (
                interseccion / union
            ),
            "retencion": (
                interseccion
                / cantidad_referencia
            ),
            "precision": (
                interseccion
                / cantidad_semilla
            ),
        })

    mascara_payroll_semilla = (
        etiquetas_alineadas
        == cluster_payroll_k4
    )

    datos_payroll_semilla = (
        base_comparacion_k.iloc[
            np.flatnonzero(
                mascara_payroll_semilla
            )
        ]
    )

    perfil_payroll_por_semilla.append({
        "semilla": semilla,

        "clientes": (
            len(datos_payroll_semilla)
        ),

        "porcentaje_payroll": (
            100
            * datos_payroll_semilla[
                "payroll_activo"
            ].mean()
        ),

        "transacciones_mediana": (
            datos_payroll_semilla[
                "ctrx_quarter"
            ].median()
        ),

        "familias_activas_mediana": (
            datos_payroll_semilla[
                "familias_activas"
            ].median()
        ),

        "porcentaje_digital": (
            100
            * datos_payroll_semilla[
                "usa_digital"
            ].mean()
        ),
    })


estabilidad_segmentos_k4 = (
    pd.DataFrame(
        estabilidad_segmentos_k4
    )
)


perfil_payroll_por_semilla = (
    pd.DataFrame(
        perfil_payroll_por_semilla
    )
)


resumen_estabilidad_k4 = (
    estabilidad_segmentos_k4
    .groupby("cluster_k4")
    .agg(
        clientes_referencia=(
            "clientes_referencia",
            "first",
        ),

        jaccard_promedio=(
            "jaccard",
            "mean",
        ),

        jaccard_minimo=(
            "jaccard",
            "min",
        ),

        retencion_promedio=(
            "retencion",
            "mean",
        ),

        retencion_minima=(
            "retencion",
            "min",
        ),

        precision_promedio=(
            "precision",
            "mean",
        ),

        precision_minima=(
            "precision",
            "min",
        ),

        tamanio_minimo=(
            "clientes_semilla",
            "min",
        ),

        tamanio_maximo=(
            "clientes_semilla",
            "max",
        ),
    )
    .reset_index()
)


print(
    "Cluster identificado por payroll:",
    cluster_payroll_k4,
)


display(
    resumen_estabilidad_k4.round(3)
)


display(
    perfil_payroll_por_semilla.round(2)
)

Cluster identificado por payroll: 1


,cluster_k4,clientes_referencia,jaccard_promedio,jaccard_minimo,retencion_promedio,retencion_minima,precision_promedio,precision_minima,tamanio_minimo,tamanio_maximo
0,1,135,0.48,0.00,0.60,0.00,0.48,0.00,167,193
1,2,979,0.76,0.59,0.83,0.68,0.89,0.77,811,1016
2,3,92,0.89,0.73,0.98,0.98,0.90,0.74,92,121
3,4,785,0.78,0.53,0.87,0.66,0.86,0.73,702,890


,semilla,clientes,porcentaje_payroll,transacciones_mediana,familias_activas_mediana,porcentaje_digital
0,214363,173,61.27,57.00,4.00,89.02
1,42,168,58.93,57.00,4.00,89.29
2,2026,175,0.00,2.00,0.00,33.14
3,12345,167,58.08,57.00,4.00,89.22
4,987654,193,0.52,18.00,2.00,76.68


La lectura será:
- jaccard_minimo ≥ 0,70: evidencia fuerte para conservar k=4.
- Entre 0,50 y 0,70: segmento interpretable, pero provisional.
- Menor que 0,50: sus integrantes cambian demasiado; convendría k=3.

Este resultado va en contra de elegir k=4:
- El grupo de payroll tiene Jaccard promedio = 0,48 y mínimo 0.
- En dos semillas deja de representar payroll: 0% y 0,52%.
- También cambia drásticamente en transacciones y familias activas.
Por lo tanto, no puede afirmarse todavía que el cuarto grupo sea estable. Sin embargo, hay una precaución: la correspondencia global puede sacrificar un grupo pequeño para alinear mejor los tres grandes. Hagamos una última comprobación buscando, en cada semilla, el cluster con mayor payroll, independientemente de su número.

In [60]:
# ============================================================
# Búsqueda semántica del perfil de payroll en cada semilla
# ============================================================

mascara_payroll_referencia = (
    etiquetas_referencia_k4
    == cluster_payroll_k4
)


comparacion_semantica_payroll = []
detalle_clusters_semillas = []


for semilla, perfil_semilla in zip(
    RF_SEEDS,
    perfiles_semilla_utilizados,
):
    etiquetas_semilla = (
        etiquetas_desde_perfiles(
            perfil_semilla,
            4,
        )
    )

    datos_semilla = (
        base_comparacion_k.copy()
    )

    datos_semilla[
        "cluster_semilla"
    ] = etiquetas_semilla


    resumen_semilla = (
        datos_semilla
        .groupby("cluster_semilla")
        .agg(
            clientes=(
                "numero_de_cliente",
                "nunique",
            ),

            porcentaje_payroll=(
                "payroll_activo",
                porcentaje,
            ),

            transacciones_mediana=(
                "ctrx_quarter",
                "median",
            ),

            familias_activas_mediana=(
                "familias_activas",
                "median",
            ),

            porcentaje_digital=(
                "usa_digital",
                porcentaje,
            ),
        )
        .reset_index()
    )

    resumen_semilla[
        "semilla"
    ] = semilla

    detalle_clusters_semillas.append(
        resumen_semilla
    )


    # Se elige por significado: el grupo con mayor
    # proporción de clientes con payroll.
    fila_max_payroll = resumen_semilla.loc[
        resumen_semilla[
            "porcentaje_payroll"
        ].idxmax()
    ]

    cluster_semantico = int(
        fila_max_payroll[
            "cluster_semilla"
        ]
    )

    mascara_semantica = (
        etiquetas_semilla
        == cluster_semantico
    )

    interseccion = np.sum(
        mascara_payroll_referencia
        & mascara_semantica
    )

    union = np.sum(
        mascara_payroll_referencia
        | mascara_semantica
    )

    comparacion_semantica_payroll.append({
        "semilla": semilla,

        "cluster_con_mayor_payroll": (
            cluster_semantico
        ),

        "clientes": int(
            fila_max_payroll[
                "clientes"
            ]
        ),

        "porcentaje_payroll": (
            fila_max_payroll[
                "porcentaje_payroll"
            ]
        ),

        "transacciones_mediana": (
            fila_max_payroll[
                "transacciones_mediana"
            ]
        ),

        "familias_activas_mediana": (
            fila_max_payroll[
                "familias_activas_mediana"
            ]
        ),

        "porcentaje_digital": (
            fila_max_payroll[
                "porcentaje_digital"
            ]
        ),

        "jaccard_con_referencia": (
            interseccion / union
        ),

        "retencion_referencia": (
            interseccion
            / mascara_payroll_referencia.sum()
        ),

        "precision_semantica": (
            interseccion
            / mascara_semantica.sum()
        ),
    })


comparacion_semantica_payroll = (
    pd.DataFrame(
        comparacion_semantica_payroll
    )
)


detalle_clusters_semillas = (
    pd.concat(
        detalle_clusters_semillas,
        ignore_index=True,
    )
)


display(
    comparacion_semantica_payroll.round(3)
)

,semilla,cluster_con_mayor_payroll,clientes,porcentaje_payroll,transacciones_mediana,familias_activas_mediana,porcentaje_digital,jaccard_con_referencia,retencion_referencia,precision_semantica
0,214363,1,173,61.27,57.00,4.00,89.02,0.78,1.00,0.78
1,42,1,168,58.93,57.00,4.00,89.29,0.80,1.00,0.80
2,2026,1,1016,10.43,45.00,3.00,89.27,0.13,1.00,0.13
3,12345,1,167,58.08,57.00,4.00,89.22,0.81,1.00,0.81
4,987654,2,811,11.84,55.00,3.00,91.12,0.16,0.98,0.16


In [61]:
# ============================================================
# Estabilidad del perfil de payroll al retirar un bosque
# ============================================================

estabilidad_payroll_sin_un_bosque = []


for bosque_omitido, semilla_omitida in enumerate(
    RF_SEEDS
):
    perfil_reducido = np.mean(
        np.stack(
            [
                perfil
                for indice, perfil
                in enumerate(
                    perfiles_semilla_utilizados
                )
                if indice != bosque_omitido
            ],
            axis=0,
        ),
        axis=0,
        dtype=np.float64,
    )

    etiquetas_reducidas = (
        etiquetas_desde_perfiles(
            perfil_reducido,
            4,
        )
    )

    datos_reducidos = (
        base_comparacion_k.copy()
    )

    datos_reducidos[
        "cluster_reducido"
    ] = etiquetas_reducidas


    resumen_reducido = (
        datos_reducidos
        .groupby("cluster_reducido")
        .agg(
            clientes=(
                "numero_de_cliente",
                "nunique",
            ),

            porcentaje_payroll=(
                "payroll_activo",
                porcentaje,
            ),

            transacciones_mediana=(
                "ctrx_quarter",
                "median",
            ),

            familias_activas_mediana=(
                "familias_activas",
                "median",
            ),

            porcentaje_digital=(
                "usa_digital",
                porcentaje,
            ),
        )
        .reset_index()
    )


    fila_max_payroll = resumen_reducido.loc[
        resumen_reducido[
            "porcentaje_payroll"
        ].idxmax()
    ]

    cluster_semantico = int(
        fila_max_payroll[
            "cluster_reducido"
        ]
    )

    mascara_semantica = (
        etiquetas_reducidas
        == cluster_semantico
    )

    interseccion = np.sum(
        mascara_payroll_referencia
        & mascara_semantica
    )

    union = np.sum(
        mascara_payroll_referencia
        | mascara_semantica
    )


    estabilidad_payroll_sin_un_bosque.append({
        "semilla_omitida": semilla_omitida,

        "cluster_con_mayor_payroll": (
            cluster_semantico
        ),

        "clientes": int(
            fila_max_payroll[
                "clientes"
            ]
        ),

        "porcentaje_payroll": (
            fila_max_payroll[
                "porcentaje_payroll"
            ]
        ),

        "transacciones_mediana": (
            fila_max_payroll[
                "transacciones_mediana"
            ]
        ),

        "familias_activas_mediana": (
            fila_max_payroll[
                "familias_activas_mediana"
            ]
        ),

        "porcentaje_digital": (
            fila_max_payroll[
                "porcentaje_digital"
            ]
        ),

        "jaccard_con_referencia": (
            interseccion / union
        ),

        "retencion_referencia": (
            interseccion
            / mascara_payroll_referencia.sum()
        ),

        "precision_semantica": (
            interseccion
            / mascara_semantica.sum()
        ),
    })


estabilidad_payroll_sin_un_bosque = (
    pd.DataFrame(
        estabilidad_payroll_sin_un_bosque
    )
)


display(
    estabilidad_payroll_sin_un_bosque.round(3)
)

,semilla_omitida,cluster_con_mayor_payroll,clientes,porcentaje_payroll,transacciones_mediana,familias_activas_mediana,porcentaje_digital,jaccard_con_referencia,retencion_referencia,precision_semantica
0,214363,1,156,60.26,63.50,4.00,89.74,0.84,0.98,0.85
1,42,1,172,59.30,57.00,4.00,88.95,0.79,1.00,0.79
2,2026,1,171,60.23,57.00,4.00,88.89,0.79,1.00,0.79
3,12345,1,143,63.64,69.00,4.00,92.31,0.92,0.98,0.93
4,987654,1,133,66.92,73.00,4.00,93.98,0.97,0.98,0.99


**Decisión sobre la cantidad de perfiles.** El dendrograma y las medidas globales indican que `k=3` constituye el corte geométricamente más marcado: presenta el mayor espacio vertical y una estabilidad superior entre bosques individuales. El valor descriptivo de *silhouette* es prácticamente igual para `k=3` y `k=4`, por lo que no permite resolver la elección por sí solo.

Al pasar de `k=3` a `k=4`, un segmento de 1.114 clientes se divide en dos ramas de 979 y 135 integrantes. La rama menor presenta una vinculación claramente asociada con la acreditación de haberes, mayor actividad transaccional y más familias de productos activas.

La estabilidad de esta rama se evaluó sobre el ensamble utilizado en la solución final. Al retirar alternativamente cada uno de los cinco bosques, el perfil volvió a aparecer con entre 133 y 172 clientes, entre 59% y 67% de presencia de payroll y una similitud de Jaccard de entre 0,79 y 0,97 respecto de la solución completa. Además, retuvo al menos el 98% de sus integrantes de referencia.

En consecuencia, se adopta `k=4` porque agrega un perfil de negocio estable e interpretable que quedaría absorbido dentro de una categoría más general con `k=3`. La elección no se presenta como un óptimo matemático: combina la estructura jerárquica, la estabilidad del ensamble y la utilidad descriptiva de la segmentación. La taxonomía deberá validarse posteriormente con nuevas cohortes y períodos adicionales.

### Sensibilidad a la selección de continuadores

Los continuadores constituyen una muestra de referencia seleccionada aleatoriamente dentro de cada mes. Para comprobar que la segmentación no depende exclusivamente de esa selección, se repite el procedimiento completo con dos muestras alternativas.

En cada repetición se vuelven a ajustar la imputación, los cinco bosques, los perfiles de atributos y el clustering jerárquico. Las asignaciones obtenidas se comparan con la solución principal mediante ARI. También se verifica si el perfil asociado con payroll conserva su composición.

In [62]:
# ============================================================
# Sensibilidad a la selección de continuadores
# ============================================================

def construir_perfiles_alternativos(
    muestra,
    semillas_bosque,
):
    matriz_cruda = (
        muestra[FEATURES_RF]
        .replace(
            [np.inf, -np.inf],
            np.nan,
        )
        .astype(np.float32)
    )

    objetivo = (
        muestra["grupo"]
        .astype(np.int8)
        .to_numpy()
    )

    imputador = SimpleImputer(
        strategy="median",
        add_indicator=True,
        keep_empty_features=True,
    )

    matriz_modelo = (
        imputador
        .fit_transform(matriz_cruda)
        .astype(np.float32)
    )

    mascara_bajas_local = (
        objetivo == 1
    )

    matriz_bajas_local = (
        matriz_modelo[
            mascara_bajas_local
        ]
    )

    ids_bajas_local = (
        muestra.loc[
            mascara_bajas_local,
            "numero_de_cliente",
        ]
        .astype("int64")
        .to_numpy()
    )

    perfiles_locales = []
    oob_locales = []

    for semilla_bosque in semillas_bosque:
        modelo_local = RandomForestClassifier(
            n_estimators=N_TREES,
            min_samples_leaf=MIN_SAMPLES_LEAF,
            max_features="sqrt",
            class_weight="balanced",
            oob_score=True,
            n_jobs=-1,
            random_state=semilla_bosque,
        )

        modelo_local.fit(
            matriz_modelo,
            objetivo,
        )

        perfiles_locales.append(
            construir_perfil_bosque(
                modelo_local,
                matriz_bajas_local,
            )
        )

        oob_locales.append(
            modelo_local.oob_score_
        )

    perfil_promedio_local = np.mean(
        np.stack(
            perfiles_locales,
            axis=0,
        ),
        axis=0,
        dtype=np.float64,
    )

    atributos_locales_utilizados = (
        perfil_promedio_local.sum(axis=0)
        > 0
    )

    perfil_promedio_local = (
        perfil_promedio_local[
            :,
            atributos_locales_utilizados,
        ]
    )

    return {
        "perfiles": perfil_promedio_local,
        "ids": ids_bajas_local,
        "oob_promedio": np.mean(oob_locales),
        "atributos_utilizados": int(
            atributos_locales_utilizados.sum()
        ),
        "features_modelo": (
            matriz_modelo.shape[1]
        ),
    }


etiquetas_referencia_controles = (
    etiquetas_k4.copy()
)

cluster_payroll_referencia = int(
    fila_payroll["cluster_k4"]
)

mascara_payroll_referencia_controles = (
    etiquetas_referencia_controles
    == cluster_payroll_referencia
)


sensibilidad_controles = []


for semilla_control in (
    CONTROL_SENSITIVITY_SEEDS
):
    controles_alternativos = (
        seleccionar_controles(
            semilla_control
        )
    )

    muestra_alternativa = (
        construir_muestra(
            controles_alternativos
        )
    )

    resultado_alternativo = (
        construir_perfiles_alternativos(
            muestra_alternativa,
            RF_SEEDS,
        )
    )

    assert np.array_equal(
        resultado_alternativo["ids"],
        ids_cluster,
    )

    etiquetas_alternativas = (
        etiquetas_desde_perfiles(
            resultado_alternativo[
                "perfiles"
            ],
            K_FINAL,
        )
    )

    ari_global = adjusted_rand_score(
        etiquetas_referencia_controles,
        etiquetas_alternativas,
    )

    tamanios_alternativos = (
        pd.Series(
            etiquetas_alternativas
        )
        .value_counts()
        .sort_values(
            ascending=False
        )
        .tolist()
    )


    # Identificación semántica del perfil
    # con mayor presencia de payroll.
    datos_alternativos = (
        base_comparacion_k.copy()
    )

    datos_alternativos[
        "cluster_alternativo"
    ] = etiquetas_alternativas


    resumen_alternativo = (
        datos_alternativos
        .groupby(
            "cluster_alternativo"
        )
        .agg(
            clientes=(
                "numero_de_cliente",
                "nunique",
            ),

            porcentaje_payroll=(
                "payroll_activo",
                porcentaje,
            ),

            transacciones_mediana=(
                "ctrx_quarter",
                "median",
            ),

            familias_activas_mediana=(
                "familias_activas",
                "median",
            ),
        )
        .reset_index()
    )


    fila_payroll_alternativa = (
        resumen_alternativo.loc[
            resumen_alternativo[
                "porcentaje_payroll"
            ].idxmax()
        ]
    )

    cluster_payroll_alternativo = int(
        fila_payroll_alternativa[
            "cluster_alternativo"
        ]
    )

    mascara_payroll_alternativa = (
        etiquetas_alternativas
        == cluster_payroll_alternativo
    )

    interseccion = np.sum(
        mascara_payroll_referencia_controles
        & mascara_payroll_alternativa
    )

    union = np.sum(
        mascara_payroll_referencia_controles
        | mascara_payroll_alternativa
    )


    sensibilidad_controles.append({
        "semilla_controles": (
            semilla_control
        ),

        "oob_promedio": (
            resultado_alternativo[
                "oob_promedio"
            ]
        ),

        "features_modelo": (
            resultado_alternativo[
                "features_modelo"
            ]
        ),

        "atributos_utilizados": (
            resultado_alternativo[
                "atributos_utilizados"
            ]
        ),

        "ARI_con_solucion_principal": (
            ari_global
        ),

        "tamanios": (
            tamanios_alternativos
        ),

        "clientes_perfil_payroll": int(
            fila_payroll_alternativa[
                "clientes"
            ]
        ),

        "porcentaje_payroll": (
            fila_payroll_alternativa[
                "porcentaje_payroll"
            ]
        ),

        "transacciones_payroll": (
            fila_payroll_alternativa[
                "transacciones_mediana"
            ]
        ),

        "familias_activas_payroll": (
            fila_payroll_alternativa[
                "familias_activas_mediana"
            ]
        ),

        "jaccard_perfil_payroll": (
            interseccion / union
        ),

        "retencion_perfil_payroll": (
            interseccion
            / (
                mascara_payroll_referencia_controles
                .sum()
            )
        ),

        "precision_perfil_payroll": (
            interseccion
            / mascara_payroll_alternativa.sum()
        ),
    })


sensibilidad_controles = (
    pd.DataFrame(
        sensibilidad_controles
    )
)


display(
    sensibilidad_controles.round(3)
)

,semilla_controles,oob_promedio,features_modelo,atributos_utilizados,ARI_con_solucion_principal,tamanios,clientes_perfil_payroll,porcentaje_payroll,transacciones_payroll,familias_activas_payroll,jaccard_perfil_payroll,retencion_perfil_payroll,precision_perfil_payroll
0,9031,0.84,299,234,0.69,"[902, 814, 174, 101]",174,59.77,57.00,4.00,0.78,1.00,0.78
1,17539,0.83,299,233,0.48,"[890, 805, 176, 120]",176,60.23,56.50,4.00,0.77,1.00,0.77


**Hipótesis intermedia.** La comparación entre `k=3` y `k=4` muestra que el cuarto corte separa un grupo de 135 clientes con mayor presencia de payroll, mayor actividad transaccional y más familias activas. Desde una perspectiva de negocio, esta separación podría justificar `k=4`.

Esta interpretación todavía no constituye la decisión final. Antes de adoptarla se evalúa si el grupo reaparece al cambiar las semillas, retirar bosques del ensamble y modificar la selección de continuadores.

**Resultado de la sensibilidad a continuadores.** El desempeño descriptivo del Random Forest permanece estable, con valores OOB de entre 0,83 y 0,84. Las soluciones alternativas también conservan tamaños de segmentos similares a los de la solución principal.

Sin embargo, el ARI global varía entre 0,48 y 0,69. Esto indica que una parte de las asignaciones individuales cambia cuando se modifica la muestra de continuadores. En consecuencia, las fronteras generales de la taxonomía no deben interpretarse como categorías naturales e inmutables.

El perfil asociado con payroll presenta una estabilidad considerablemente mayor. En ambas muestras alternativas vuelve a aparecer con 174–176 clientes, aproximadamente 60% de presencia de payroll, una mediana de 56–57 transacciones y cuatro familias activas. Su similitud de Jaccard con la solución principal se ubica entre 0,77 y 0,78 y conserva al 100% de los integrantes del perfil de referencia.

### Decisión final sobre la cantidad de perfiles

El dendrograma y las medidas globales indican que `k=3` constituye el corte geométricamente más marcado. Presenta el mayor espacio vertical y una estabilidad superior entre bosques individuales. El valor descriptivo de *silhouette* es prácticamente igual para `k=3` y `k=4`, por lo que no resuelve la elección por sí solo.

Al pasar a `k=4`, un segmento de 1.114 clientes se divide en dos ramas de 979 y 135 integrantes. La rama menor presenta una vinculación claramente asociada con la acreditación de haberes, mayor actividad transaccional y más familias de productos activas.

La estabilidad de esta rama se comprobó sobre el ensamble utilizado en la solución final. Al retirar alternativamente cada bosque, el perfil reaparece con entre 133 y 172 clientes, entre 59% y 67% de presencia de payroll y una similitud de Jaccard de entre 0,79 y 0,97. Frente a selecciones alternativas de continuadores también se conserva, con 174–176 clientes, aproximadamente 60% de payroll y una similitud de Jaccard de entre 0,77 y 0,78.

La partición completa presenta una sensibilidad mayor frente a la selección de continuadores, con valores ARI de entre 0,48 y 0,69. Por lo tanto, las fronteras entre los restantes segmentos deben considerarse provisionales.

En consecuencia, se adopta `k=4` porque agrega un perfil de negocio interpretable y reproducible que quedaría absorbido con `k=3`. La elección no se presenta como un óptimo matemático ni como una taxonomía definitiva: combina la estructura jerárquica, la estabilidad del ensamble y la utilidad descriptiva, y deberá validarse con nuevas cohortes y períodos adicionales.

## 14. Etiquetas finales y perfiles de negocio

Los números asignados por el clustering son arbitrarios y no poseen significado comercial. Por ese motivo, las denominaciones se establecen después de examinar las características observadas de cada grupo.

Se utilizan reglas explícitas y reproducibles:

- el grupo con mayor presencia de payroll se denomina **Vinculados por nómina**;
- el grupo con mayor presencia de crédito se denomina **Vinculados por crédito**;
- entre los restantes, el de menor actividad transaccional se denomina **Desvinculación avanzada**;
- el grupo restante, caracterizado por el uso de tarjetas, canales digitales, débitos y transferencias, se denomina **Transaccionales multicanal**.

Estas denominaciones resumen comportamientos predominantes y no implican causalidad ni una valoración individual del cliente.

In [63]:
# ============================================================
# Construcción y caracterización inicial de los cuatro clusters
# ============================================================

labels_raw = fcluster(
    linkage_final,
    t=K_FINAL,
    criterion="maxclust",
)


base_clusters = (
    strict_complete
    .set_index("numero_de_cliente")
    .loc[ids_cluster]
    .reset_index()
    .copy()
)


base_clusters[
    "cluster_raw"
] = labels_raw


assert len(base_clusters) == 1991
assert base_clusters[
    "numero_de_cliente"
].is_unique


# Indicadores de presencia y utilización.
base_clusters["payroll_activo"] = (
    base_clusters["cpayroll_trx"] > 0
).astype(int)

base_clusters["usa_tarjeta"] = (
    base_clusters[
        "tarjetas_transacciones"
    ] > 0
).astype(int)

base_clusters["usa_digital"] = (
    base_clusters[
        "transacciones_digitales"
    ] > 0
).astype(int)

base_clusters["emite_transferencias"] = (
    base_clusters[
        "ctransferencias_emitidas"
    ] > 0
).astype(int)

base_clusters["tiene_credito"] = (
    base_clusters[
        "cantidad_creditos"
    ] > 0
).astype(int)

base_clusters["tiene_inversion"] = (
    base_clusters[
        "cantidad_inversiones"
    ] > 0
).astype(int)

base_clusters["tiene_seguro"] = (
    base_clusters[
        "cantidad_seguros"
    ] > 0
).astype(int)

base_clusters["usa_debitos"] = (
    base_clusters[
        "debitos_automaticos"
    ] > 0
).astype(int)

base_clusters["paga_servicios"] = (
    base_clusters[
        "pagos_servicios"
    ] > 0
).astype(int)

base_clusters["saldo_negativo"] = (
    base_clusters[
        "mcuentas_saldo"
    ] < 0
).astype(int)

base_clusters[
    "mantenimiento_positivo"
] = (
    base_clusters[
        "mcomisiones_mantenimiento"
    ] > 0
).astype(int)


columnas_familias = [
    "payroll_activo",
    "usa_tarjeta",
    "usa_digital",
    "emite_transferencias",
    "tiene_credito",
    "tiene_inversion",
    "tiene_seguro",
    "usa_debitos",
    "paga_servicios",
]


base_clusters["familias_activas"] = (
    base_clusters[
        columnas_familias
    ]
    .sum(axis=1)
)


def pct(serie):
    return 100 * serie.mean()


perfil_clusters_raw = (
    base_clusters
    .groupby("cluster_raw")
    .agg(
        clientes=(
            "numero_de_cliente",
            "nunique",
        ),

        edad_mediana=(
            "cliente_edad",
            "median",
        ),

        antiguedad_mediana=(
            "cliente_antiguedad",
            "median",
        ),

        productos_mediana=(
            "cproductos",
            "median",
        ),

        familias_activas_mediana=(
            "familias_activas",
            "median",
        ),

        transacciones_t0=(
            "ctrx_quarter",
            "median",
        ),

        transacciones_3m=(
            "ctrx_quarter_media_3",
            "median",
        ),

        porcentaje_payroll=(
            "payroll_activo",
            pct,
        ),

        porcentaje_usa_tarjeta=(
            "usa_tarjeta",
            pct,
        ),

        porcentaje_digital=(
            "usa_digital",
            pct,
        ),

        porcentaje_transferencias=(
            "emite_transferencias",
            pct,
        ),

        porcentaje_credito=(
            "tiene_credito",
            pct,
        ),

        porcentaje_inversion=(
            "tiene_inversion",
            pct,
        ),

        porcentaje_seguro=(
            "tiene_seguro",
            pct,
        ),

        porcentaje_debitos=(
            "usa_debitos",
            pct,
        ),

        porcentaje_servicios=(
            "paga_servicios",
            pct,
        ),

        saldo_mediano=(
            "mcuentas_saldo",
            "median",
        ),

        porcentaje_saldo_negativo=(
            "saldo_negativo",
            pct,
        ),

        rentabilidad_mensual_mediana=(
            "mrentabilidad",
            "median",
        ),

        rentabilidad_anual_mediana=(
            "mrentabilidad_annual",
            "median",
        ),

        porcentaje_mantenimiento=(
            "mantenimiento_positivo",
            pct,
        ),
    )
    .reset_index()
)


perfil_clusters_raw[
    "porcentaje_clientes"
] = (
    100
    * perfil_clusters_raw[
        "clientes"
    ]
    / len(base_clusters)
)


display(
    perfil_clusters_raw.round(2)
)

,cluster_raw,clientes,edad_mediana,antiguedad_mediana,productos_mediana,familias_activas_mediana,transacciones_t0,transacciones_3m,porcentaje_payroll,porcentaje_usa_tarjeta,porcentaje_digital,porcentaje_transferencias,porcentaje_credito,porcentaje_inversion,porcentaje_seguro,porcentaje_debitos,porcentaje_servicios,saldo_mediano,porcentaje_saldo_negativo,rentabilidad_mensual_mediana,rentabilidad_anual_mediana,porcentaje_mantenimiento,porcentaje_clientes
0,1,135,45.00,69.00,7.00,4.00,72.00,77.00,66.67,56.30,93.33,61.48,9.63,12.59,21.48,52.59,40.00,"7,178.78",22.22,"1,153.53","9,086.31",17.78,6.78
1,2,979,47.00,115.00,7.00,3.00,37.00,40.00,1.02,69.87,86.52,32.18,4.29,7.66,21.55,61.59,19.71,117.30,43.51,"1,703.68","9,670.50",68.64,49.17
2,3,92,45.00,61.50,6.00,2.00,12.00,11.83,3.26,10.87,64.13,5.43,100.00,1.09,16.30,20.65,0.00,"-2,767.68",83.70,"2,067.44","21,948.20",73.91,4.62
3,4,785,45.00,67.00,5.00,1.00,4.00,5.33,1.02,9.94,43.95,3.18,0.51,2.04,10.32,14.39,0.76,"-7,420.04",88.54,"1,902.38","12,759.96",75.80,39.43


In [65]:
# ============================================================
# Asignación reproducible de nombres comerciales
# ============================================================

perfiles_raw = (
    perfil_clusters_raw
    .set_index("cluster_raw")
)


raw_payroll = (
    perfiles_raw[
        "porcentaje_payroll"
    ].idxmax()
)


raw_credito = (
    perfiles_raw[
        "porcentaje_credito"
    ].idxmax()
)


clusters_restantes = [
    cluster
    for cluster
    in perfiles_raw.index

    if cluster not in {
        raw_payroll,
        raw_credito,
    }
]


raw_desvinculado = (
    perfiles_raw.loc[
        clusters_restantes,
        "transacciones_t0",
    ]
    .idxmin()
)


raw_transaccional = next(
    cluster
    for cluster
    in clusters_restantes

    if cluster != raw_desvinculado
)


mapa_perfiles = {
    raw_transaccional: (
        "Transaccionales multicanal"
    ),

    raw_payroll: (
        "Vinculados por nómina"
    ),

    raw_credito: (
        "Vinculados por crédito"
    ),

    raw_desvinculado: (
        "Desvinculación avanzada"
    ),
}


codigo_perfil = {
    nombre: indice + 1

    for indice, nombre
    in enumerate(PROFILE_ORDER)
}


base_clusters["perfil"] = (
    base_clusters[
        "cluster_raw"
    ]
    .map(mapa_perfiles)
)


base_clusters["cluster_final"] = (
    base_clusters[
        "perfil"
    ]
    .map(codigo_perfil)
    .astype(int)
)


assert base_clusters[
    "perfil"
].notna().all()

assert base_clusters[
    "cluster_final"
].nunique() == 4


asignacion_clusters_finales = (
    base_clusters[
        [
            "numero_de_cliente",
            "cluster_final",
            "perfil",
        ]
    ]
    .copy()
)


assert asignacion_clusters_finales[
    "numero_de_cliente"
].is_unique


hash_etiquetas = hashlib.sha256(
    asignacion_clusters_finales
    .sort_values(
        "numero_de_cliente"
    )[
        "cluster_final"
    ]
    .to_numpy(
        dtype=np.int16
    )
    .tobytes()
).hexdigest()[:16]


print(
    "Hash de etiquetas finales:",
    hash_etiquetas,
)


display(
    base_clusters
    .groupby(
        [
            "cluster_final",
            "perfil",
        ],
        observed=True,
    )
    .size()
    .rename("clientes")
    .reset_index()
    .sort_values("cluster_final")
)

Hash de etiquetas finales: b2597691501eaff3


,cluster_final,perfil,clientes
0,1,Transaccionales multicanal,979
1,2,Vinculados por nómina,135
2,3,Vinculados por crédito,92
3,4,Desvinculación avanzada,785


In [66]:
def pct(s):
    return 100 * s.mean()

perfil_general = base_clusters.groupby(["cluster_final", "perfil"], observed=True).agg(
    clientes=("numero_de_cliente", "nunique"),
    edad_mediana=("cliente_edad", "median"),
    antiguedad_mediana=("cliente_antiguedad", "median"),
    productos_mediana=("cproductos", "median"),
    familias_activas_mediana=("familias_activas", "median"),
    transacciones_t0=("ctrx_quarter", "median"),
    transacciones_3m=("ctrx_quarter_media_3", "median"),
    porcentaje_payroll=("payroll_activo", pct),
    porcentaje_usa_tarjeta=("usa_tarjeta", pct),
    porcentaje_digital=("usa_digital", pct),
    porcentaje_transferencias=("emite_transferencias", pct),
    porcentaje_credito=("tiene_credito", pct),
    porcentaje_inversion=("tiene_inversion", pct),
    porcentaje_seguro=("tiene_seguro", pct),
    porcentaje_debitos=("usa_debitos", pct),
    porcentaje_servicios=("paga_servicios", pct),
    saldo_mediano=("mcuentas_saldo", "median"),
    porcentaje_saldo_negativo=("saldo_negativo", pct),
    rentabilidad_mediana=("mrentabilidad", "median"),
).reset_index()
perfil_general["porcentaje_clientes"] = 100 * perfil_general["clientes"] / perfil_general["clientes"].sum()
perfil_general = perfil_general.sort_values("cluster_final")
display(perfil_general.round(2))
n_por_perfil = perfil_general.set_index("perfil")["clientes"].astype(int).to_dict()

composicion_cohorte = pd.crosstab(
    base_clusters["perfil"], base_clusters["foto_mes_ancla"], normalize="index"
).mul(100).reindex(PROFILE_ORDER)
display(composicion_cohorte.round(2).rename_axis(index="perfil", columns="mes de anclaje (%)"))


,cluster_final,perfil,clientes,edad_mediana,antiguedad_mediana,productos_mediana,familias_activas_mediana,transacciones_t0,transacciones_3m,porcentaje_payroll,porcentaje_usa_tarjeta,porcentaje_digital,porcentaje_transferencias,porcentaje_credito,porcentaje_inversion,porcentaje_seguro,porcentaje_debitos,porcentaje_servicios,saldo_mediano,porcentaje_saldo_negativo,rentabilidad_mediana,porcentaje_clientes
0,1,Transaccionales multicanal,979,47.00,115.00,7.00,3.00,37.00,40.00,1.02,69.87,86.52,32.18,4.29,7.66,21.55,61.59,19.71,117.30,43.51,"1,703.68",49.17
1,2,Vinculados por nómina,135,45.00,69.00,7.00,4.00,72.00,77.00,66.67,56.30,93.33,61.48,9.63,12.59,21.48,52.59,40.00,"7,178.78",22.22,"1,153.53",6.78
2,3,Vinculados por crédito,92,45.00,61.50,6.00,2.00,12.00,11.83,3.26,10.87,64.13,5.43,100.00,1.09,16.30,20.65,0.00,"-2,767.68",83.70,"2,067.44",4.62
3,4,Desvinculación avanzada,785,45.00,67.00,5.00,1.00,4.00,5.33,1.02,9.94,43.95,3.18,0.51,2.04,10.32,14.39,0.76,"-7,420.04",88.54,"1,902.38",39.43


mes de anclaje (%),202105,202106
perfil,,
Transaccionales multicanal,51.79,48.21
Vinculados por nómina,60.74,39.26
Vinculados por crédito,43.48,56.52
Desvinculación avanzada,63.69,36.31


### Caracterización general

Los cuatro perfiles presentan diferencias coherentes en múltiples dimensiones de comportamiento.

**Transaccionales multicanal** reúne al 49,2% de los clientes. Se caracteriza por una actividad intermedia-alta, uso extendido de tarjetas y canales digitales, débitos automáticos y transferencias. La presencia de payroll y crédito es reducida.

**Vinculados por nómina** representa al 6,8%. Es el grupo de mayor actividad transaccional, mayor cantidad de familias activas, mayor uso de transferencias y mejor posición de saldo. La acreditación de haberes constituye su rasgo más distintivo.

**Vinculados por crédito** comprende al 4,6%. Todos sus integrantes poseen algún crédito, pero presentan baja actividad transaccional, escaso uso de tarjetas y una elevada proporción de saldos negativos. La rentabilidad observada no debe interpretarse aisladamente como una relación saludable, dado que puede estar asociada con crédito, cargos y comisiones.

**Desvinculación avanzada** representa al 39,4%. Presenta la menor cantidad de productos y familias activas, actividad transaccional mínima, escaso uso de tarjetas, transferencias y servicios, y la posición de saldo más deteriorada.

Los cuatro perfiles contienen clientes de mayo y junio. La composición temporal no es idéntica, pero ningún grupo se explica exclusivamente por una única cohorte.

In [70]:
def dibujar_caja_perfil(
    ax,
    x,
    y,
    ancho,
    alto,
    titulo,
    detalle,
    color,
):
    recuadro = FancyBboxPatch(
        (x, y),
        ancho,
        alto,
        boxstyle=(
            "round,pad=0.02,"
            "rounding_size=0.025"
        ),
        linewidth=1.8,
        edgecolor=color,
        facecolor=color + "20",
        transform=ax.transAxes,
    )

    ax.add_patch(recuadro)

    ax.text(
        x + ancho / 2,
        y + alto * 0.76,
        titulo,
        ha="center",
        va="center",
        weight="bold",
        fontsize=10,
        transform=ax.transAxes,
    )

    ax.text(
        x + ancho / 2,
        y + alto * 0.38,
        detalle,
        ha="center",
        va="center",
        fontsize=9,
        linespacing=1.35,
        transform=ax.transAxes,
    )


fig, ax = plt.subplots(
    figsize=(14, 6.5)
)

ax.axis("off")


dibujar_caja_perfil(
    ax,
    0.35,
    0.76,
    0.30,
    0.15,
    "Bajas con tres meses completos",
    f"n={len(base_clusters):,}",
    "#444444",
)


posiciones = np.linspace(
    0.01,
    0.76,
    len(PROFILE_ORDER),
)


for _, fila in perfil_general.iterrows():
    posicion = (
        int(fila["cluster_final"])
        - 1
    )

    x = posiciones[posicion]

    color = PROFILE_COLORS[
        fila["perfil"]
    ]

    ax.annotate(
        "",
        xy=(x + 0.115, 0.58),
        xytext=(0.50, 0.76),
        xycoords=ax.transAxes,
        arrowprops={
            "arrowstyle": "->",
            "color": "#777777",
            "linewidth": 1.5,
        },
    )

    detalle = (
        f"n={int(fila['clientes']):,}"
        f" · {fila['porcentaje_clientes']:.1f}%\n"
        f"Transacciones: "
        f"{fila['transacciones_t0']:.0f}"
        f" · Familias: "
        f"{fila['familias_activas_mediana']:.0f}\n"
        f"Payroll: "
        f"{fila['porcentaje_payroll']:.1f}%"
        f" · Digital: "
        f"{fila['porcentaje_digital']:.1f}%\n"
        f"Saldo mediano: "
        f"${fila['saldo_mediano']:,.0f}"
    )

    dibujar_caja_perfil(
        ax,
        x,
        0.18,
        0.23,
        0.32,
        fila["perfil"],
        detalle,
        color,
    )


ax.set_title(
    "Cuatro perfiles de clientes antes de dejar el paquete",
    fontsize=16,
    pad=12,
)


fig.tight_layout()


ruta_arbol_perfiles = (
    OUTPUT_DIR
    / "04_arbol_perfiles.png"
)


fig.savefig(
    ruta_arbol_perfiles,
    dpi=170,
    bbox_inches="tight",
)


plt.close(fig)

![Cuatro perfiles de clientes antes de dejar el paquete](./eda_miranda_outputs/04_arbol_perfiles.png)

## 15. Trayectorias previas y abandono de familias

Las curvas siguientes utilizan exactamente los mismos clientes en `t−2`, `t−1` y `t0`. El denominador permanece constante dentro de cada perfil. Una disminución observada es una trayectoria descriptiva; no permite atribuir la baja a una causa concreta.


In [71]:
filas_trayectoria = []
for _, r in base_clusters.iterrows():
    for etiqueta_mes, sufijo in [("t−2", "lag_2"), ("t−1", "lag_1"), ("t0", None)]:
        def valor(campo):
            return r[campo] if sufijo is None else r[f"{campo}_{sufijo}"]
        filas_trayectoria.append({
            "numero_de_cliente": r["numero_de_cliente"],
            "perfil": r["perfil"],
            "mes_relativo": etiqueta_mes,
            "transacciones": valor("ctrx_quarter"),
            "saldo": valor("mcuentas_saldo"),
            "productos": valor("cproductos"),
            "usa_tarjeta": 100 * (valor("tarjetas_transacciones") > 0),
            "usa_digital": 100 * (valor("transacciones_digitales") > 0),
        })
trayectorias = pd.DataFrame(filas_trayectoria)
orden_meses = ["t−2", "t−1", "t0"]
resumen_trayectorias = trayectorias.groupby(["perfil", "mes_relativo"], observed=True).agg(
    clientes=("numero_de_cliente", "nunique"),
    transacciones=("transacciones", "median"),
    saldo=("saldo", "median"),
    productos=("productos", "median"),
    usa_tarjeta=("usa_tarjeta", "mean"),
    usa_digital=("usa_digital", "mean"),
).reset_index()
resumen_trayectorias["mes_relativo"] = pd.Categorical(resumen_trayectorias["mes_relativo"], orden_meses, ordered=True)
display(resumen_trayectorias.sort_values(["perfil", "mes_relativo"]).round(2))


,perfil,mes_relativo,clientes,transacciones,saldo,productos,usa_tarjeta,usa_digital
2,Desvinculación avanzada,t−2,785,6.00,"-5,497.90",6.00,20.64,40.89
1,Desvinculación avanzada,t−1,785,5.00,"-7,009.22",6.00,14.78,40.38
0,Desvinculación avanzada,t0,785,4.00,"-7,420.04",5.00,9.94,43.95
5,Transaccionales multicanal,t−2,979,42.00,"2,119.47",7.00,78.75,87.84
4,Transaccionales multicanal,t−1,979,40.00,"1,902.23",7.00,75.89,85.19
3,Transaccionales multicanal,t0,979,37.00,117.30,7.00,69.87,86.52
8,Vinculados por crédito,t−2,92,12.50,"-2,167.04",6.00,19.57,61.96
7,Vinculados por crédito,t−1,92,12.00,"-3,069.22",6.00,14.13,61.96
6,Vinculados por crédito,t0,92,12.00,"-2,767.68",6.00,10.87,64.13
11,Vinculados por nómina,t−2,135,85.00,"19,791.00",7.00,60.00,92.59


In [72]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
metricas = [
    ("transacciones", "Transacciones trimestrales", "mediana"),
    ("saldo", "Saldo de cuentas", "$ nominales, mediana"),
    ("usa_tarjeta", "Uso de tarjeta", "% de clientes"),
    ("usa_digital", "Uso de canales digitales", "% de clientes"),
]
for ax, (col, titulo, ylab) in zip(axes.ravel(), metricas):
    for perfil in PROFILE_ORDER:
        d = resumen_trayectorias[resumen_trayectorias["perfil"] == perfil].sort_values("mes_relativo")
        ax.plot(d["mes_relativo"].astype(str), d[col], marker="o", lw=2.2,
                color=PROFILE_COLORS[perfil], label=f"{perfil} (n={n_por_perfil[perfil]:,})")
    ax.set_title(titulo)
    ax.set_xlabel("Mes relativo al último mes activo")
    ax.set_ylabel(ylab)
axes[0, 0].legend(frameon=False, fontsize=9)
fig.suptitle("Trayectorias balanceadas durante los tres meses previos", fontsize=15)
fig.text(0.5, 0.01, "Población: 1.991 bajas estrictas con historia completa; n constante dentro de cada perfil.", ha="center", fontsize=9)
fig.tight_layout(rect=[0, 0.03, 1, 0.96])
fig.savefig(OUTPUT_DIR / "05_trayectorias.png", dpi=160, bbox_inches="tight")
plt.close(fig)


![Trayectorias durante los tres meses previos](./eda_miranda_outputs/05_trayectorias.png)

### 15.1. Abandono total de productos y comportamientos

Se considera abandono cuando una variable es igual a cero en `t0`, pero había sido positiva en `t−1` o `t−2`. Por lo tanto, el indicador identifica la desaparición completa de una relación o comportamiento dentro de la ventana observada, no una mera reducción.

Los porcentajes se calculan sobre la totalidad de clientes de cada perfil. Esto permite medir cuánto representa cada señal de abandono dentro del segmento.

In [74]:
mapa_abandonos = {
    "abandono_cpayroll_trx_3m": "Payroll",
    "abandono_tarjetas_cantidad_3m": "Tenencia de tarjeta",
    "abandono_tarjetas_transacciones_3m": "Uso de tarjeta",
    "abandono_debitos_automaticos_3m": "Débitos automáticos",
    "abandono_pagos_servicios_3m": "Pago de servicios",
    "abandono_transacciones_digitales_3m": "Canales digitales",
    "abandono_cantidad_creditos_3m": "Créditos",
    "abandono_cantidad_inversiones_3m": "Inversiones",
    "abandono_cantidad_seguros_3m": "Seguros",
}

columnas_abandono = list(mapa_abandonos)
orden_familias = list(mapa_abandonos.values())

# ------------------------------------------------------------
# Auditoría de los indicadores
# ------------------------------------------------------------

auditoria_abandonos = pd.DataFrame({
    "variable": columnas_abandono,
    "nulos": [
        base_clusters[columna].isna().sum()
        for columna in columnas_abandono
    ],
    "valores_no_binarios": [
        (~base_clusters[columna].isin([0, 1])).sum()
        for columna in columnas_abandono
    ],
})

display(auditoria_abandonos)

assert auditoria_abandonos["nulos"].sum() == 0
assert auditoria_abandonos["valores_no_binarios"].sum() == 0

# ------------------------------------------------------------
# Porcentaje de abandono dentro de cada perfil
# ------------------------------------------------------------

abandono_largo = (
    base_clusters[
        ["perfil"] + columnas_abandono
    ]
    .melt(
        id_vars="perfil",
        var_name="variable",
        value_name="abandono",
    )
)

abandono_largo["familia"] = (
    abandono_largo["variable"]
    .map(mapa_abandonos)
)

abandono_resumen = (
    abandono_largo
    .groupby(
        ["perfil", "familia"],
        observed=True,
    )["abandono"]
    .mean()
    .mul(100)
    .reset_index()
)

tabla_abandono = (
    abandono_resumen
    .pivot(
        index="familia",
        columns="perfil",
        values="abandono",
    )
    .reindex(
        index=orden_familias,
        columns=PROFILE_ORDER,
    )
)

tabla_abandono.index.name = "Familia abandonada"

display(
    tabla_abandono.round(2)
)

# ------------------------------------------------------------
# Visualización
# ------------------------------------------------------------

etiquetas_perfiles = {
    "Transaccionales multicanal": "Transaccionales\nmulticanal",
    "Vinculados por nómina": "Vinculados\npor nómina",
    "Vinculados por crédito": "Vinculados\npor crédito",
    "Desvinculación avanzada": "Desvinculación\navanzada",
}

fig, ax = plt.subplots(
    figsize=(12.5, 6.5)
)

sns.heatmap(
    tabla_abandono,
    annot=True,
    fmt=".1f",
    cmap="YlOrBr",
    linewidths=0.5,
    linecolor="white",
    cbar_kws={
        "label": "% de clientes del perfil",
    },
    ax=ax,
)

ax.set_title(
    "Abandono total de productos o comportamientos entre t−2 y t0",
    fontsize=14,
    pad=14,
)

ax.set_xlabel("Perfil")
ax.set_ylabel("Familia")

ax.set_xticklabels(
    [
        (
            f"{etiquetas_perfiles[perfil]}"
            f"\n(n={n_por_perfil[perfil]:,})"
        )
        for perfil in PROFILE_ORDER
    ],
    rotation=0,
)

fig.tight_layout()

fig.savefig(
    OUTPUT_DIR / "06_abandono_familias.png",
    dpi=160,
    bbox_inches="tight",
)

plt.close(fig)

,variable,nulos,valores_no_binarios
0,abandono_cpayroll_trx_3m,0,0
1,abandono_tarjetas_cantidad_3m,0,0
2,abandono_tarjetas_transacciones_3m,0,0
3,abandono_debitos_automaticos_3m,0,0
4,abandono_pagos_servicios_3m,0,0
5,abandono_transacciones_digitales_3m,0,0
6,abandono_cantidad_creditos_3m,0,0
7,abandono_cantidad_inversiones_3m,0,0
8,abandono_cantidad_seguros_3m,0,0


perfil,Transaccionales multicanal,Vinculados por nómina,Vinculados por crédito,Desvinculación avanzada
Familia abandonada,,,,
Payroll,3.17,33.33,1.09,1.91
Tenencia de tarjeta,6.84,5.19,5.43,14.52
Uso de tarjeta,10.32,8.15,8.70,11.59
Débitos automáticos,7.05,7.41,5.43,7.01
Pago de servicios,9.30,6.67,0.00,1.27
Canales digitales,6.84,2.96,6.52,14.52
Créditos,6.03,3.70,0.00,3.06
Inversiones,7.15,5.93,0.00,0.76
Seguros,3.98,1.48,5.43,2.80


![Abandono de familias por perfil](./eda_miranda_outputs/06_abandono_familias.png)

### Lectura de los abandonos observados

El perfil **Vinculados por nómina** presenta la señal más distintiva: el 33,3% perdió la acreditación de haberes antes de `t0`, mientras que el 66,7% todavía la conserva en el último mes activo. En conjunto, los resultados indican que la vinculación reciente por nómina caracteriza a la totalidad del segmento, aunque no siempre permanezca activa hasta el final.

El perfil **Desvinculación avanzada** registra las mayores pérdidas de tenencia de tarjeta y canales digitales, ambas del 14,5%, además de una pérdida de uso de tarjeta del 11,6%. Estas transiciones se agregan a los bajos niveles de actividad y al deterioro de saldo ya observados.

Los **Transaccionales multicanal** exhiben una contracción más distribuida: se observan pérdidas de uso de tarjeta, pago de servicios, inversiones, débitos automáticos y canales digitales. Sin embargo, el perfil conserva en `t0` niveles de actividad superiores a los de los segmentos con mayor desvinculación.

En **Vinculados por crédito** no se observa abandono del crédito, resultado coherente con que el 100% mantiene algún crédito en `t0`. Los valores iguales a cero en otras familias no deben interpretarse automáticamente como retención: también pueden responder a una exposición previa reducida o inexistente.

Estos indicadores identifican transiciones desde un valor positivo en `t−2` o `t−1` hacia cero en `t0`. No distinguen entre cancelación voluntaria, vencimiento normal, decisión comercial u otra causa, por lo que describen señales previas a la salida sin establecer causalidad.

**Precaución.** “Perder” un crédito, inversión o seguro también puede reflejar vencimiento o cancelación normal. Estos indicadores describen el cambio observado; no demuestran insatisfacción ni causalidad.


## 16. Valor económico descriptivo

No se presume que rentabilidad, comisiones y márgenes sean componentes aditivos. Primero se audita esa hipótesis. En ausencia de un diccionario contable completo, cada campo se presenta por separado y no se calcula una supuesta “rentabilidad neta sin mantenimiento”.


In [75]:
%%sql

WITH auditoria AS (
    SELECT
        COUNT(*) AS filas_totales,

        COUNT(*) FILTER (
            WHERE mcomisiones IS NOT NULL
              AND mcomisiones_mantenimiento IS NOT NULL
              AND mcomisiones_otras IS NOT NULL
        ) AS filas_comisiones_comparables,

        COUNT(*) FILTER (
            WHERE mcomisiones IS NOT NULL
              AND mcomisiones_mantenimiento IS NOT NULL
              AND mcomisiones_otras IS NOT NULL
              AND ABS(
                  mcomisiones
                  - (
                      mcomisiones_mantenimiento
                      + mcomisiones_otras
                  )
              ) <= 0.01
        ) AS identidad_comisiones,

        COUNT(*) FILTER (
            WHERE mrentabilidad IS NOT NULL
              AND mcomisiones IS NOT NULL
              AND mactivos_margen IS NOT NULL
              AND mpasivos_margen IS NOT NULL
        ) AS filas_rentabilidad_comparables,

        COUNT(*) FILTER (
            WHERE mrentabilidad IS NOT NULL
              AND mcomisiones IS NOT NULL
              AND mactivos_margen IS NOT NULL
              AND mpasivos_margen IS NOT NULL
              AND ABS(
                  mrentabilidad
                  - (
                      mcomisiones
                      + mactivos_margen
                      + mpasivos_margen
                  )
              ) <= 0.01
        ) AS identidad_rentabilidad,

        MEDIAN(
            mrentabilidad
            - (
                mcomisiones
                + mactivos_margen
                + mpasivos_margen
            )
        ) FILTER (
            WHERE mrentabilidad IS NOT NULL
              AND mcomisiones IS NOT NULL
              AND mactivos_margen IS NOT NULL
              AND mpasivos_margen IS NOT NULL
        ) AS residuo_rentabilidad_mediano,

        MAX(
            ABS(
                mrentabilidad
                - (
                    mcomisiones
                    + mactivos_margen
                    + mpasivos_margen
                )
            )
        ) FILTER (
            WHERE mrentabilidad IS NOT NULL
              AND mcomisiones IS NOT NULL
              AND mactivos_margen IS NOT NULL
              AND mpasivos_margen IS NOT NULL
        ) AS residuo_rentabilidad_maximo

    FROM raw
)

SELECT
    *,

    ROUND(
        100.0
        * identidad_comisiones
        / NULLIF(filas_comisiones_comparables, 0),
        2
    ) AS porcentaje_identidad_comisiones,

    ROUND(
        100.0
        * identidad_rentabilidad
        / NULLIF(filas_rentabilidad_comparables, 0),
        2
    ) AS porcentaje_identidad_rentabilidad

FROM auditoria;

,filas_totales,filas_comisiones_comparables,identidad_comisiones,filas_rentabilidad_comparables,identidad_rentabilidad,residuo_rentabilidad_mediano,residuo_rentabilidad_maximo,porcentaje_identidad_comisiones,porcentaje_identidad_rentabilidad
0,983061,983061,642692,983061,2255,31.65,"115,270.69",65.38,0.23


In [76]:
def porcentaje_positivo(serie):
    valores = serie.dropna()

    if len(valores) == 0:
        return np.nan

    return 100 * (valores > 0).mean()


perfil_economico = (
    base_clusters
    .groupby(
        ["cluster_final", "perfil"],
        observed=True,
    )
    .agg(
        clientes=(
            "numero_de_cliente",
            "nunique",
        ),

        rentabilidad_mensual_mediana=(
            "mrentabilidad",
            "median",
        ),

        rentabilidad_mensual_p25=(
            "mrentabilidad",
            lambda serie: serie.quantile(0.25),
        ),

        rentabilidad_mensual_p75=(
            "mrentabilidad",
            lambda serie: serie.quantile(0.75),
        ),

        porcentaje_rentabilidad_positiva=(
            "mrentabilidad",
            porcentaje_positivo,
        ),

        rentabilidad_anual_mediana=(
            "mrentabilidad_annual",
            "median",
        ),

        comisiones_totales_medianas=(
            "mcomisiones",
            "median",
        ),

        mantenimiento_mediano=(
            "mcomisiones_mantenimiento",
            "median",
        ),

        porcentaje_mantenimiento_positivo=(
            "mcomisiones_mantenimiento",
            porcentaje_positivo,
        ),

        otras_comisiones_medianas=(
            "mcomisiones_otras",
            "median",
        ),

        margen_activos_mediano=(
            "mactivos_margen",
            "median",
        ),

        margen_pasivos_mediano=(
            "mpasivos_margen",
            "median",
        ),

        saldo_mediano=(
            "mcuentas_saldo",
            "median",
        ),

        porcentaje_saldo_negativo=(
            "saldo_negativo",
            pct,
        ),
    )
    .reset_index()
    .sort_values("cluster_final")
)

display(
    perfil_economico.round(2)
)

,cluster_final,perfil,clientes,rentabilidad_mensual_mediana,rentabilidad_mensual_p25,rentabilidad_mensual_p75,porcentaje_rentabilidad_positiva,rentabilidad_anual_mediana,comisiones_totales_medianas,mantenimiento_mediano,porcentaje_mantenimiento_positivo,otras_comisiones_medianas,margen_activos_mediano,margen_pasivos_mediano,saldo_mediano,porcentaje_saldo_negativo
0,1,Transaccionales multicanal,979,"1,703.68",397.50,"2,578.16",85.50,"9,670.50","1,721.92","1,774.04",68.64,"1,721.92",-49.32,153.18,117.30,43.51
1,2,Vinculados por nómina,135,"1,153.53",477.88,"3,074.62",86.67,"9,086.31",372.29,0.00,17.78,372.29,-3.28,359.65,"7,178.78",22.22
2,3,Vinculados por crédito,92,"2,067.44",414.30,"3,001.23",88.04,"21,948.20","1,592.20","1,774.04",73.91,"1,592.20",287.92,10.36,"-2,767.68",83.70
3,4,Desvinculación avanzada,785,"1,902.38","1,145.57","2,797.87",90.70,"12,759.96","1,592.20","1,774.04",75.80,"1,592.20",155.23,1.06,"-7,420.04",88.54


### Lectura económica

La igualdad entre comisiones totales, mantenimiento y otras comisiones se verifica en el 65,38% de los registros. Por lo tanto, estos campos guardan relación en una parte importante de la base, pero no constituyen una identidad contable universal.

La rentabilidad coincide con la suma de comisiones y márgenes solamente en el 0,23% de los registros. En consecuencia, no se interpreta la rentabilidad como la suma de esos conceptos ni se calcula una rentabilidad corregida restando el mantenimiento.

El perfil **Vinculados por crédito** presenta la mayor rentabilidad mensual y anual mediana registrada, pero también un 83,7% de clientes con saldo negativo. Esta combinación debe interpretarse junto con su vinculación crediticia y no como evidencia aislada de una relación económicamente saludable.

La **Desvinculación avanzada** también presenta rentabilidad registrada positiva, pese a que el 88,5% mantiene saldo negativo y el 75,8% registra mantenimiento positivo. La rentabilidad observada puede estar asociada con cargos, financiación u otros conceptos no identificables de forma aditiva.

Los **Vinculados por nómina** presentan la menor rentabilidad mensual mediana, pero muestran el saldo más favorable y la menor proporción de saldos negativos. Esto refleja un perfil financiero diferente, sin implicar por sí solo menor valor futuro.

Los indicadores se presentan de manera descriptiva y por separado. Con la información disponible no se estiman costo de riesgo, valor futuro, rentabilidad incremental ni efecto causal de una eventual acción de retención.

In [77]:
componentes_economicos = [
    (
        "Rentabilidad mensual",
        "rentabilidad_mensual_mediana",
    ),
    (
        "Rentabilidad anual registrada",
        "rentabilidad_anual_mediana",
    ),
    (
        "Comisiones totales",
        "comisiones_totales_medianas",
    ),
    (
        "Mantenimiento",
        "mantenimiento_mediano",
    ),
    (
        "Otras comisiones",
        "otras_comisiones_medianas",
    ),
    (
        "Margen de activos",
        "margen_activos_mediano",
    ),
    (
        "Margen de pasivos",
        "margen_pasivos_mediano",
    ),
    (
        "Saldo de cuentas",
        "saldo_mediano",
    ),
]

plot_economico = (
    perfil_economico
    .set_index("perfil")
    .reindex(PROFILE_ORDER)
)

fig, axes = plt.subplots(
    2,
    4,
    figsize=(17, 9),
)

for ax, (titulo, columna) in zip(
    axes.ravel(),
    componentes_economicos,
):
    valores = plot_economico[columna]

    barras = ax.bar(
        range(len(valores)),
        valores,
        color=[
            PROFILE_COLORS[perfil]
            for perfil in valores.index
        ],
    )

    ax.axhline(
        0,
        color="#555555",
        linewidth=0.9,
    )

    ax.set_xticks(
        range(len(valores))
    )

    ax.set_xticklabels(
        [
            (
                f"{etiquetas_perfiles[perfil]}"
                f"\nn={n_por_perfil[perfil]:,}"
            )
            for perfil in valores.index
        ],
        fontsize=7.5,
        rotation=0,
    )

    ax.set_title(
        titulo,
        fontsize=11,
    )

    ax.set_ylabel(
        "$ nominales · mediana",
        fontsize=9,
    )

    for barra, valor in zip(
        barras,
        valores,
    ):
        desplazamiento = (
            3
            if valor >= 0
            else -12
        )

        ax.annotate(
            f"{valor:,.0f}",
            (
                barra.get_x()
                + barra.get_width() / 2,
                valor,
            ),
            xytext=(0, desplazamiento),
            textcoords="offset points",
            ha="center",
            va=(
                "bottom"
                if valor >= 0
                else "top"
            ),
            fontsize=8,
        )

fig.suptitle(
    "Indicadores económicos registrados por perfil",
    fontsize=16,
)

fig.text(
    0.5,
    0.01,
    (
        "Cada panel utiliza su propia escala. "
        "Los indicadores se presentan por separado y no son aditivos."
    ),
    ha="center",
    fontsize=9,
)

fig.tight_layout(
    rect=[0, 0.04, 1, 0.95]
)

fig.savefig(
    OUTPUT_DIR / "07_economia_descriptiva.png",
    dpi=160,
    bbox_inches="tight",
)

plt.close(fig)

![Indicadores económicos por perfil](./eda_miranda_outputs/07_economia_descriptiva.png)

## 17. Generalización a historia parcial

Los perfiles se observan directamente en 1.991 bajas. Para las 1.986 restantes se estima el perfil compatible a partir de `t0`. Las etiquetas observadas nunca se reemplazan. La validación por mes mide fidelidad respecto de una taxonomía construida conjuntamente con mayo y junio; no es una validación causal ni completamente inductiva.


In [78]:
# ============================================================
# 1. Identificación explícita de las variables temporales
# ============================================================

columnas_temporales_generadas = set()

for campo in campos_temporales:
    columnas_temporales_generadas.update({
        f"{campo}_lag_1",
        f"{campo}_lag_2",
        f"{campo}_delta_1",
        f"{campo}_delta_2",
        f"{campo}_media_3",
        f"{campo}_minimo_3",
        f"{campo}_maximo_3",
        f"{campo}_rango_3",
        f"{campo}_pendiente_3",
    })

for campo in campos_eventos:
    columnas_temporales_generadas.update({
        f"{campo}_lag_1",
        f"{campo}_lag_2",
        f"{campo}_delta_1",
        f"{campo}_delta_2",
        f"abandono_{campo}_3m",
    })


# Para extender la taxonomía se usan solamente variables
# contemporáneas disponibles en el último mes activo.

FEATURES_EXTENSION_T0 = [
    columna
    for columna in FEATURES_RF
    if columna not in columnas_temporales_generadas
]

features_temporales_retiradas = [
    columna
    for columna in FEATURES_RF
    if columna in columnas_temporales_generadas
]


print(
    "Features utilizadas por el clustering:",
    len(FEATURES_RF),
)

print(
    "Features temporales retiradas para la extensión:",
    len(features_temporales_retiradas),
)

print(
    "Features contemporáneas utilizadas:",
    len(FEATURES_EXTENSION_T0),
)


assert len(columnas_temporales_generadas) == 117

assert (
    set(FEATURES_EXTENSION_T0)
    .isdisjoint(columnas_temporales_generadas)
)


# ============================================================
# 2. Modelo que reproduce los perfiles utilizando sólo t0
# ============================================================

X_extension = (
    base_clusters[FEATURES_EXTENSION_T0]
    .replace(
        [np.inf, -np.inf],
        np.nan,
    )
)

y_extension = (
    base_clusters["cluster_final"]
    .astype(int)
)

mes_extension = (
    base_clusters["foto_mes_ancla"]
    .astype(int)
)


modelo_extension = Pipeline([
    (
        "imputacion",
        SimpleImputer(
            strategy="median",
            add_indicator=True,
            keep_empty_features=True,
        ),
    ),

    (
        "bosque",
        RandomForestClassifier(
            n_estimators=500,
            min_samples_leaf=10,
            max_features="sqrt",
            class_weight="balanced",
            n_jobs=-1,
            random_state=214363,
        ),
    ),
])


# ============================================================
# 3. Validación fuera de mes
# ============================================================

logo = LeaveOneGroupOut()

pred_fuera_mes = cross_val_predict(
    modelo_extension,
    X_extension,
    y_extension,
    groups=mes_extension,
    cv=logo,
    method="predict",
)


metricas_extension = pd.DataFrame([{
    "balanced_accuracy": balanced_accuracy_score(
        y_extension,
        pred_fuera_mes,
    ),

    "f1_macro": f1_score(
        y_extension,
        pred_fuera_mes,
        average="macro",
    ),
}])


display(
    metricas_extension.round(4)
)

display(
    pd.DataFrame(
        classification_report(
            y_extension,
            pred_fuera_mes,
            output_dict=True,
            zero_division=0,
        )
    )
    .T
    .round(3)
)


# ============================================================
# 4. Ajuste final y proyección de las historias parciales
# ============================================================

modelo_extension.fit(
    X_extension,
    y_extension,
)


X_strict_all_t0 = (
    strict_all[FEATURES_EXTENSION_T0]
    .replace(
        [np.inf, -np.inf],
        np.nan,
    )
)


strict_all = strict_all.copy()

strict_all["cluster_predicho_t0"] = (
    modelo_extension.predict(
        X_strict_all_t0
    )
)

probabilidades_strict = (
    modelo_extension.predict_proba(
        X_strict_all_t0
    )
)

strict_all["confianza_prediccion"] = (
    probabilidades_strict.max(axis=1)
)


strict_extended = strict_all.merge(
    asignacion_clusters_finales[
        [
            "numero_de_cliente",
            "cluster_final",
            "perfil",
        ]
    ],
    on="numero_de_cliente",
    how="left",
    validate="one_to_one",
)


strict_extended["origen_cluster"] = np.where(
    strict_extended["cluster_final"].notna(),
    "observado_3m",
    "proyectado_t0",
)


strict_extended["cluster_analisis"] = (
    strict_extended["cluster_final"]
    .fillna(
        strict_extended["cluster_predicho_t0"]
    )
    .astype(int)
)


nombre_por_codigo = {
    codigo: perfil
    for perfil, codigo in codigo_perfil.items()
}


strict_extended["perfil_analisis"] = (
    strict_extended["cluster_analisis"]
    .map(nombre_por_codigo)
)


resumen_extension = (
    strict_extended
    .groupby(
        [
            "origen_cluster",
            "perfil_analisis",
        ],
        observed=True,
    )
    .agg(
        clientes=(
            "numero_de_cliente",
            "nunique",
        ),

        confianza_mediana=(
            "confianza_prediccion",
            "median",
        ),
    )
    .reset_index()
)


resumen_extension["porcentaje"] = (
    100
    * resumen_extension["clientes"]
    / resumen_extension
      .groupby("origen_cluster")["clientes"]
      .transform("sum")
)


display(
    resumen_extension.round(2)
)


# ============================================================
# 5. Controles de integridad
# ============================================================

conteos_origen = (
    strict_extended["origen_cluster"]
    .value_counts()
    .to_dict()
)


print(
    "Distribución por origen:",
    conteos_origen,
)

print(
    "Clientes extendidos:",
    strict_extended["numero_de_cliente"].nunique(),
)


assert strict_extended["numero_de_cliente"].is_unique
assert len(strict_extended) == 3977
assert conteos_origen["observado_3m"] == 1991
assert conteos_origen["proyectado_t0"] == 1986

Features utilizadas por el clustering: 265
Features temporales retiradas para la extensión: 117
Features contemporáneas utilizadas: 148


,balanced_accuracy,f1_macro
0,0.87,0.85


,precision,recall,f1-score,support
1,0.91,0.84,0.87,979.00
2,0.65,0.72,0.68,135.00
3,0.88,0.99,0.93,92.00
4,0.88,0.94,0.91,785.00
accuracy,0.88,0.88,0.88,0.88
macro avg,0.83,0.87,0.85,"1,991.00"
weighted avg,0.88,0.88,0.88,"1,991.00"


,origen_cluster,perfil_analisis,clientes,confianza_mediana,porcentaje
0,observado_3m,Desvinculación avanzada,785,0.84,39.43
1,observado_3m,Transaccionales multicanal,979,0.74,49.17
2,observado_3m,Vinculados por crédito,92,0.87,4.62
3,observado_3m,Vinculados por nómina,135,0.74,6.78
4,proyectado_t0,Desvinculación avanzada,805,0.82,40.53
5,proyectado_t0,Transaccionales multicanal,966,0.73,48.64
6,proyectado_t0,Vinculados por crédito,92,0.80,4.63
7,proyectado_t0,Vinculados por nómina,123,0.57,6.19


Distribución por origen: {'observado_3m': 1991, 'proyectado_t0': 1986}
Clientes extendidos: 3977


In [79]:
# ============================================================
# Comparación de perfiles observados y proyectados
# ============================================================

tabla_extension = (
    resumen_extension
    .pivot(
        index="perfil_analisis",
        columns="origen_cluster",
        values="porcentaje",
    )
    .reindex(
        index=PROFILE_ORDER,
        columns=[
            "observado_3m",
            "proyectado_t0",
        ],
    )
)

tabla_extension["diferencia_pp"] = (
    tabla_extension["proyectado_t0"]
    - tabla_extension["observado_3m"]
)


confianza_proyectada = (
    resumen_extension[
        resumen_extension["origen_cluster"]
        == "proyectado_t0"
    ]
    .set_index("perfil_analisis")
    .reindex(PROFILE_ORDER)[
        "confianza_mediana"
    ]
)


comparacion_extension = (
    tabla_extension
    .rename(
        columns={
            "observado_3m": "porcentaje_observado",
            "proyectado_t0": "porcentaje_proyectado",
        }
    )
    .assign(
        confianza_mediana_proyectada=
        confianza_proyectada
    )
)

display(
    comparacion_extension.round(2)
)


# ============================================================
# Visualización
# ============================================================

fig, axes = plt.subplots(
    1,
    2,
    figsize=(15, 6),
    gridspec_kw={
        "width_ratios": [1.45, 1],
    },
)

posiciones = np.arange(
    len(PROFILE_ORDER)
)

ancho = 0.36


# Panel 1: distribuciones

barras_observadas = axes[0].bar(
    posiciones - ancho / 2,
    tabla_extension["observado_3m"],
    width=ancho,
    color="#2F6B9A",
    label="Observado con 3 meses",
)

barras_proyectadas = axes[0].bar(
    posiciones + ancho / 2,
    tabla_extension["proyectado_t0"],
    width=ancho,
    color="#B6C9DA",
    label="Proyectado utilizando t0",
)

axes[0].set_title(
    "Distribución de los perfiles"
)

axes[0].set_ylabel(
    "% dentro de cada origen"
)

axes[0].set_xticks(
    posiciones
)

axes[0].set_xticklabels(
    [
        etiquetas_perfiles[perfil]
        for perfil in PROFILE_ORDER
    ],
    fontsize=9,
)

axes[0].legend(
    frameon=False
)

axes[0].bar_label(
    barras_observadas,
    fmt="%.1f%%",
    padding=3,
    fontsize=9,
)

axes[0].bar_label(
    barras_proyectadas,
    fmt="%.1f%%",
    padding=3,
    fontsize=9,
)


# Panel 2: diferencia proyectado menos observado

colores_diferencias = [
    PROFILE_COLORS[perfil]
    for perfil in PROFILE_ORDER
]

barras_diferencias = axes[1].barh(
    posiciones,
    tabla_extension["diferencia_pp"],
    color=colores_diferencias,
)

axes[1].axvline(
    0,
    color="#555555",
    linewidth=1,
)

axes[1].set_title(
    "Diferencia de composición"
)

axes[1].set_xlabel(
    "Proyectado − observado (puntos porcentuales)"
)

axes[1].set_yticks(
    posiciones
)

axes[1].set_yticklabels(
    [
        etiquetas_perfiles[perfil]
        for perfil in PROFILE_ORDER
    ],
    fontsize=9,
)

axes[1].invert_yaxis()


for barra, diferencia in zip(
    barras_diferencias,
    tabla_extension["diferencia_pp"],
):
    axes[1].annotate(
        f"{diferencia:+.2f}",
        (
            diferencia,
            barra.get_y()
            + barra.get_height() / 2,
        ),
        xytext=(
            4 if diferencia >= 0 else -4,
            0,
        ),
        textcoords="offset points",
        ha=(
            "left"
            if diferencia >= 0
            else "right"
        ),
        va="center",
        fontsize=9,
    )


fig.suptitle(
    "Extensión de los perfiles a las bajas con historia parcial",
    fontsize=15,
)

fig.text(
    0.5,
    0.01,
    (
        "Observados: n=1.991 · Proyectados: n=1.986 · "
        "Modelo de extensión basado exclusivamente en variables de t0."
    ),
    ha="center",
    fontsize=9,
)

fig.tight_layout(
    rect=[0, 0.04, 1, 0.94]
)

fig.savefig(
    OUTPUT_DIR / "08_observado_proyectado.png",
    dpi=160,
    bbox_inches="tight",
)

plt.close(fig)

origen_cluster,porcentaje_observado,porcentaje_proyectado,diferencia_pp,confianza_mediana_proyectada
perfil_analisis,,,,
Transaccionales multicanal,49.17,48.64,-0.53,0.73
Vinculados por nómina,6.78,6.19,-0.59,0.57
Vinculados por crédito,4.62,4.63,0.01,0.80
Desvinculación avanzada,39.43,40.53,1.11,0.82


## 18. Sensibilidad con censurados

Se realizan dos comprobaciones diferentes:

1. **Compatibilidad:** asignar a todos los censurados uno de los perfiles ya definidos.
2. **Sensibilidad estructural:** incorporar censurados con tres meses completos a la matriz de proximidad, repetir el clustering y medir cuánto cambia la partición de las bajas estrictas compartidas.

La segunda prueba sí permite detectar una alteración de la estructura; la primera, por construcción, no puede crear perfiles nuevos.


In [ ]:
sensibilidad_t0 = con.execute(
    """
    SELECT
        e.numero_evento, e.ultimo_mes_activo AS foto_mes_ancla,
        e.primer_mes_ausente, e.poblacion,
        f.* EXCLUDE (foto_mes)
    FROM eventos_baja e
    JOIN fe_panel f ON e.numero_de_cliente=f.numero_de_cliente
                   AND e.ultimo_mes_activo=f.foto_mes
    WHERE e.poblacion IN ('censurada','intermitente')
    ORDER BY poblacion, foto_mes_ancla, f.numero_de_cliente
    """
).df()
sensibilidad_t0["historia_3m_completa"] = (
    (sensibilidad_t0["observaciones_hasta_mes"] >= 3)
    & (sensibilidad_t0["salto_1"] == 1)
    & (sensibilidad_t0["salto_2"] == 2)
).astype(int)
sensibilidad_t0["cluster_compatible"] = modelo_extension.predict(sensibilidad_t0[FEATURES_EXTENSION])
sensibilidad_t0["perfil_compatible"] = sensibilidad_t0["cluster_compatible"].map(nombre_por_codigo)

distribucion_sensibilidad = sensibilidad_t0.groupby(["poblacion", "perfil_compatible"], observed=True).size().rename("clientes").reset_index()
distribucion_sensibilidad["porcentaje"] = 100 * distribucion_sensibilidad["clientes"] / distribucion_sensibilidad.groupby("poblacion")["clientes"].transform("sum")
display(distribucion_sensibilidad.round(2))

cens_complete = sensibilidad_t0[(sensibilidad_t0["poblacion"] == "censurada") & (sensibilidad_t0["historia_3m_completa"] == 1)].copy()
combined_complete = pd.concat([base_clusters, cens_complete], ignore_index=True, sort=False)
combined_complete = combined_complete.sort_values("numero_de_cliente", kind="mergesort").reset_index(drop=True)

proximidades_ampliadas = []
for modelo in rf_models:
    hojas = modelo.apply(combined_complete[FEATURES_RF].astype(np.float32))
    proximidades_ampliadas.append(matriz_proximidad(hojas))
P_ampliada = np.mean(proximidades_ampliadas, axis=0, dtype=np.float64)
Z_ampliada = linkage(squareform(np.clip(1-P_ampliada,0,1), checks=False), method="average")
labels_ampliadas = fcluster(Z_ampliada, t=K_FINAL, criterion="maxclust")
combined_complete["cluster_ampliado_raw"] = labels_ampliadas

labels_estrictas_ampliadas = combined_complete.loc[
    combined_complete["numero_de_cliente"].isin(base_clusters["numero_de_cliente"]),
    ["numero_de_cliente", "cluster_ampliado_raw"],
].set_index("numero_de_cliente").loc[ids_cluster, "cluster_ampliado_raw"].to_numpy()

ari_censura_estructural = adjusted_rand_score(labels_raw, labels_estrictas_ampliadas)
resumen_censura_estructural = pd.DataFrame([{
    "estrictas_completas": len(base_clusters),
    "censuradas_completas": len(cens_complete),
    "ARI_estrictas_antes_despues": ari_censura_estructural,
    "tamanios_clustering_ampliado": sorted(pd.Series(labels_ampliadas).value_counts().tolist(), reverse=True),
}])
display(resumen_censura_estructural.round(4))

if ari_censura_estructural < 0.60:
    display(Markdown(
        "**Resultado de sensibilidad:** al incorporar censurados la estructura cambia de forma "
        f"material (ARI={ari_censura_estructural:.2f}). Los perfiles principales describen con utilidad "
        "a las bajas estrictas, pero no deben presentarse como una taxonomía universal o estable fuera de esa población."
    ))
else:
    display(Markdown(
        f"**Resultado de sensibilidad:** la estructura conserva una concordancia sustantiva (ARI={ari_censura_estructural:.2f}) "
        "al incorporar censurados, aunque esos casos no se reclasifican como bajas confirmadas."
    ))


,poblacion,perfil_compatible,clientes,porcentaje
0,censurada,Desvinculación avanzada,473,42.88
1,censurada,Digitales selectivos,297,26.93
2,censurada,Vinculados multicanal,285,25.84
3,censurada,Vinculados por nómina,48,4.35
4,intermitente,Desvinculación avanzada,3,13.04
5,intermitente,Digitales selectivos,2,8.70
6,intermitente,Vinculados multicanal,13,56.52
7,intermitente,Vinculados por nómina,5,21.74


,estrictas_completas,censuradas_completas,ARI_estrictas_antes_despues,tamanios_clustering_ampliado
0,1991,1097,0.49,"[2036, 809, 204, 39]"


**Resultado de sensibilidad:** al incorporar censurados la estructura cambia de forma material (ARI=0.49). Los perfiles principales describen con utilidad a las bajas estrictas, pero no deben presentarse como una taxonomía universal o estable fuera de esa población.

![Comparación entre perfiles observados y proyectados](./eda_miranda_outputs/08_observado_proyectado.png)

### Interpretación de la extensión

El modelo basado exclusivamente en variables contemporáneas alcanza una balanced accuracy de 0,87 y un F1 macro de 0,85 al evaluarse fuera del mes utilizado para el ajuste. La distribución proyectada para las 1.986 historias parciales conserva una composición muy similar a la observada en los 1.991 clientes con historia completa.

La mayor diferencia corresponde a **Desvinculación avanzada**, con un incremento de 1,10 puntos porcentuales. Las restantes diferencias se mantienen por debajo de 0,60 puntos porcentuales.

El perfil **Vinculados por nómina** es el más difícil de reproducir utilizando solamente `t0`: presenta un F1 de 0,68 y una confianza mediana de 0,57 entre los casos proyectados. Por lo tanto, sus asignaciones individuales deben interpretarse con mayor cautela.

La confianza informada corresponde a la proporción de votos del bosque para la clase seleccionada. No constituye una probabilidad calibrada de pertenencia ni transforma las asignaciones proyectadas en observaciones directas.

## 19. Sensibilidad con casos censurados e intermitentes

Se realizan dos comprobaciones diferentes:

1. **Compatibilidad descriptiva:** se asigna a cada caso censurado o intermitente el perfil existente más compatible con su información en `t0`. Esta asignación no los convierte en bajas confirmadas ni permite descubrir perfiles nuevos.

2. **Sensibilidad estructural:** se incorporan los casos censurados con tres meses completos, se reconstruyen los perfiles de último corte y se repite el agrupamiento jerárquico mediante Ward. Esta prueba permite evaluar cuánto cambia la segmentación principal al modificar la población analizada.

Los casos intermitentes se mantienen separados porque vuelven a aparecer posteriormente y, por lo tanto, no representan una salida confirmada.


In [80]:
%%sql

CREATE OR REPLACE TABLE sensibilidad_poblaciones AS

SELECT
    e.numero_evento,
    e.ultimo_mes_activo AS foto_mes_ancla,
    e.primer_mes_ausente,
    e.poblacion,

    f.* EXCLUDE (foto_mes),

    CASE
        WHEN f.observaciones_hasta_mes >= 3
         AND f.salto_1 = 1
         AND f.salto_2 = 2
            THEN 1
        ELSE 0
    END AS historia_3m_completa

FROM eventos_baja AS e

INNER JOIN fe_panel AS f
    ON e.numero_de_cliente =
       f.numero_de_cliente

   AND e.ultimo_mes_activo =
       f.foto_mes

WHERE e.poblacion IN (
    'censurada',
    'intermitente'
);


SELECT
    poblacion,

    COUNT(*) AS filas,

    COUNT(
        DISTINCT numero_de_cliente
    ) AS clientes,

    SUM(
        historia_3m_completa
    ) AS clientes_con_historia_completa

FROM sensibilidad_poblaciones

GROUP BY poblacion
ORDER BY poblacion;

,poblacion,filas,clientes,clientes_con_historia_completa
0,censurada,1103,1103,"1,097.00"
1,intermitente,23,23,8.00


In [81]:
sensibilidad_t0 = %sql SELECT * FROM sensibilidad_poblaciones ORDER BY poblacion, foto_mes_ancla, numero_de_cliente


# ============================================================
# 1. Control de integridad
# ============================================================

features_faltantes_sensibilidad = [
    columna
    for columna in FEATURES_EXTENSION_T0
    if columna not in sensibilidad_t0.columns
]


print(
    "Features faltantes:",
    features_faltantes_sensibilidad,
)

print(
    "Clientes:",
    sensibilidad_t0[
        "numero_de_cliente"
    ].nunique(),
)

print(
    "Duplicados:",
    sensibilidad_t0[
        "numero_de_cliente"
    ].duplicated().sum(),
)


assert not features_faltantes_sensibilidad
assert sensibilidad_t0["numero_de_cliente"].is_unique


# ============================================================
# 2. Asignación del perfil compatible
# ============================================================

X_sensibilidad_t0 = (
    sensibilidad_t0[
        FEATURES_EXTENSION_T0
    ]
    .replace(
        [np.inf, -np.inf],
        np.nan,
    )
)


sensibilidad_t0["cluster_compatible"] = (
    modelo_extension.predict(
        X_sensibilidad_t0
    )
)


probabilidades_sensibilidad = (
    modelo_extension.predict_proba(
        X_sensibilidad_t0
    )
)


sensibilidad_t0["confianza_compatibilidad"] = (
    probabilidades_sensibilidad.max(axis=1)
)


sensibilidad_t0["perfil_compatible"] = (
    sensibilidad_t0["cluster_compatible"]
    .map(nombre_por_codigo)
)


assert sensibilidad_t0[
    "perfil_compatible"
].notna().all()


# ============================================================
# 3. Distribución descriptiva
# ============================================================

distribucion_sensibilidad = (
    sensibilidad_t0
    .groupby(
        [
            "poblacion",
            "perfil_compatible",
        ],
        observed=True,
    )
    .agg(
        clientes=(
            "numero_de_cliente",
            "nunique",
        ),

        confianza_mediana=(
            "confianza_compatibilidad",
            "median",
        ),
    )
    .reset_index()
)


distribucion_sensibilidad[
    "porcentaje_poblacion"
] = (
    100
    * distribucion_sensibilidad["clientes"]
    / distribucion_sensibilidad
      .groupby("poblacion")["clientes"]
      .transform("sum")
)


display(
    distribucion_sensibilidad.round(2)
)

Features faltantes: []
Clientes: 1126
Duplicados: 0


,poblacion,perfil_compatible,clientes,confianza_mediana,porcentaje_poblacion
0,censurada,Desvinculación avanzada,349,0.79,31.64
1,censurada,Transaccionales multicanal,509,0.73,46.15
2,censurada,Vinculados por crédito,169,0.77,15.32
3,censurada,Vinculados por nómina,76,0.64,6.89
4,intermitente,Desvinculación avanzada,5,0.60,21.74
5,intermitente,Transaccionales multicanal,12,0.80,52.17
6,intermitente,Vinculados por nómina,6,0.73,26.09


La compatibilidad descriptiva muestra dos resultados relevantes:
- Los censurados se parecen principalmente a transaccionales multicanal (46,15%), pero tienen mucha mayor presencia del perfil de crédito: 15,32% frente al 4,62% de las bajas observadas.
- Los 23 intermitentes son demasiado pocos para extraer conclusiones generales. Se mantienen separados y no se incorporarán a la segmentación.

### 19.2. Sensibilidad estructural ante la incorporación de censurados

La asignación anterior obliga a cada caso a ingresar en uno de los cuatro perfiles existentes. Para comprobar si los censurados podrían modificar la estructura, se repite el procedimiento de segmentación incorporando exclusivamente los 1.097 censurados con tres meses consecutivos.

Se mantienen sin cambios la imputación aprendida, los cinco bosques, la representación mediante el último corte de cada árbol, el promedio entre semillas, el método jerárquico de Ward y `k=4`.

In [82]:
# ============================================================
# 1. Población ampliada
# ============================================================

censurados_completos = (
    sensibilidad_t0[
        (
            sensibilidad_t0["poblacion"]
            == "censurada"
        )
        & (
            sensibilidad_t0["historia_3m_completa"]
            == 1
        )
    ]
    .copy()
)


base_ampliada = (
    pd.concat(
        [
            base_clusters,
            censurados_completos,
        ],
        ignore_index=True,
        sort=False,
    )
    .sort_values(
        "numero_de_cliente",
        kind="mergesort",
    )
    .reset_index(drop=True)
)


print(
    "Bajas estrictas:",
    len(base_clusters),
)

print(
    "Censurados completos:",
    len(censurados_completos),
)

print(
    "Población ampliada:",
    len(base_ampliada),
)

print(
    "Duplicados:",
    base_ampliada[
        "numero_de_cliente"
    ].duplicated().sum(),
)


assert len(censurados_completos) == 1097
assert base_ampliada["numero_de_cliente"].is_unique


# ============================================================
# 2. Misma transformación utilizada por los bosques originales
# ============================================================

X_ampliada_cruda = (
    base_ampliada[FEATURES_RF]
    .replace(
        [np.inf, -np.inf],
        np.nan,
    )
    .astype(np.float32)
)


X_ampliada_rf = (
    imputador_rf
    .transform(X_ampliada_cruda)
    .astype(np.float32)
)


print(
    "Features antes de imputar:",
    X_ampliada_cruda.shape[1],
)

print(
    "Features entregadas a los bosques:",
    X_ampliada_rf.shape[1],
)

print(
    "Nulos después de imputar:",
    int(np.isnan(X_ampliada_rf).sum()),
)

print(
    "Infinitos después de imputar:",
    int(np.isinf(X_ampliada_rf).sum()),
)


assert X_ampliada_cruda.shape[1] == len(FEATURES_RF)
assert X_ampliada_rf.shape[1] == X_rf.shape[1]
assert np.isnan(X_ampliada_rf).sum() == 0
assert np.isinf(X_ampliada_rf).sum() == 0


# ============================================================
# 3. Reconstrucción de los perfiles de último corte
# ============================================================

perfiles_ampliados_por_semilla = [
    construir_perfil_bosque(
        modelo,
        X_ampliada_rf,
    )
    for modelo in rf_models
]


perfil_ampliado_promedio = np.mean(
    np.stack(
        perfiles_ampliados_por_semilla,
        axis=0,
    ),
    axis=0,
    dtype=np.float64,
)


atributos_utilizados_ampliados = (
    perfil_ampliado_promedio.sum(axis=0)
    > 0
)


matriz_perfiles_ampliada = (
    perfil_ampliado_promedio[
        :,
        atributos_utilizados_ampliados,
    ]
)


suma_perfiles_ampliados = (
    matriz_perfiles_ampliada.sum(axis=1)
)


print(
    "Atributos utilizados:",
    matriz_perfiles_ampliada.shape[1],
)

print(
    "Suma mínima por cliente:",
    suma_perfiles_ampliados.min(),
)

print(
    "Suma máxima por cliente:",
    suma_perfiles_ampliados.max(),
)


assert np.allclose(
    suma_perfiles_ampliados,
    1,
)


# ============================================================
# 4. Nuevo agrupamiento jerárquico mediante Ward
# ============================================================

linkage_ampliado = linkage(
    matriz_perfiles_ampliada,
    method="ward",
)


labels_ampliados = fcluster(
    linkage_ampliado,
    t=K_FINAL,
    criterion="maxclust",
)


base_ampliada["cluster_ampliado_raw"] = (
    labels_ampliados
)


tamanios_ampliados = (
    pd.Series(labels_ampliados)
    .value_counts()
    .sort_values(ascending=False)
    .tolist()
)


# ============================================================
# 5. Comparación sobre las mismas 1.991 bajas estrictas
# ============================================================

cluster_ampliado_por_cliente = pd.Series(
    labels_ampliados,
    index=base_ampliada[
        "numero_de_cliente"
    ].astype("int64"),
)


labels_estrictas_ampliados = (
    cluster_ampliado_por_cliente
    .reindex(ids_cluster)
    .to_numpy()
)


ari_censura_estructural = adjusted_rand_score(
    labels_raw,
    labels_estrictas_ampliados,
)


resumen_censura_estructural = pd.DataFrame([{
    "bajas_estrictas": len(base_clusters),

    "censurados_completos": (
        len(censurados_completos)
    ),

    "poblacion_ampliada": (
        len(base_ampliada)
    ),

    "atributos_utilizados": (
        matriz_perfiles_ampliada.shape[1]
    ),

    "ARI_estrictas_antes_despues": (
        ari_censura_estructural
    ),

    "tamanios_clustering_ampliado": (
        tamanios_ampliados
    ),
}])


display(
    resumen_censura_estructural.round(4)
)


# ============================================================
# 6. Alineación de nombres para interpretar los cambios
# ============================================================

perfiles_originales = (
    base_clusters
    .set_index("numero_de_cliente")
    .loc[ids_cluster, "perfil"]
    .to_numpy()
)


comparacion_estructural = pd.DataFrame({
    "perfil_original": perfiles_originales,
    "cluster_ampliado_raw": (
        labels_estrictas_ampliados
    ),
})


tabla_conteos = (
    pd.crosstab(
        comparacion_estructural[
            "perfil_original"
        ],
        comparacion_estructural[
            "cluster_ampliado_raw"
        ],
    )
    .reindex(
        index=PROFILE_ORDER,
        fill_value=0,
    )
)


filas_asignadas, columnas_asignadas = (
    linear_sum_assignment(
        -tabla_conteos.to_numpy()
    )
)


mapa_cluster_ampliado = {
    tabla_conteos.columns[columna]:
    tabla_conteos.index[fila]

    for fila, columna
    in zip(
        filas_asignadas,
        columnas_asignadas,
    )
}


comparacion_estructural[
    "perfil_ampliado_alineado"
] = (
    comparacion_estructural[
        "cluster_ampliado_raw"
    ]
    .map(mapa_cluster_ampliado)
)


tabla_transicion_estructural = (
    pd.crosstab(
        comparacion_estructural[
            "perfil_original"
        ],
        comparacion_estructural[
            "perfil_ampliado_alineado"
        ],
        normalize="index",
    )
    .mul(100)
    .reindex(
        index=PROFILE_ORDER,
        columns=PROFILE_ORDER,
        fill_value=0,
    )
)


display(
    tabla_transicion_estructural.round(2)
)

Bajas estrictas: 1991
Censurados completos: 1097
Población ampliada: 3088
Duplicados: 0
Features antes de imputar: 265
Features entregadas a los bosques: 299
Nulos después de imputar: 0
Infinitos después de imputar: 0
Atributos utilizados: 231
Suma mínima por cliente: 1.0000000020489088
Suma máxima por cliente: 1.0000000169500705


,bajas_estrictas,censurados_completos,poblacion_ampliada,atributos_utilizados,ARI_estrictas_antes_despues,tamanios_clustering_ampliado
0,1991,1097,3088,231,0.79,"[1363, 1183, 277, 265]"


perfil_ampliado_alineado,Transaccionales multicanal,Vinculados por nómina,Vinculados por crédito,Desvinculación avanzada
perfil_original,,,,
Transaccionales multicanal,88.66,2.55,0.92,7.87
Vinculados por nómina,0.00,100.00,0.00,0.00
Vinculados por crédito,0.00,0.00,100.00,0.00
Desvinculación avanzada,1.40,1.91,0.51,96.18


In [83]:
# ============================================================
# 1. Asignar nombres alineados a toda la población ampliada
# ============================================================

base_ampliada[
    "perfil_ampliado_alineado"
] = (
    base_ampliada[
        "cluster_ampliado_raw"
    ]
    .map(mapa_cluster_ampliado)
)


assert base_ampliada[
    "perfil_ampliado_alineado"
].notna().all()


# ============================================================
# 2. Distribución por población
# ============================================================

distribucion_estructural = (
    base_ampliada
    .groupby(
        [
            "poblacion",
            "perfil_ampliado_alineado",
        ],
        observed=True,
    )
    .agg(
        clientes=(
            "numero_de_cliente",
            "nunique",
        )
    )
    .reset_index()
)


distribucion_estructural[
    "porcentaje_poblacion"
] = (
    100
    * distribucion_estructural["clientes"]
    / distribucion_estructural
      .groupby("poblacion")["clientes"]
      .transform("sum")
)


tabla_distribucion_estructural = (
    distribucion_estructural
    .pivot(
        index="perfil_ampliado_alineado",
        columns="poblacion",
        values="porcentaje_poblacion",
    )
    .reindex(
        index=PROFILE_ORDER,
        columns=[
            "estricta",
            "censurada",
        ],
    )
    .fillna(0)
)


display(
    distribucion_estructural
    .sort_values(
        [
            "poblacion",
            "perfil_ampliado_alineado",
        ]
    )
    .round(2)
)


# ============================================================
# 3. Visualización
# ============================================================

fig, axes = plt.subplots(
    1,
    2,
    figsize=(16, 6.5),
    gridspec_kw={
        "width_ratios": [1.25, 1],
    },
)


# Panel 1: transición de las bajas estrictas originales

sns.heatmap(
    tabla_transicion_estructural,
    annot=True,
    fmt=".1f",
    cmap="Blues",
    vmin=0,
    vmax=100,
    linewidths=0.5,
    linecolor="white",
    cbar_kws={
        "label": "% del perfil original",
    },
    ax=axes[0],
)

axes[0].set_title(
    "Conservación de los perfiles originales"
)

axes[0].set_xlabel(
    "Perfil después de incorporar censurados"
)

axes[0].set_ylabel(
    "Perfil original"
)

axes[0].set_xticklabels(
    [
        etiquetas_perfiles[perfil]
        for perfil in PROFILE_ORDER
    ],
    rotation=0,
    fontsize=8,
)

axes[0].set_yticklabels(
    [
        etiquetas_perfiles[perfil]
        for perfil in PROFILE_ORDER
    ],
    rotation=0,
    fontsize=8,
)


# Panel 2: composición de la solución ampliada

tabla_distribucion_estructural.plot(
    kind="bar",
    ax=axes[1],
    color=[
        "#2F6B9A",
        "#D9A441",
    ],
    width=0.72,
)

axes[1].set_title(
    "Composición de la solución ampliada"
)

axes[1].set_xlabel(
    "Perfil"
)

axes[1].set_ylabel(
    "% dentro de cada población"
)

axes[1].set_xticklabels(
    [
        etiquetas_perfiles[perfil]
        for perfil in PROFILE_ORDER
    ],
    rotation=0,
    fontsize=8,
)

axes[1].legend(
    [
        "Bajas estrictas",
        "Casos censurados",
    ],
    title="Población",
    frameon=False,
)


for contenedor in axes[1].containers:
    axes[1].bar_label(
        contenedor,
        fmt="%.1f%%",
        padding=2,
        fontsize=8,
    )


fig.suptitle(
    "Sensibilidad estructural ante la incorporación de casos censurados",
    fontsize=15,
)

fig.text(
    0.5,
    0.01,
    (
        "Bajas estrictas: n=1.991 · "
        "Censurados con historia completa: n=1.097 · "
        f"ARI sobre las bajas estrictas: "
        f"{ari_censura_estructural:.2f}"
    ),
    ha="center",
    fontsize=9,
)

fig.tight_layout(
    rect=[0, 0.04, 1, 0.94]
)

fig.savefig(
    OUTPUT_DIR / "09_sensibilidad_censura.png",
    dpi=160,
    bbox_inches="tight",
)

plt.close(fig)

,poblacion,perfil_ampliado_alineado,clientes,porcentaje_poblacion
0,censurada,Desvinculación avanzada,351,32.00
1,censurada,Transaccionales multicanal,484,44.12
2,censurada,Vinculados por crédito,172,15.68
3,censurada,Vinculados por nómina,90,8.20
4,estricta,Desvinculación avanzada,832,41.79
5,estricta,Transaccionales multicanal,879,44.15
6,estricta,Vinculados por crédito,105,5.27
7,estricta,Vinculados por nómina,175,8.79


![Sensibilidad estructural ante censurados](./eda_miranda_outputs/09_sensibilidad_censura.png)

Las barras azules de la derecha representan la clasificación de las bajas estrictas después de reconstruir el clustering con censurados, no los tamaños originales. Por eso difieren de 49,17%, 6,78%, 4,62% y 39,43%.
La comparación ampliada muestra:
- Multicanal: prácticamente idéntico, 44,15% frente a 44,12%.
- Nómina: muy parecido, 8,79% frente a 8,20%.
- Crédito: más frecuente entre censurados, 15,68% frente a 5,27%.
- Desvinculación avanzada: menos frecuente entre censurados, 32,00% frente a 41,79%.
Esto demuestra dos cosas compatibles: la estructura de cuatro perfiles es estable, pero la composición de los censurados no es idéntica a la de las bajas confirmadas.

## 20. Intermitentes: desaparición y regreso

Los clientes intermitentes presentan al menos una ausencia seguida por una reaparición. Por este motivo no se consideran bajas permanentes ni se incorporan a la segmentación principal.

Debido a que solamente se observan 23 casos, sus resultados se presentan de manera descriptiva. Se compara el último mes activo anterior a la ausencia con el primer mes observado después del regreso.

In [84]:
%%sql

CREATE OR REPLACE TABLE cambios_intermitentes AS

WITH eventos AS (
    SELECT *
    FROM eventos_baja
    WHERE poblacion = 'intermitente'
),

retornos AS (
    SELECT
        e.numero_de_cliente,
        e.numero_evento,
        MIN(r.foto_mes) AS primer_mes_retorno

    FROM eventos AS e

    INNER JOIN raw AS r
        ON e.numero_de_cliente =
           r.numero_de_cliente

       AND r.foto_mes >
           e.primer_mes_ausente

    GROUP BY
        e.numero_de_cliente,
        e.numero_evento
)

SELECT
    e.numero_de_cliente,
    e.ultimo_mes_activo,
    e.primer_mes_ausente,
    r.primer_mes_retorno,

    calendario_retorno.indice_mes
    - calendario_ausencia.indice_mes
        AS meses_ausente,

    antes.ctrx_quarter
        AS transacciones_antes,

    despues.ctrx_quarter
        AS transacciones_despues,

    despues.ctrx_quarter
    - antes.ctrx_quarter
        AS cambio_transacciones,

    antes.cproductos
        AS productos_antes,

    despues.cproductos
        AS productos_despues,

    despues.cproductos
    - antes.cproductos
        AS cambio_productos,

    antes.mcuentas_saldo
        AS saldo_antes,

    despues.mcuentas_saldo
        AS saldo_despues,

    despues.mcuentas_saldo
    - antes.mcuentas_saldo
        AS cambio_saldo,

    antes.mrentabilidad
        AS rentabilidad_antes,

    despues.mrentabilidad
        AS rentabilidad_despues,

    despues.mrentabilidad
    - antes.mrentabilidad
        AS cambio_rentabilidad

FROM eventos AS e

INNER JOIN retornos AS r
    USING (
        numero_de_cliente,
        numero_evento
    )

INNER JOIN calendario AS calendario_ausencia
    ON e.primer_mes_ausente =
       calendario_ausencia.foto_mes

INNER JOIN calendario AS calendario_retorno
    ON r.primer_mes_retorno =
       calendario_retorno.foto_mes

INNER JOIN raw AS antes
    ON e.numero_de_cliente =
       antes.numero_de_cliente

   AND e.ultimo_mes_activo =
       antes.foto_mes

INNER JOIN raw AS despues
    ON e.numero_de_cliente =
       despues.numero_de_cliente

   AND r.primer_mes_retorno =
       despues.foto_mes

ORDER BY e.numero_de_cliente;


SELECT
    COUNT(*) AS filas,

    COUNT(
        DISTINCT numero_de_cliente
    ) AS clientes,

    MIN(meses_ausente)
        AS ausencia_minima,

    MEDIAN(meses_ausente)
        AS ausencia_mediana,

    MAX(meses_ausente)
        AS ausencia_maxima

FROM cambios_intermitentes;

,filas,clientes,ausencia_minima,ausencia_mediana,ausencia_maxima
0,23,23,1,1.00,4


In [85]:
intermitentes_cambio = %sql SELECT * FROM cambios_intermitentes ORDER BY numero_de_cliente


assert len(intermitentes_cambio) == 23
assert intermitentes_cambio[
    "numero_de_cliente"
].is_unique


columnas_cambio = {
    "cambio_transacciones": "Transacciones",
    "cambio_productos": "Productos",
    "cambio_saldo": "Saldo",
    "cambio_rentabilidad": "Rentabilidad",
}


filas_resumen_intermitentes = []

for columna, nombre in columnas_cambio.items():
    valores = (
        intermitentes_cambio[columna]
        .dropna()
    )

    filas_resumen_intermitentes.append({
        "metrica": nombre,
        "casos_comparables": len(valores),
        "cambio_mediano": valores.median(),
        "cambio_p25": valores.quantile(0.25),
        "cambio_p75": valores.quantile(0.75),
        "porcentaje_aumento": (
            100 * (valores > 0).mean()
        ),
        "porcentaje_sin_cambio": (
            100 * (valores == 0).mean()
        ),
        "porcentaje_disminucion": (
            100 * (valores < 0).mean()
        ),
    })


resumen_intermitentes = pd.DataFrame(
    filas_resumen_intermitentes
)


display(
    resumen_intermitentes.round(2)
)


display(
    distribucion_sensibilidad[
        distribucion_sensibilidad[
            "poblacion"
        ] == "intermitente"
    ][
        [
            "perfil_compatible",
            "clientes",
            "confianza_mediana",
            "porcentaje_poblacion",
        ]
    ]
    .sort_values(
        "clientes",
        ascending=False,
    )
    .round(2)
)

,metrica,casos_comparables,cambio_mediano,cambio_p25,cambio_p75,porcentaje_aumento,porcentaje_sin_cambio,porcentaje_disminucion
0,Transacciones,23,-9.00,-27.50,1.00,26.09,4.35,69.57
1,Productos,23,1.00,0.00,1.00,52.17,30.43,17.39
2,Saldo,23,69.66,"-5,706.88","23,814.84",56.52,0.00,43.48
3,Rentabilidad,23,-436.09,"-2,414.48",631.29,39.13,0.00,60.87


,perfil_compatible,clientes,confianza_mediana,porcentaje_poblacion
5,Transaccionales multicanal,12,0.80,52.17
6,Vinculados por nómina,6,0.73,26.09
4,Desvinculación avanzada,5,0.60,21.74


Los resultados muestran que el regreso no implica recuperación:
- 69,57% regresa con menos transacciones; cambio mediano: −9.
- 60,87% regresa con menor rentabilidad; cambio mediano: −$436.
- 52,17% aumenta productos, con una mediana de +1.
- El saldo es muy heterogéneo: mediana +$70, pero con un intervalo intercuartílico muy amplio.
- La ausencia suele ser breve: mediana de un mes.
- No aparece ningún intermitente compatible con el perfil de crédito.

In [86]:
metricas_intermitentes = [
    (
        "cambio_transacciones",
        "Cambio de transacciones",
        "Cantidad",
        "#2F6B9A",
    ),
    (
        "cambio_productos",
        "Cambio de productos",
        "Cantidad",
        "#2A9D8F",
    ),
    (
        "cambio_saldo",
        "Cambio de saldo",
        "$ nominales",
        "#8C6BB1",
    ),
    (
        "cambio_rentabilidad",
        "Cambio de rentabilidad",
        "$ nominales",
        "#D97732",
    ),
]


generador_jitter = np.random.default_rng(
    214363
)


fig, axes = plt.subplots(
    2,
    2,
    figsize=(13, 8),
)


for ax, (
    columna,
    titulo,
    etiqueta_x,
    color,
) in zip(
    axes.ravel(),
    metricas_intermitentes,
):
    valores = (
        intermitentes_cambio[columna]
        .dropna()
        .to_numpy()
    )

    q25 = np.quantile(
        valores,
        0.25,
    )

    mediana = np.median(
        valores
    )

    q75 = np.quantile(
        valores,
        0.75,
    )

    jitter = generador_jitter.normal(
        0,
        0.035,
        len(valores),
    )

    ax.scatter(
        valores,
        jitter,
        color=color,
        alpha=0.60,
        s=35,
        zorder=2,
    )

    ax.hlines(
        y=0,
        xmin=q25,
        xmax=q75,
        color=color,
        linewidth=9,
        alpha=0.35,
        zorder=3,
    )

    ax.scatter(
        mediana,
        0,
        color=color,
        edgecolor="black",
        linewidth=0.8,
        s=110,
        zorder=4,
        label="Mediana",
    )

    ax.axvline(
        0,
        color="#555555",
        linestyle="--",
        linewidth=1,
    )

    porcentaje_disminucion = (
        100
        * (valores < 0).mean()
    )

    porcentaje_aumento = (
        100
        * (valores > 0).mean()
    )

    ax.set_title(
        titulo
    )

    ax.set_xlabel(
        etiqueta_x
    )

    ax.set_yticks([])

    ax.text(
        0.02,
        0.95,
        (
            f"Mediana: {mediana:,.2f}\n"
            f"Aumenta: {porcentaje_aumento:.1f}% · "
            f"Disminuye: {porcentaje_disminucion:.1f}%"
        ),
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=9,
    )


fig.suptitle(
    "Cambios observados después del regreso de los clientes intermitentes",
    fontsize=15,
)

fig.text(
    0.5,
    0.01,
    (
        "Cada punto representa un cliente. "
        "La banda gruesa muestra el intervalo intercuartílico "
        "y el punto destacado representa la mediana. n=23."
    ),
    ha="center",
    fontsize=9,
)

fig.tight_layout(
    rect=[0, 0.04, 1, 0.95]
)

fig.savefig(
    OUTPUT_DIR / "10_intermitentes_retorno.png",
    dpi=160,
    bbox_inches="tight",
)

plt.close(fig)

![Cambios observados al regresar](./eda_miranda_outputs/10_intermitentes_retorno.png)

### Interpretación de los intermitentes

La ausencia de los clientes intermitentes tiene una mediana de un mes, aunque se observan casos de hasta cuatro meses. La reaparición no representa necesariamente una recuperación del comportamiento previo.

El 69,57% regresa con menos transacciones y el cambio mediano es de −9. La rentabilidad registrada disminuye en el 60,87% de los casos, con una variación mediana de −$436. En cambio, el 52,17% aumenta la cantidad de productos y la variación mediana es de un producto adicional.

El saldo presenta una dispersión considerable: aunque el cambio mediano es positivo, el intervalo intercuartílico abarca desde aproximadamente −$5.707 hasta $23.815. No resulta adecuado resumir este comportamiento como una mejora general.

Los resultados se basan en sólo 23 clientes. Se utilizan para documentar la existencia de desapariciones temporales, pero no para inferir patrones poblacionales ni diseñar una estrategia específica.

## 21. Distribución de clientes VIP

La condición VIP se analiza después de construir los perfiles. Se informa tanto la proporción de clientes VIP dentro de cada segmento como la distribución de los VIP entre los cuatro perfiles.

También se verifica si `cliente_vip` ingresó al Random Forest y si llegó a actuar como último corte definitorio de las hojas. Una presencia escasa se interpreta descriptivamente y no se utiliza para renombrar segmentos.

In [87]:
# ============================================================
# 1. Auditoría de la variable VIP
# ============================================================

assert "cliente_vip" in base_clusters.columns


valores_vip_observados = sorted(
    base_clusters[
        "cliente_vip"
    ]
    .dropna()
    .unique()
    .tolist()
)


auditoria_vip = pd.DataFrame([{
    "clientes": (
        base_clusters[
            "numero_de_cliente"
        ].nunique()
    ),

    "nulos_cliente_vip": (
        base_clusters[
            "cliente_vip"
        ].isna().sum()
    ),

    "valores_observados": (
        valores_vip_observados
    ),

    "cliente_vip_ingreso_al_bosque": (
        "cliente_vip"
        in FEATURES_RF
    ),

    "cliente_vip_fue_ultimo_corte": (
        "cliente_vip"
        in perfiles_bosque_final.columns
    ),
}])


display(
    auditoria_vip
)


assert set(
    valores_vip_observados
).issubset({
    0,
    1,
})


# ============================================================
# 2. Identificación de clientes VIP
# ============================================================

base_clusters = base_clusters.copy()

base_clusters["es_vip"] = (
    base_clusters["cliente_vip"]
    .fillna(0)
    .eq(1)
    .astype(int)
)


total_vip = int(
    base_clusters["es_vip"].sum()
)


print(
    "Clientes VIP:",
    total_vip,
)

print(
    "Porcentaje VIP:",
    round(
        100
        * total_vip
        / len(base_clusters),
        4,
    ),
)


# ============================================================
# 3. Distribución VIP por perfil
# ============================================================

resumen_vip = (
    base_clusters
    .groupby(
        [
            "cluster_final",
            "perfil",
        ],
        observed=True,
    )
    .agg(
        clientes=(
            "numero_de_cliente",
            "nunique",
        ),

        clientes_vip=(
            "es_vip",
            "sum",
        ),
    )
    .reset_index()
    .sort_values(
        "cluster_final"
    )
)


resumen_vip[
    "porcentaje_vip_dentro_perfil"
] = (
    100
    * resumen_vip["clientes_vip"]
    / resumen_vip["clientes"]
)


if total_vip > 0:
    resumen_vip[
        "porcentaje_de_todos_los_vip"
    ] = (
        100
        * resumen_vip["clientes_vip"]
        / total_vip
    )

else:
    resumen_vip[
        "porcentaje_de_todos_los_vip"
    ] = np.nan


display(
    resumen_vip.round(4)
)


# ============================================================
# 4. Participación de VIP como último corte
# ============================================================

if (
    "cliente_vip"
    in perfiles_bosque_final.columns
):
    frecuencia_vip_ultimo_corte = (
        100
        * perfiles_bosque_final[
            "cliente_vip"
        ].mean()
    )

else:
    frecuencia_vip_ultimo_corte = 0.0


print(
    "Frecuencia promedio como último corte:",
    round(
        frecuencia_vip_ultimo_corte,
        4,
    ),
    "%",
)

,clientes,nulos_cliente_vip,valores_observados,cliente_vip_ingreso_al_bosque,cliente_vip_fue_ultimo_corte
0,1991,0,"[0, 1]",True,False


Clientes VIP: 3
Porcentaje VIP: 0.1507


,cluster_final,perfil,clientes,clientes_vip,porcentaje_vip_dentro_perfil,porcentaje_de_todos_los_vip
0,1,Transaccionales multicanal,979,2,0.20,66.67
1,2,Vinculados por nómina,135,1,0.74,33.33
2,3,Vinculados por crédito,92,0,0.00,0.00
3,4,Desvinculación avanzada,785,0,0.00,0.00


Frecuencia promedio como último corte: 0.0 %


In [88]:
print(
    "Ingresó como feature:",
    "cliente_vip" in FEATURES_RF,
)

print(
    "Fue utilizado como último corte:",
    "cliente_vip" in perfiles_bosque_final.columns,
)

if "cliente_vip" in perfiles_bosque_final.columns:
    print(
        "Frecuencia promedio como último corte:",
        round(
            100
            * perfiles_bosque_final["cliente_vip"].mean(),
            4,
        ),
        "%",
    )
else:
    print(
        "Frecuencia promedio como último corte:",
        0.0,
        "%",
    )

Ingresó como feature: True
Fue utilizado como último corte: False
Frecuencia promedio como último corte: 0.0 %


### Interpretación de la condición VIP

Sólo 3 de los 1.991 clientes de la segmentación principal poseen condición VIP, equivalentes al 0,15% de la población. Dos pertenecen a **Transaccionales multicanal** y uno a **Vinculados por nómina**.

La variable `cliente_vip` ingresó como feature candidata del Random Forest, pero no fue utilizada como último corte en ninguno de los perfiles analizados. Su frecuencia promedio como atributo definitorio es, por lo tanto, 0%.

La escasa cantidad de casos no permite evaluar una asociación entre condición VIP y perfil. No se genera una visualización específica ni se utiliza esta característica para renombrar segmentos, priorizar acciones o formular conclusiones comerciales.

## 22. Recomendaciones preliminares

Las acciones propuestas traducen patrones descriptivos en hipótesis que deberán validarse mediante experimentos controlados. No se afirma que una intervención vaya a evitar la salida ni que todos los integrantes de un perfil deban recibir el mismo tratamiento.

| Perfil | Evidencia observada | Hipótesis de intervención | Resultado que debería medirse | Riesgo principal |
|---|---|---|---|---|
| **Transaccionales multicanal** | Conservan actividad, productos, tarjetas y canales digitales, aunque muestran una reducción progresiva de transacciones, uso de tarjeta y saldo | Realizar un contacto temprano para identificar fricciones recientes y probar beneficios vinculados con los productos todavía utilizados | Retención incremental, recuperación de actividad y costo por cliente retenido | Subsidiar clientes que habrían permanecido o salido de todas formas |
| **Vinculados por nómina** | Presentan elevada actividad y uso digital; el 66,7% conserva payroll en `t0` y el 33,3% lo perdió durante la ventana | Investigar la interrupción de la acreditación y probar beneficios condicionados a la continuidad o recuperación de nómina | Recuperación de payroll, permanencia posterior y margen incremental | Confundir cambios laborales, estacionales o administrativos con intención de salida |
| **Vinculados por crédito** | Todos poseen crédito, presentan baja actividad transaccional y el 83,7% mantiene saldo negativo; la rentabilidad registrada no puede descomponerse de forma aditiva | Aplicar una revisión conjunta de vinculación y riesgo antes de ofrecer refinanciación, beneficios o asistencia | Regularización, permanencia, riesgo incremental y resultado económico neto | Aumentar exposición crediticia o interpretar cargos registrados como valor económico sostenible |
| **Desvinculación avanzada** | Muestran pocas transacciones y familias activas, escaso uso de tarjetas, saldo mediano negativo y 88,5% de saldos negativos | Utilizar un contacto diagnóstico de bajo costo y reservar incentivos para clientes con recuperación plausible | Tasa de contacto, recuperación de actividad, regularización y resultado neto | Financiar masivamente relaciones deterioradas o sin respuesta probable |

La condición VIP no se utiliza para definir una estrategia diferenciada porque sólo se observan tres casos.

Para transformar estas hipótesis en decisiones se necesitan, como mínimo:

- costo de contacto;
- costo del incentivo;
- probabilidad de permanencia sin intervención;
- efecto incremental de la acción;
- margen futuro esperado;
- riesgo crediticio;
- horizonte de evaluación.

El punto de equilibrio mínimo puede expresarse como:

\[
\text{efectividad incremental mínima}
=
\frac{
\text{costo total por cliente tratado}
}{
\text{valor esperado de una retención incremental}
}
\]

Las acciones deberían implementarse inicialmente como pilotos con grupo de control, asignación aleatoria y métricas definidas antes del contacto.